In [2]:
import torch
import transformers
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

/home/ali/miniconda3/envs/ai-bootcamp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu130
Transformers: 5.14.1
Datasets: 5.0.1
Accelerate: 1.14.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU


In [4]:
import os

for proxy_variable in ["ALL_PROXY", "all_proxy"]:
    removed_value = os.environ.pop(proxy_variable, None)
    print(f"{proxy_variable}: removed={removed_value}")

print("HTTP_PROXY:", os.environ.get("HTTP_PROXY"))
print("HTTPS_PROXY:", os.environ.get("HTTPS_PROXY"))

ALL_PROXY: removed=socks://127.0.0.1:10808
all_proxy: removed=socks://127.0.0.1:10808
HTTP_PROXY: http://127.0.0.1:10808
HTTPS_PROXY: http://127.0.0.1:10808


<div dir="rtl" align="right">

## بررسی Tokenizer مدل DistilBERT

در این مرحله، Tokenizer مخصوص مدل `DistilBERT` بارگذاری می‌شود. Tokenizer متن را به Tokenهای قابل‌فهم برای مدل تبدیل کرده و سپس برای هر Token یک شناسه عددی تولید می‌کند.

فعلاً فقط یک جمله نمونه بررسی می‌شود و هنوز مدل اصلی، داده‌های پروژه یا GPU برای آموزش استفاده نمی‌شوند.

</div>

In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = (
    "Great product, but the battery stopped working after two days."
)

encoded_sample = tokenizer(
    sample_text,
    truncation=True,
    max_length=512,
)

print("Model:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)
print("Sample token count:", len(encoded_sample["input_ids"]))
print(
    "Sample tokens:",
    tokenizer.convert_ids_to_tokens(encoded_sample["input_ids"]),
)

Model: distilbert/distilbert-base-uncased
Tokenizer class: BertTokenizer
Model max length: 512
Sample token count: 14
Sample tokens: ['[CLS]', 'great', 'product', ',', 'but', 'the', 'battery', 'stopped', 'working', 'after', 'two', 'days', '.', '[SEP]']


<div dir="rtl" align="right">

## بارگذاری داده‌های آموزش و اعتبارسنجی

در این مرحله، فقط ستون‌های موردنیاز مدل بارگذاری می‌شوند:

- امتیاز واقعی هر Review در ستون `overall`
- عنوان Review در ستون `summary`
- متن پاک‌سازی‌شده Review در ستون `reviewText_clean`

برای کاهش مصرف حافظه، سایر ستون‌های دیتاست خوانده نمی‌شوند. هنوز Tokenization یا آموزش مدل انجام نمی‌شود.

</div>

In [6]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("/home/ali/datasets/AI_Project2/nlp")

TRAIN_PATH = DATA_DIR / "nlp_train_clean.csv"
VALIDATION_PATH = DATA_DIR / "nlp_validation_clean.csv"

required_columns = [
    "overall",
    "summary",
    "reviewText_clean",
]

train_df = pd.read_csv(
    TRAIN_PATH,
    usecols=required_columns,
)

validation_df = pd.read_csv(
    VALIDATION_PATH,
    usecols=required_columns,
)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nLabel distribution:")
print(
    train_df["overall"]
    .value_counts()
    .sort_index()
)

Train shape: (664451, 3)
Validation shape: (166161, 3)

Train columns:
['overall', 'summary', 'reviewText_clean']

Label distribution:
overall
1     65971
2     44857
3     64383
4    124422
5    364818
Name: count, dtype: int64


<div dir="rtl" align="right">

## آماده‌سازی داده آزمایشی برای DistilBERT

برای آزمایش اولیه، یک زیرمجموعه طبقه‌بندی‌شده از داده‌های اصلی انتخاب می‌شود:

- ۵۰٬۰۰۰ نمونه برای آموزش
- ۱۰٬۰۰۰ نمونه برای اعتبارسنجی

نمونه‌گیری به‌صورت Stratified انجام می‌شود تا نسبت امتیازهای ۱ تا ۵ مشابه دیتاست اصلی باقی بماند.

عنوان Review و متن آن در یک ستون ترکیب می‌شوند. همچنین برچسب‌های اصلی ۱ تا ۵ به مقادیر ۰ تا ۴ تبدیل می‌شوند، زیرا مدل‌های طبقه‌بندی Hugging Face برچسب‌ها را از صفر شماره‌گذاری می‌کنند.

</div>

In [7]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
PILOT_TRAIN_SIZE = 50_000
PILOT_VALIDATION_SIZE = 10_000

pilot_train_df, _ = train_test_split(
    train_df,
    train_size=PILOT_TRAIN_SIZE,
    random_state=RANDOM_STATE,
    stratify=train_df["overall"],
)

pilot_validation_df, _ = train_test_split(
    validation_df,
    train_size=PILOT_VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=validation_df["overall"],
)

for dataframe in [pilot_train_df, pilot_validation_df]:
    dataframe["summary"] = dataframe["summary"].fillna("").astype(str)
    dataframe["reviewText_clean"] = (
        dataframe["reviewText_clean"].fillna("").astype(str)
    )

    dataframe["text"] = (
        dataframe["summary"].str.strip()
        + " [SEP] "
        + dataframe["reviewText_clean"].str.strip()
    )

    dataframe["label"] = dataframe["overall"].astype(int) - 1

pilot_train_df = pilot_train_df[["text", "label"]].reset_index(drop=True)
pilot_validation_df = pilot_validation_df[
    ["text", "label"]
].reset_index(drop=True)

print("Pilot train shape:", pilot_train_df.shape)
print("Pilot validation shape:", pilot_validation_df.shape)

print("\nPilot train label distribution:")
print(
    pilot_train_df["label"]
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nSample text:")
print(pilot_train_df.loc[0, "text"][:500])

print("\nSample label:", pilot_train_df.loc[0, "label"])

Pilot train shape: (50000, 2)
Pilot validation shape: (10000, 2)

Pilot train label distribution:
label
0    0.0993
1    0.0675
2    0.0969
3    0.1873
4    0.5491
Name: proportion, dtype: float64

Sample text:
Great product. Fantastic customer support [SEP] This works great. If you have an older cable box issues may come up. The easy fix is: for the tv near the transmitter do not use HDMI. Other inputs will work fine and be the same quality. Other than that it's great.

Sample label: 4


<div dir="rtl" align="right">

## بررسی طول Tokenهای متن

در این مرحله، طول Tokenهای ۱۰٬۰۰۰ نمونه از مجموعه آموزش آزمایشی بررسی می‌شود.

هدف این بررسی، انتخاب طول ورودی مناسب برای مدل است. طول بیشتر اطلاعات بیشتری از Review حفظ می‌کند، اما مصرف حافظه GPU و زمان آموزش را افزایش می‌دهد.

هنوز Padding، Truncation یا آموزش مدل انجام نمی‌شود.

</div>

In [8]:
import numpy as np

TOKEN_LENGTH_SAMPLE_SIZE = 10_000

length_sample = pilot_train_df["text"].sample(
    n=TOKEN_LENGTH_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
).tolist()

length_encoding = tokenizer(
    length_sample,
    add_special_tokens=True,
    truncation=False,
    padding=False,
    return_length=True,
)

token_lengths = np.asarray(length_encoding["length"])

print("Number of examined texts:", len(token_lengths))
print("Minimum token length:", token_lengths.min())
print("Mean token length:", round(token_lengths.mean(), 2))
print("Maximum token length:", token_lengths.max())

print("\nToken length percentiles:")
for percentile in [50, 75, 90, 95, 99]:
    value = np.percentile(token_lengths, percentile)
    print(f"{percentile}th percentile: {value:.0f}")

print("\nTexts exceeding common limits:")
for limit in [128, 256, 512]:
    percentage = (token_lengths > limit).mean() * 100
    print(f"More than {limit} tokens: {percentage:.2f}%")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (735 > 512). Running this sequence through the model will result in indexing errors


Number of examined texts: 10000
Minimum token length: 33
Mean token length: 154.29
Maximum token length: 3637

Token length percentiles:
50th percentile: 106
75th percentile: 173
90th percentile: 291
95th percentile: 400
99th percentile: 784

Texts exceeding common limits:
More than 128 tokens: 38.50%
More than 256 tokens: 12.63%
More than 512 tokens: 2.99%


<div dir="rtl" align="right">

## تبدیل داده‌های آزمایشی به Tokenهای DistilBERT

در این مرحله، متن‌های مجموعه آموزش و اعتبارسنجی آزمایشی با Tokenizer مدل `DistilBERT` به ورودی عددی مدل تبدیل می‌شوند.

حداکثر طول ورودی روی ۵۱۲ Token، یعنی بیشترین طول پشتیبانی‌شده توسط DistilBERT، تنظیم شده است. با این مقدار حدود ۹۷ درصد متن‌های بررسی‌شده به‌طور کامل حفظ می‌شوند.

Padding به‌صورت ثابت انجام نمی‌شود و هنگام تشکیل هر Batch، فقط تا طول بلندترین نمونه همان Batch اضافه خواهد شد تا حافظه کمتری مصرف شود.

هنوز مدل اصلی بارگذاری یا آموزش داده نمی‌شود.

</div>

In [10]:
from datasets import Dataset

MAX_LENGTH = 512

pilot_train_dataset = Dataset.from_pandas(
    pilot_train_df,
    preserve_index=False,
)

pilot_validation_dataset = Dataset.from_pandas(
    pilot_validation_df,
    preserve_index=False,
)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized_train_dataset = pilot_train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing pilot train data",
)

tokenized_validation_dataset = pilot_validation_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing pilot validation data",
)

print("Maximum input length:", MAX_LENGTH)
print("Tokenized train samples:", len(tokenized_train_dataset))
print("Tokenized validation samples:", len(tokenized_validation_dataset))

print("\nDataset columns:")
print(tokenized_train_dataset.column_names)

print(
    "\nFirst sample token count:",
    len(tokenized_train_dataset[0]["input_ids"]),
)

Tokenizing pilot validation data: 100%|██████████| 10000/10000 [00:00<00:00, 15326.80 examples/s]

Maximum input length: 512
Tokenized train samples: 50000
Tokenized validation samples: 10000

Dataset columns:
['label', 'input_ids', 'token_type_ids', 'attention_mask']

First sample token count: 61


<div dir="rtl" align="right">

## بارگذاری مدل DistilBERT برای طبقه‌بندی امتیاز Review

در این مرحله، وزن‌های ازپیش‌آموزش‌دیده مدل `DistilBERT` بارگذاری می‌شوند و یک لایه طبقه‌بندی جدید با پنج خروجی به انتهای مدل اضافه می‌شود.

هر خروجی متناظر با یکی از امتیازهای ۱ تا ۵ است. وزن‌های لایه طبقه‌بندی جدید در ابتدا تصادفی هستند و طی Fine-tuning روی داده‌های پروژه آموزش داده خواهند شد.

مدل فعلاً روی حافظه RAM بارگذاری می‌شود و هنوز آموزش یا انتقال آن به GPU انجام نمی‌شود.

</div>

In [11]:
from transformers import AutoModelForSequenceClassification

NUM_LABELS = 5

id2label = {
    0: "1_star",
    1: "2_stars",
    2: "3_stars",
    3: "4_stars",
    4: "5_stars",
}

label2id = {
    label_name: label_id
    for label_id, label_name in id2label.items()
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Current device:", next(model.parameters()).device)
print("Label mapping:", model.config.id2label)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 25127.63it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model class: DistilBertForSequenceClassification
Number of labels: 5
Total parameters: 66,957,317
Trainable parameters: 66,957,317
Current device: cpu
Label mapping: {0: '1_star', 1: '2_stars', 2: '3_stars', 3: '4_stars', 4: '5_stars'}


<div dir="rtl" align="right">

## آزمایش اجرای یک Batch روی GPU

در این مرحله، چهار نمونه با طول ۵۱۲ Token انتخاب می‌شوند و یک مرحله Forward و Backward روی GPU اجرا می‌شود.

هدف این آزمایش، بررسی موارد زیر است:

- سازگاری کامل مدل با GPU
- امکان استفاده از طول ورودی ۵۱۲
- میزان تقریبی مصرف حافظه GPU
- پشتیبانی کارت گرافیک از محاسبات `bfloat16`

این مرحله فقط یک آزمایش حافظه است و هنوز آموزش اصلی مدل آغاز نمی‌شود.

</div>

In [12]:
import gc

import torch
from transformers import DataCollatorWithPadding

DEVICE = torch.device("cuda")
TEST_BATCH_SIZE = 4

# چهار نمونه با بیشترین طول ورودی
longest_indices = sorted(
    range(len(tokenized_train_dataset)),
    key=lambda index: len(
        tokenized_train_dataset[index]["input_ids"]
    ),
    reverse=True,
)[:TEST_BATCH_SIZE]

test_samples = [
    tokenized_train_dataset[index]
    for index in longest_indices
]

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

test_batch = data_collator(test_samples)
test_batch = {
    key: value.to(DEVICE)
    for key, value in test_batch.items()
}

model.to(DEVICE)
model.train()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

use_bf16 = torch.cuda.is_bf16_supported()

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
    enabled=use_bf16,
):
    outputs = model(**test_batch)
    loss = outputs.loss

loss.backward()

peak_memory_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("Device:", next(model.parameters()).device)
print("BF16 supported:", use_bf16)
print("Batch size:", TEST_BATCH_SIZE)
print("Input shape:", tuple(test_batch["input_ids"].shape))
print("Loss:", round(loss.item(), 4))
print("Peak allocated VRAM:", round(peak_memory_gb, 2), "GB")

model.zero_grad(set_to_none=True)

del test_batch, test_samples, outputs, loss
gc.collect()
torch.cuda.empty_cache()

Device: cuda:0
BF16 supported: True
Batch size: 4
Input shape: (4, 512)
Loss: 1.5137
Peak allocated VRAM: 0.77 GB


<div dir="rtl" align="right">

## آزمایش یک گام آموزشی واقعی با Batch Size برابر ۱۶

در این مرحله، یک Batch شامل ۱۶ متن با طول کامل ۵۱۲ Token روی GPU اجرا می‌شود.

برخلاف آزمایش قبلی، این بار Optimizer نیز ساخته می‌شود و مراحل کامل یک گام آموزشی انجام می‌شوند:

1. اجرای Forward
2. محاسبه Loss
3. محاسبه Gradientها با Backward
4. به‌روزرسانی وزن‌های مدل با Optimizer

هدف، اندازه‌گیری واقع‌بینانه‌تر مصرف حافظه GPU و بررسی امکان استفاده از Batch Size برابر ۱۶ در آموزش اصلی است.

این مرحله فقط یک گام آزمایشی است و آموزش اصلی مدل محسوب نمی‌شود.

</div>

In [13]:
import gc

import torch
from torch.optim import AdamW

TRAIN_TEST_BATCH_SIZE = 16
LEARNING_RATE = 2e-5

# انتخاب ۱۶ نمونه دارای طول کامل ۵۱۲ توکن
full_length_indices = [
    index
    for index in range(len(tokenized_train_dataset))
    if len(tokenized_train_dataset[index]["input_ids"]) == MAX_LENGTH
][:TRAIN_TEST_BATCH_SIZE]

training_test_samples = [
    tokenized_train_dataset[index]
    for index in full_length_indices
]

training_test_batch = data_collator(training_test_samples)

training_test_batch = {
    key: value.to(DEVICE)
    for key, value in training_test_batch.items()
}

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
)

model.train()
model.zero_grad(set_to_none=True)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

with torch.autocast(
    device_type="cuda",
    dtype=torch.bfloat16,
    enabled=True,
):
    training_test_outputs = model(**training_test_batch)
    training_test_loss = training_test_outputs.loss

training_test_loss.backward()
optimizer.step()
optimizer.zero_grad(set_to_none=True)

peak_memory_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

reserved_memory_gb = (
    torch.cuda.max_memory_reserved()
    / 1024**3
)

print("Batch size:", TRAIN_TEST_BATCH_SIZE)
print(
    "Input shape:",
    tuple(training_test_batch["input_ids"].shape),
)
print("Loss:", round(training_test_loss.item(), 4))
print(
    "Peak allocated VRAM:",
    round(peak_memory_gb, 2),
    "GB",
)
print(
    "Peak reserved VRAM:",
    round(reserved_memory_gb, 2),
    "GB",
)

del (
    training_test_batch,
    training_test_samples,
    training_test_outputs,
    training_test_loss,
    optimizer,
)

model.zero_grad(set_to_none=True)
gc.collect()
torch.cuda.empty_cache()

Batch size: 16
Input shape: (16, 512)
Loss: 1.5791
Peak allocated VRAM: 1.94 GB
Peak reserved VRAM: 2.39 GB


<div dir="rtl" align="right">

## بازنشانی مدل DistilBERT پیش از آموزش

در آزمایش حافظه قبلی، یک گام کامل آموزشی روی مدل اجرا شد و وزن‌های آن به‌صورت جزئی تغییر کردند.

در این مرحله، مدل آزمایشی از حافظه حذف شده و نسخه اصلی ازپیش‌آموزش‌دیده دوباره بارگذاری می‌شود تا Fine-tuning واقعی از یک وضعیت تمیز و قابل‌تکرار آغاز شود.

لایه طبقه‌بندی پنج‌کلاسه نیز دوباره با وزن‌های تصادفی جدید ساخته می‌شود.

</div>

In [14]:
import gc

import torch
from transformers import AutoModelForSequenceClassification

# حذف مدل تغییرکرده در آزمایش حافظه
model.to("cpu")
del model

gc.collect()
torch.cuda.empty_cache()

# تنظیم Seed برای مقداردهی تکرارپذیر لایه طبقه‌بندی
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print("Model reloaded successfully.")
print("Model class:", model.__class__.__name__)
print("Current device:", next(model.parameters()).device)
print("Number of labels:", model.config.num_labels)
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 32709.23it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model reloaded successfully.
Model class: DistilBertForSequenceClassification
Current device: cpu
Number of labels: 5
Allocated GPU memory: 0.02 GB


<div dir="rtl" align="right">

## تعریف معیارهای ارزیابی مدل

در این مرحله، تابع محاسبه معیارهای ارزیابی مدل تعریف می‌شود.

معیار اصلی پروژه `Micro F1` است. علاوه بر آن، معیارهای زیر نیز برای تحلیل دقیق‌تر عملکرد مدل محاسبه می‌شوند:

- `Accuracy`
- `Macro F1` برای بررسی عملکرد متوازن‌تر روی کلاس‌های ۱ تا ۵
- `MAE` برای اندازه‌گیری میانگین فاصله امتیاز پیش‌بینی‌شده از امتیاز واقعی
- `Within ±1` برای محاسبه درصد پیش‌بینی‌هایی که حداکثر یک ستاره با مقدار واقعی فاصله دارند

در پایان، تابع با یک مثال کوچک آزمایش می‌شود تا از صحت عملکرد آن مطمئن شویم.

</div>

In [15]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
)
from transformers import EvalPrediction


def compute_metrics(eval_prediction):
    logits = eval_prediction.predictions
    true_labels = eval_prediction.label_ids

    predicted_labels = np.argmax(logits, axis=-1)

    return {
        "micro_f1": f1_score(
            true_labels,
            predicted_labels,
            average="micro",
        ),
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels,
        ),
        "macro_f1": f1_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),
        "mae": mean_absolute_error(
            true_labels,
            predicted_labels,
        ),
        "within_1": np.mean(
            np.abs(true_labels - predicted_labels) <= 1
        ),
    }


# آزمایش تابع با داده ساختگی
test_logits = np.array([
    [5.0, 1.0, 0.0, 0.0, 0.0],
    [0.0, 4.0, 1.0, 0.0, 0.0],
    [0.0, 1.0, 2.0, 4.0, 0.0],
    [0.0, 0.0, 0.0, 1.0, 5.0],
    [0.0, 0.0, 0.0, 4.0, 1.0],
])

test_true_labels = np.array([0, 1, 2, 4, 4])

test_prediction = EvalPrediction(
    predictions=test_logits,
    label_ids=test_true_labels,
)

test_metrics = compute_metrics(test_prediction)

print("Metric function test:")
for metric_name, metric_value in test_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

Metric function test:
micro_f1: 0.6000
accuracy: 0.6000
macro_f1: 0.5333
mae: 0.4000
within_1: 1.0000


<div dir="rtl" align="right">

## پیکربندی آموزش آزمایشی DistilBERT

در این مرحله، تنظیمات Fine-tuning مدل `DistilBERT` و شیء `Trainer` ساخته می‌شوند، اما هنوز آموزش آغاز نمی‌شود.

تنظیمات اصلی آزمایش:

- دو Epoch آموزش روی ۵۰٬۰۰۰ نمونه
- Batch Size آموزش برابر ۱۶
- Batch Size ارزیابی برابر ۳۲
- استفاده از محاسبات `BF16`
- نرخ یادگیری `2e-5`
- ارزیابی و ذخیره مدل در پایان هر Epoch
- انتخاب بهترین Checkpoint بر اساس `Micro F1`
- نگهداری حداکثر دو Checkpoint برای جلوگیری از مصرف بیش از حد فضای ذخیره‌سازی

این تنظیمات به‌عنوان مبنای مقایسه مدل‌های Transformer بعدی استفاده خواهند شد.

</div>

In [16]:
from pathlib import Path

from transformers import Trainer, TrainingArguments

PILOT_OUTPUT_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "distilbert_base_uncased_pilot"
)

PILOT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

training_args = TrainingArguments(
    output_dir=str(PILOT_OUTPUT_DIR),

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    bf16=True,
    tf32=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer created successfully.")
print("Output directory:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print(
    "Train batch size:",
    training_args.per_device_train_batch_size,
)
print(
    "Evaluation batch size:",
    training_args.per_device_eval_batch_size,
)
print("BF16:", training_args.bf16)
print("TF32:", training_args.tf32)
print("Evaluation strategy:", training_args.eval_strategy)
print("Best-model metric:", training_args.metric_for_best_model)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer created successfully.
Output directory: /home/ali/GoogleDrive/AI_Project2/models/nlp/distilbert_base_uncased_pilot
Epochs: 2
Train batch size: 16
Evaluation batch size: 32
BF16: True
TF32: True
Evaluation strategy: IntervalStrategy.EPOCH
Best-model metric: micro_f1
Training samples: 50000
Validation samples: 10000


<div dir="rtl" align="right">

## اصلاح تنظیمات Warmup آموزش

در نسخه فعلی کتابخانه Transformers، پارامتر `warmup_ratio` منسوخ شده است. برای جلوگیری از هشدار و حفظ همان Warmup ده‌درصدی، تعداد گام‌های Warmup مستقیماً روی ۶۲۵ تنظیم می‌شود.

پس از اصلاح تنظیمات، شیء `Trainer` دوباره ساخته می‌شود. هنوز آموزش مدل آغاز نخواهد شد.

</div>

In [17]:
from transformers import Trainer, TrainingArguments

WARMUP_STEPS = 625

training_args = TrainingArguments(
    output_dir=str(PILOT_OUTPUT_DIR),

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    bf16=True,
    tf32=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer recreated successfully.")
print("Warmup steps:", training_args.warmup_steps)
print("Epochs:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Best-model metric:", training_args.metric_for_best_model)

Trainer recreated successfully.
Warmup steps: 625
Epochs: 2
Train batch size: 16
Best-model metric: micro_f1


<div dir="rtl" align="right">

## آموزش آزمایشی مدل DistilBERT

در این مرحله، مدل `DistilBERT` به‌مدت دو Epoch روی ۵۰٬۰۰۰ نمونه آموزشی Fine-tune می‌شود.

در پایان هر Epoch، عملکرد مدل روی ۱۰٬۰۰۰ نمونه اعتبارسنجی محاسبه و یک Checkpoint ذخیره می‌شود. بهترین Checkpoint براساس معیار اصلی پروژه، یعنی `Micro F1`، انتخاب خواهد شد.

برای جلوگیری از ادامه ناخواسته یک اجرای قبلی، آموزش از ابتدا آغاز می‌شود.

</div>

In [18]:
import time

import torch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_minutes = (
    time.perf_counter() - training_start_time
) / 60

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\nTraining completed.")
print(
    "Training time:",
    round(training_elapsed_minutes, 2),
    "minutes",
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Final training loss:",
    round(train_result.training_loss, 4),
)

Epoch,Training Loss,Validation Loss,Micro F1,Accuracy,Macro F1,Mae,Within 1
1,0.691306,0.655729,0.729200,0.729200,0.576906,0.327800,0.955600
2,0.533517,0.627293,0.751100,0.751100,0.615374,0.299100,0.962200


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.47it/s]



Training completed.
Training time: 9.71 minutes
Peak allocated VRAM: 2.5 GB
Best checkpoint: /home/ali/GoogleDrive/AI_Project2/models/nlp/distilbert_base_uncased_pilot/checkpoint-6250
Best validation Micro F1: 0.7511
Final training loss: 0.6786


<div dir="rtl" align="right">

## ثبت نتیجه آزمایش DistilBERT

در این مرحله، مشخصات و معیارهای حاصل از اولین آزمایش Transformer در یک فایل CSV ذخیره می‌شوند.

این فایل در آزمایش مدل‌های بعدی به‌روزرسانی خواهد شد تا مدل‌ها با شرایط یکسان از نظر `Micro F1`، زمان آموزش و مصرف حافظه GPU مقایسه شوند.

</div>

In [19]:
from pathlib import Path

import pandas as pd

RESULTS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "transformer_pilot_results.csv"
)

RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

distilbert_result = {
    "model_name": MODEL_NAME,
    "train_samples": len(tokenized_train_dataset),
    "validation_samples": len(tokenized_validation_dataset),
    "max_length": MAX_LENGTH,
    "epochs": int(training_args.num_train_epochs),
    "train_batch_size": training_args.per_device_train_batch_size,
    "learning_rate": training_args.learning_rate,
    "micro_f1": trainer.state.best_metric,
    "macro_f1": 0.615374,
    "mae": 0.299100,
    "within_1": 0.962200,
    "training_minutes": training_elapsed_minutes,
    "peak_vram_gb": peak_training_vram_gb,
    "best_checkpoint": trainer.state.best_model_checkpoint,
}

results_df = pd.DataFrame([distilbert_result])

results_df.to_csv(
    RESULTS_PATH,
    index=False,
)

print("Results saved successfully.")
print("Path:", RESULTS_PATH)
print()
print(results_df.to_string(index=False))

Results saved successfully.
Path: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/transformer_pilot_results.csv

                        model_name  train_samples  validation_samples  max_length  epochs  train_batch_size  learning_rate  micro_f1  macro_f1    mae  within_1  training_minutes  peak_vram_gb                                                                            best_checkpoint
distilbert/distilbert-base-uncased          50000               10000         512       2                16        0.00002    0.7511  0.615374 0.2991    0.9622          9.705474      2.500323 /home/ali/GoogleDrive/AI_Project2/models/nlp/distilbert_base_uncased_pilot/checkpoint-6250


<div dir="rtl" align="right">

## آماده‌سازی مدل دوم: DistilRoBERTa

در این مرحله، اشیای مربوط به مدل `DistilBERT` از حافظه RAM و GPU حذف می‌شوند تا فضای کافی برای آزمایش مدل دوم فراهم شود.

سپس فقط Tokenizer مدل `DistilRoBERTa-base` بارگذاری و بررسی می‌شود. هنوز مدل اصلی بارگذاری یا آموزش داده نخواهد شد.

</div>

In [20]:
import gc

import torch
from transformers import AutoTokenizer

# حذف اشیای سنگین مربوط به مدل قبلی
objects_to_remove = [
    "trainer",
    "model",
    "train_result",
    "tokenizer",
    "data_collator",
    "tokenized_train_dataset",
    "tokenized_validation_dataset",
    "pilot_train_dataset",
    "pilot_validation_dataset",
]

for object_name in objects_to_remove:
    globals().pop(object_name, None)

gc.collect()
torch.cuda.empty_cache()

# مدل دوم
MODEL_NAME = "distilbert/distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = (
    "Great product, but the battery stopped working after two days."
)

encoded_sample = tokenizer(
    sample_text,
    truncation=True,
    max_length=512,
)

print("Model:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)
print("Separator token:", tokenizer.sep_token)
print("Sample token count:", len(encoded_sample["input_ids"]))
print("Allocated GPU memory:", round(
    torch.cuda.memory_allocated() / 1024**3,
    2,
), "GB")

Model: distilbert/distilroberta-base
Tokenizer class: RobertaTokenizer
Model max length: 512
Separator token: </s>
Sample token count: 14
Allocated GPU memory: 0.02 GB


<div dir="rtl" align="right">

## اصلاح جداسازی عنوان و متن Review

در مرحله قبل، عبارت `[SEP]` به‌اشتباه به‌صورت یک الگوی Regex تفسیر شد. در این نسخه، جداکننده به‌عنوان متن ثابت در نظر گرفته می‌شود تا عنوان و متن Review از محل صحیح جدا شوند.

سپس همان ۵۰٬۰۰۰ نمونه آموزش و ۱۰٬۰۰۰ نمونه اعتبارسنجی با Tokenizer مدل `DistilRoBERTa` و حداکثر طول ۵۱۲ Token پردازش می‌شوند.

</div>

In [23]:
from datasets import Dataset

TEXT_SEPARATOR = " [SEP] "
MAX_LENGTH = 512


def create_text_pair_dataframe(dataframe):
    text_parts = dataframe["text"].str.split(
        TEXT_SEPARATOR,
        n=1,
        expand=True,
        regex=False,
    )

    if text_parts.shape[1] != 2 or text_parts[1].isna().any():
        failed_rows = (
            text_parts.shape[0]
            if text_parts.shape[1] != 2
            else int(text_parts[1].isna().sum())
        )
        raise ValueError(
            f"{failed_rows} texts could not be separated."
        )

    paired_dataframe = dataframe[["label"]].copy()
    paired_dataframe["summary"] = text_parts[0].fillna("")
    paired_dataframe["review"] = text_parts[1].fillna("")

    return paired_dataframe[
        ["summary", "review", "label"]
    ]


pilot_train_pairs = create_text_pair_dataframe(
    pilot_train_df,
)

pilot_validation_pairs = create_text_pair_dataframe(
    pilot_validation_df,
)

pilot_train_dataset = Dataset.from_pandas(
    pilot_train_pairs,
    preserve_index=False,
)

pilot_validation_dataset = Dataset.from_pandas(
    pilot_validation_pairs,
    preserve_index=False,
)


def tokenize_roberta_batch(batch):
    return tokenizer(
        batch["summary"],
        batch["review"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized_train_dataset = pilot_train_dataset.map(
    tokenize_roberta_batch,
    batched=True,
    remove_columns=["summary", "review"],
    desc="Tokenizing DistilRoBERTa train data",
)

tokenized_validation_dataset = pilot_validation_dataset.map(
    tokenize_roberta_batch,
    batched=True,
    remove_columns=["summary", "review"],
    desc="Tokenizing DistilRoBERTa validation data",
)

first_input_ids = tokenized_train_dataset[0]["input_ids"]
first_tokens = tokenizer.convert_ids_to_tokens(first_input_ids)

print("Tokenized train samples:", len(tokenized_train_dataset))
print(
    "Tokenized validation samples:",
    len(tokenized_validation_dataset),
)
print("Dataset columns:", tokenized_train_dataset.column_names)
print("First sample token count:", len(first_input_ids))
print("Separator token:", tokenizer.sep_token)
print(
    "Separator occurrences:",
    first_tokens.count(tokenizer.sep_token),
)
print("First 30 tokens:")
print(first_tokens[:30])

Tokenizing DistilRoBERTa validation data: 100%|██████████| 10000/10000 [00:00<00:00, 22487.42 examples/s]

Tokenized train samples: 50000
Tokenized validation samples: 10000
Dataset columns: ['label', 'input_ids', 'attention_mask']
First sample token count: 60
Separator token: </s>
Separator occurrences: 3
First 30 tokens:
['<s>', 'Great', 'Ġproduct', '.', 'ĠFantastic', 'Ġcustomer', 'Ġsupport', '</s>', '</s>', 'This', 'Ġworks', 'Ġgreat', '.', 'ĠIf', 'Ġyou', 'Ġhave', 'Ġan', 'Ġolder', 'Ġcable', 'Ġbox', 'Ġissues', 'Ġmay', 'Ġcome', 'Ġup', '.', 'ĠThe', 'Ġeasy', 'Ġfix', 'Ġis', ':']


<div dir="rtl" align="right">

## بارگذاری DistilRoBERTa برای طبقه‌بندی پنج‌کلاسه

در این مرحله، مدل ازپیش‌آموزش‌دیده `DistilRoBERTa-base` بارگذاری می‌شود و یک لایه طبقه‌بندی جدید با پنج خروجی به انتهای آن اضافه می‌شود.

هر خروجی متناظر با یکی از امتیازهای ۱ تا ۵ است. لایه طبقه‌بندی جدید در مرحله Fine-tuning روی داده‌های پروژه آموزش خواهد دید.

برای اینکه مقایسه با DistilBERT قابل‌تکرار باشد، Seed همان مقدار ۴۲ تنظیم می‌شود. هنوز آموزش مدل آغاز نمی‌شود.

</div>

In [26]:
import torch
from transformers import AutoModelForSequenceClassification

NUM_LABELS = 5

id2label = {
    0: "1_star",
    1: "2_stars",
    2: "3_stars",
    3: "4_stars",
    4: "5_stars",
}

label2id = {
    label_name: label_id
    for label_id, label_name in id2label.items()
}

torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model:", MODEL_NAME)
print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Current device:", next(model.parameters()).device)

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 20915.61it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert/distilroberta-base
Model class: RobertaForSequenceClassification
Number of labels: 5
Total parameters: 82,122,245
Trainable parameters: 82,122,245
Current device: cpu


<div dir="rtl" align="right">

## پیکربندی آموزش آزمایشی DistilRoBERTa

در این مرحله، تنظیمات Fine-tuning مدل `DistilRoBERTa-base` و شیء `Trainer` ساخته می‌شوند، اما هنوز آموزش آغاز نمی‌شود.

برای مقایسه منصفانه با DistilBERT، تمام شرایط آزمایش ثابت باقی می‌مانند:

- همان ۵۰٬۰۰۰ نمونه آموزش
- همان ۱۰٬۰۰۰ نمونه اعتبارسنجی
- حداکثر طول ورودی ۵۱۲ Token
- دو Epoch آموزش
- Batch Size آموزش برابر ۱۶
- نرخ یادگیری `2e-5`
- تعداد ۶۲۵ گام Warmup
- انتخاب بهترین Checkpoint براساس `Micro F1`

</div>

In [27]:
from pathlib import Path

from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

PILOT_OUTPUT_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "distilroberta_base_pilot"
)

PILOT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(PILOT_OUTPUT_DIR),

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=625,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    bf16=True,
    tf32=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("DistilRoBERTa Trainer created successfully.")
print("Output directory:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Evaluation batch size:", training_args.per_device_eval_batch_size)
print("Warmup steps:", training_args.warmup_steps)
print("Best-model metric:", training_args.metric_for_best_model)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))

DistilRoBERTa Trainer created successfully.
Output directory: /home/ali/GoogleDrive/AI_Project2/models/nlp/distilroberta_base_pilot
Epochs: 2
Train batch size: 16
Evaluation batch size: 32
Warmup steps: 625
Best-model metric: micro_f1
Training samples: 50000
Validation samples: 10000


<div dir="rtl" align="right">

## آموزش آزمایشی مدل DistilRoBERTa

در این مرحله، مدل `DistilRoBERTa-base` به‌مدت دو Epoch روی همان ۵۰٬۰۰۰ نمونه آموزشی Fine-tune می‌شود.

در پایان هر Epoch، مدل روی همان ۱۰٬۰۰۰ نمونه اعتبارسنجی ارزیابی خواهد شد. بهترین Checkpoint براساس معیار `Micro F1` انتخاب می‌شود تا نتیجه آن با DistilBERT قابل مقایسه باشد.

آموزش از ابتدا آغاز می‌شود و هیچ Checkpoint قبلی ادامه داده نخواهد شد.

</div>

In [28]:
import time

import torch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_minutes = (
    time.perf_counter() - training_start_time
) / 60

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\nTraining completed.")
print(
    "Training time:",
    round(training_elapsed_minutes, 2),
    "minutes",
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Final training loss:",
    round(train_result.training_loss, 4),
)

Epoch,Training Loss,Validation Loss,Micro F1,Accuracy,Macro F1,Mae,Within 1
1,0.656064,0.620013,0.745400,0.745400,0.592757,0.309700,0.957700
2,0.517267,0.614210,0.760800,0.760800,0.634244,0.281300,0.968300


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s]



Training completed.
Training time: 9.89 minutes
Peak allocated VRAM: 2.71 GB
Best checkpoint: /home/ali/GoogleDrive/AI_Project2/models/nlp/distilroberta_base_pilot/checkpoint-6250
Best validation Micro F1: 0.7608
Final training loss: 0.6585


<div dir="rtl" align="right">

## ثبت نتیجه آزمایش DistilRoBERTa

در این مرحله، نتیجه مدل `DistilRoBERTa-base` به فایل مقایسه آزمایش‌های Transformer اضافه می‌شود.

رکورد قبلی DistilBERT حفظ خواهد شد و فایل شامل نتایج هر دو مدل می‌شود تا مقایسه آن‌ها براساس معیارهای اصلی، زمان آموزش و مصرف حافظه GPU امکان‌پذیر باشد.

</div>

In [29]:
import pandas as pd

distilroberta_result = {
    "model_name": MODEL_NAME,
    "train_samples": len(tokenized_train_dataset),
    "validation_samples": len(tokenized_validation_dataset),
    "max_length": MAX_LENGTH,
    "epochs": int(training_args.num_train_epochs),
    "train_batch_size": training_args.per_device_train_batch_size,
    "learning_rate": training_args.learning_rate,
    "micro_f1": trainer.state.best_metric,
    "macro_f1": 0.634244,
    "mae": 0.281300,
    "within_1": 0.968300,
    "training_minutes": training_elapsed_minutes,
    "peak_vram_gb": peak_training_vram_gb,
    "best_checkpoint": trainer.state.best_model_checkpoint,
}

results_df = pd.read_csv(RESULTS_PATH)

results_df = results_df[
    results_df["model_name"] != MODEL_NAME
].copy()

results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([distilroberta_result]),
    ],
    ignore_index=True,
)

results_df = results_df.sort_values(
    by="micro_f1",
    ascending=False,
).reset_index(drop=True)

results_df.to_csv(
    RESULTS_PATH,
    index=False,
)

print("DistilRoBERTa result saved successfully.")
print("Path:", RESULTS_PATH)
print()
print(
    results_df[
        [
            "model_name",
            "micro_f1",
            "macro_f1",
            "mae",
            "within_1",
            "training_minutes",
            "peak_vram_gb",
        ]
    ].to_string(index=False)
)

DistilRoBERTa result saved successfully.
Path: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/transformer_pilot_results.csv

                        model_name  micro_f1  macro_f1    mae  within_1  training_minutes  peak_vram_gb
     distilbert/distilroberta-base    0.7608  0.634244 0.2813    0.9683          9.891078      2.706133
distilbert/distilbert-base-uncased    0.7511  0.615374 0.2991    0.9622          9.705474      2.500323


<div dir="rtl" align="right">

## آماده‌سازی مدل سوم: DeBERTa-v3-xsmall

در این مرحله، اشیای سنگین مربوط به مدل `DistilRoBERTa` از حافظه RAM و GPU حذف می‌شوند.

سپس فقط Tokenizer مدل `DeBERTa-v3-xsmall` بارگذاری و روی یک جفت عنوان و متن نمونه آزمایش می‌شود.

این مدل نسخه کوچک‌تر خانواده DeBERTa-v3 است و پیش از آزمایش `DeBERTa-v3-small` بررسی خواهد شد. هنوز مدل اصلی بارگذاری یا آموزش داده نمی‌شود.

</div>

In [30]:
import gc

import torch
from transformers import AutoTokenizer

# حذف اشیای سنگین مدل قبلی
objects_to_remove = [
    "trainer",
    "model",
    "train_result",
    "tokenizer",
    "data_collator",
    "tokenized_train_dataset",
    "tokenized_validation_dataset",
    "pilot_train_dataset",
    "pilot_validation_dataset",
]

for object_name in objects_to_remove:
    globals().pop(object_name, None)

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "microsoft/deberta-v3-xsmall"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

sample_summary = "Great product"
sample_review = (
    "The battery stopped working after only two days."
)

encoded_sample = tokenizer(
    sample_summary,
    sample_review,
    truncation=True,
    max_length=512,
)

sample_tokens = tokenizer.convert_ids_to_tokens(
    encoded_sample["input_ids"]
)

print("Model:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)
print("Separator token:", tokenizer.sep_token)
print("Sample token count:", len(encoded_sample["input_ids"]))
print("Sample tokens:", sample_tokens)
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

Model: microsoft/deberta-v3-xsmall
Tokenizer class: DebertaV2Tokenizer
Model max length: 1000000000000000019884624838656
Separator token: [SEP]
Sample token count: 14
Sample tokens: ['[CLS]', '▁Great', '▁product', '[SEP]', '▁The', '▁battery', '▁stopped', '▁working', '▁after', '▁only', '▁two', '▁days', '.', '[SEP]']
Allocated GPU memory: 0.02 GB


<div dir="rtl" align="right">

## Tokenization داده‌های آزمایشی با DeBERTa-v3-xsmall

در این مرحله، همان ۵۰٬۰۰۰ نمونه آموزش و ۱۰٬۰۰۰ نمونه اعتبارسنجی با Tokenizer مدل `DeBERTa-v3-xsmall` پردازش می‌شوند.

عنوان و متن Review به‌صورت دو ورودی جداگانه به Tokenizer داده می‌شوند تا جداکننده استاندارد مدل، یعنی `[SEP]`، به‌طور خودکار میان آن‌ها قرار گیرد.

با توجه به تنظیمات رسمی مدل، حداکثر طول ورودی به‌صورت صریح روی ۵۱۲ Token قرار می‌گیرد. هنوز مدل اصلی بارگذاری یا آموزش داده نمی‌شود.

</div>

In [31]:
from datasets import Dataset

MAX_LENGTH = 512

# اصلاح مقدار نامشخص ثبت‌شده در تنظیمات Tokenizer
tokenizer.model_max_length = MAX_LENGTH

pilot_train_dataset = Dataset.from_pandas(
    pilot_train_pairs,
    preserve_index=False,
)

pilot_validation_dataset = Dataset.from_pandas(
    pilot_validation_pairs,
    preserve_index=False,
)


def tokenize_deberta_batch(batch):
    return tokenizer(
        batch["summary"],
        batch["review"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenized_train_dataset = pilot_train_dataset.map(
    tokenize_deberta_batch,
    batched=True,
    remove_columns=["summary", "review"],
    desc="Tokenizing DeBERTa-v3-xsmall train data",
)

tokenized_validation_dataset = pilot_validation_dataset.map(
    tokenize_deberta_batch,
    batched=True,
    remove_columns=["summary", "review"],
    desc="Tokenizing DeBERTa-v3-xsmall validation data",
)

first_input_ids = tokenized_train_dataset[0]["input_ids"]
first_tokens = tokenizer.convert_ids_to_tokens(first_input_ids)

print("Configured maximum length:", tokenizer.model_max_length)
print("Tokenized train samples:", len(tokenized_train_dataset))
print(
    "Tokenized validation samples:",
    len(tokenized_validation_dataset),
)
print("Dataset columns:", tokenized_train_dataset.column_names)
print("First sample token count:", len(first_input_ids))
print("Separator token:", tokenizer.sep_token)
print(
    "Separator occurrences:",
    first_tokens.count(tokenizer.sep_token),
)
print("First 30 tokens:")
print(first_tokens[:30])

Tokenizing DeBERTa-v3-xsmall validation data: 100%|██████████| 10000/10000 [00:00<00:00, 18777.10 examples/s]

Configured maximum length: 512
Tokenized train samples: 50000
Tokenized validation samples: 10000
Dataset columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
First sample token count: 60
Separator token: [SEP]
Separator occurrences: 2
First 30 tokens:
['[CLS]', '▁Great', '▁product', '.', '▁Fantastic', '▁customer', '▁support', '[SEP]', '▁This', '▁works', '▁great', '.', '▁If', '▁you', '▁have', '▁an', '▁older', '▁cable', '▁box', '▁issues', '▁may', '▁come', '▁up', '.', '▁The', '▁easy', '▁fix', '▁is', ':', '▁for']


<div dir="rtl" align="right">

## بارگذاری DeBERTa-v3-xsmall برای طبقه‌بندی پنج‌کلاسه

در این مرحله، مدل ازپیش‌آموزش‌دیده `DeBERTa-v3-xsmall` بارگذاری می‌شود و یک لایه طبقه‌بندی جدید با پنج خروجی به انتهای مدل اضافه می‌شود.

هر خروجی متناظر با یکی از امتیازهای ۱ تا ۵ است. وزن‌های لایه طبقه‌بندی جدید در ابتدا تصادفی هستند و در مرحله Fine-tuning روی داده‌های پروژه آموزش خواهند دید.

برای قابل‌تکرار بودن آزمایش، Seed روی مقدار ۴۲ تنظیم می‌شود. هنوز آموزش مدل آغاز نمی‌شود.

</div>

In [32]:
import torch
from transformers import AutoModelForSequenceClassification

NUM_LABELS = 5

id2label = {
    0: "1_star",
    1: "2_stars",
    2: "3_stars",
    3: "4_stars",
    4: "5_stars",
}

label2id = {
    label_name: label_id
    for label_id, label_name in id2label.items()
}

torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model:", MODEL_NAME)
print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("Maximum position embeddings:", model.config.max_position_embeddings)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Current device:", next(model.parameters()).device)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 78695.37it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifi

Model: microsoft/deberta-v3-xsmall
Model class: DebertaV2ForSequenceClassification
Number of labels: 5
Maximum position embeddings: 512
Total parameters: 70,831,877
Trainable parameters: 70,831,877
Current device: cpu


<div dir="rtl" align="right">

## پیکربندی آموزش آزمایشی DeBERTa-v3-xsmall

در این مرحله، تنظیمات Fine-tuning مدل `DeBERTa-v3-xsmall` و شیء `Trainer` ساخته می‌شوند، اما هنوز آموزش آغاز نمی‌شود.

برای مقایسه منصفانه با مدل‌های قبلی، شرایط آزمایش ثابت باقی می‌مانند:

- همان ۵۰٬۰۰۰ نمونه آموزش
- همان ۱۰٬۰۰۰ نمونه اعتبارسنجی
- حداکثر طول ورودی ۵۱۲ Token
- دو Epoch آموزش
- Batch Size آموزش برابر ۱۶
- نرخ یادگیری `2e-5`
- تعداد ۶۲۵ گام Warmup
- انتخاب بهترین Checkpoint براساس `Micro F1`

</div>

In [33]:
from pathlib import Path

from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

PILOT_OUTPUT_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_xsmall_pilot"
)

PILOT_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(PILOT_OUTPUT_DIR),

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=625,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    bf16=True,
    tf32=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("DeBERTa-v3-xsmall Trainer created successfully.")
print("Output directory:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Evaluation batch size:", training_args.per_device_eval_batch_size)
print("Warmup steps:", training_args.warmup_steps)
print("Best-model metric:", training_args.metric_for_best_model)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))

DeBERTa-v3-xsmall Trainer created successfully.
Output directory: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_xsmall_pilot
Epochs: 2
Train batch size: 16
Evaluation batch size: 32
Warmup steps: 625
Best-model metric: micro_f1
Training samples: 50000
Validation samples: 10000


<div dir="rtl" align="right">

## آموزش آزمایشی مدل DeBERTa-v3-xsmall

در این مرحله، مدل `DeBERTa-v3-xsmall` به‌مدت دو Epoch روی همان ۵۰٬۰۰۰ نمونه آموزشی Fine-tune می‌شود.

در پایان هر Epoch، مدل روی همان ۱۰٬۰۰۰ نمونه اعتبارسنجی ارزیابی خواهد شد. بهترین Checkpoint براساس معیار `Micro F1` انتخاب می‌شود تا نتیجه با مدل‌های قبلی قابل مقایسه باشد.

آموزش از ابتدا آغاز می‌شود و هیچ Checkpoint قبلی ادامه داده نخواهد شد.

</div>

In [35]:
import time

import torch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_minutes = (
    time.perf_counter() - training_start_time
) / 60

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\nTraining completed.")
print(
    "Training time:",
    round(training_elapsed_minutes, 2),
    "minutes",
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Final training loss:",
    round(train_result.training_loss, 4),
)

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

<div dir="rtl" align="right">

## بازیابی نتیجه اجرای کامل DeBERTa-v3-xsmall

به‌دلیل شروع ناخواسته آموزش دوم، وزن‌های مدل موجود در حافظه دیگر نماینده اجرای اصلی نیستند.

در این مرحله، مدل مستقیماً از Checkpoint پایان اجرای کامل اول، یعنی `checkpoint-6250`، بازیابی می‌شود و دوباره روی مجموعه اعتبارسنجی آزمایشی ارزیابی خواهد شد.

به این ترتیب، نتیجه اجرای نیمه‌کاره دوم هیچ تأثیری بر مقایسه مدل‌ها نخواهد داشت.

</div>

In [38]:
from pathlib import Path

from transformers import AutoModelForSequenceClassification

FIRST_RUN_CHECKPOINT = (
    Path(PILOT_OUTPUT_DIR) / "checkpoint-6250"
)

if not FIRST_RUN_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {FIRST_RUN_CHECKPOINT}"
    )

model = AutoModelForSequenceClassification.from_pretrained(
    str(FIRST_RUN_CHECKPOINT)
)

trainer.model = model

verification_metrics = trainer.evaluate()

print("Recovered checkpoint:", FIRST_RUN_CHECKPOINT)
print("\nValidation metrics:")

for metric_name, metric_value in verification_metrics.items():
    if isinstance(metric_value, float):
        print(f"{metric_name}: {metric_value:.6f}")
    else:
        print(f"{metric_name}: {metric_value}")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 10505.52it/s]


RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

<div dir="rtl" align="right">

## پاک‌سازی اجرای DeBERTa-v3-xsmall

به‌دلیل شروع ناخواسته اجرای دوم، تمام Checkpointها و اشیای مربوط به آزمایش `DeBERTa-v3-xsmall` حذف می‌شوند.

این پاک‌سازی فقط روی مدل `DeBERTa-v3-xsmall` اثر دارد و نتایج و Checkpointهای `DistilBERT` و `DistilRoBERTa` را تغییر نمی‌دهد.

پس از این مرحله، آزمایش مدل سوم دوباره از وزن‌های اصلی ازپیش‌آموزش‌دیده آغاز خواهد شد.

</div>

In [39]:
import gc
import shutil
from pathlib import Path

import torch

DEBERTA_XSMALL_OUTPUT_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_xsmall_pilot"
)

# حذف اشیای مربوط به اجرای فعلی از RAM و GPU
objects_to_remove = [
    "trainer",
    "verification_trainer",
    "model",
    "train_result",
    "verification_metrics",
    "training_args",
    "data_collator",
]

for object_name in objects_to_remove:
    globals().pop(object_name, None)

gc.collect()
torch.cuda.empty_cache()

# حذف تمام Checkpointهای اجرای DeBERTa-v3-xsmall
if DEBERTA_XSMALL_OUTPUT_DIR.exists():
    shutil.rmtree(DEBERTA_XSMALL_OUTPUT_DIR)
    print("Deleted:", DEBERTA_XSMALL_OUTPUT_DIR)
else:
    print("Directory was already absent:", DEBERTA_XSMALL_OUTPUT_DIR)

print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)
print("DeBERTa-v3-xsmall experiment is ready to restart.")

Deleted: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_xsmall_pilot
Allocated GPU memory: 0.42 GB
DeBERTa-v3-xsmall experiment is ready to restart.


<div dir="rtl" align="right">

## بارگذاری مجدد مدل DeBERTa-v3-xsmall

پس از حذف اجرای قبلی و تمام Checkpointهای آن، مدل `DeBERTa-v3-xsmall` دوباره از وزن‌های اصلی ازپیش‌آموزش‌دیده بارگذاری می‌شود.

یک لایه طبقه‌بندی جدید با پنج خروجی برای امتیازهای ۱ تا ۵ ساخته خواهد شد. Seed نیز روی مقدار ۴۲ قرار می‌گیرد تا مقداردهی اولیه قابل‌تکرار باشد.

هنوز آموزش مدل آغاز نمی‌شود.

</div>

In [40]:
import torch
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-xsmall"
NUM_LABELS = 5

id2label = {
    0: "1_star",
    1: "2_stars",
    2: "3_stars",
    3: "4_stars",
    4: "5_stars",
}

label2id = {
    label_name: label_id
    for label_id, label_name in id2label.items()
}

torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print("Fresh model loaded successfully.")
print("Model:", MODEL_NAME)
print("Model class:", model.__class__.__name__)
print("Current device:", next(model.parameters()).device)
print(
    "Total parameters:",
    f"{sum(parameter.numel() for parameter in model.parameters()):,}",
)
print(
    "Output directory exists:",
    DEBERTA_XSMALL_OUTPUT_DIR.exists(),
)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 75230.74it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifi

Fresh model loaded successfully.
Model: microsoft/deberta-v3-xsmall
Model class: DebertaV2ForSequenceClassification
Current device: cpu
Total parameters: 70,831,877
Output directory exists: False


<div dir="rtl" align="right">

## ساخت مجدد Trainer برای DeBERTa-v3-xsmall

در این مرحله، پوشه خروجی و شیء `Trainer` مدل `DeBERTa-v3-xsmall` دوباره ساخته می‌شوند.

تمام تنظیمات با آزمایش قبلی و مدل‌های دیگر یکسان باقی می‌مانند تا مقایسه منصفانه باشد:

- ۵۰٬۰۰۰ نمونه آموزش
- ۱۰٬۰۰۰ نمونه اعتبارسنجی
- دو Epoch
- طول ورودی ۵۱۲ Token
- Batch Size آموزش برابر ۱۶
- نرخ یادگیری `2e-5`
- انتخاب بهترین Checkpoint براساس `Micro F1`

هنوز آموزش آغاز نمی‌شود.

</div>

In [41]:
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

DEBERTA_XSMALL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(DEBERTA_XSMALL_OUTPUT_DIR),

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=625,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    bf16=True,
    tf32=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Fresh Trainer created successfully.")
print("Output directory:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print("Existing checkpoints:", list(DEBERTA_XSMALL_OUTPUT_DIR.glob("checkpoint-*")))

Fresh Trainer created successfully.
Output directory: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_xsmall_pilot
Epochs: 2
Train batch size: 16
Training samples: 50000
Validation samples: 10000
Existing checkpoints: []


<div dir="rtl" align="right">

## آموزش تمیز مدل DeBERTa-v3-xsmall

در این مرحله، مدل `DeBERTa-v3-xsmall` به‌مدت دو Epoch روی ۵۰٬۰۰۰ نمونه آزمایشی Fine-tune می‌شود.

در پایان هر Epoch، مدل روی ۱۰٬۰۰۰ نمونه اعتبارسنجی ارزیابی شده و بهترین Checkpoint براساس `Micro F1` انتخاب می‌شود.

برای جلوگیری از اجرای ناخواسته مجدد، اگر Checkpointی از این آموزش وجود داشته باشد، سلول به‌جای شروع دوباره با خطا متوقف خواهد شد.

</div>

In [42]:
import time

import torch

existing_checkpoints = list(
    DEBERTA_XSMALL_OUTPUT_DIR.glob("checkpoint-*")
)

if existing_checkpoints:
    raise RuntimeError(
        "Training has already produced checkpoints. "
        "Do not run this cell again."
    )

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_minutes = (
    time.perf_counter() - training_start_time
) / 60

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\nTraining completed.")
print(
    "Training time:",
    round(training_elapsed_minutes, 2),
    "minutes",
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Final training loss:",
    round(train_result.training_loss, 4),
)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Micro F1,Accuracy,Macro F1,Mae,Within 1
1,1.318110,1.278934,0.551600,0.551600,0.142202,0.976500,0.736500
2,1.262967,1.280121,0.551600,0.551600,0.142202,0.976500,0.736500


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.72it/s]



Training completed.
Training time: 25.21 minutes
Peak allocated VRAM: 6.07 GB
Best checkpoint: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_xsmall_pilot/checkpoint-3125
Best validation Micro F1: 0.5516
Final training loss: 1.289


<div dir="rtl" align="right">

## بررسی توزیع پیش‌بینی‌های DeBERTa-v3-xsmall

نتایج ارزیابی نشان می‌دهند که مدل احتمالاً بیشتر یا تمام نمونه‌ها را به کلاس اکثریت، یعنی امتیاز ۵، نسبت داده است.

در این مرحله، پیش‌بینی‌های بهترین Checkpoint روی مجموعه اعتبارسنجی استخراج می‌شوند و تعداد پیش‌بینی‌های مربوط به هر امتیاز بررسی خواهد شد.

این مرحله فقط برای تشخیص رفتار مدل است و هیچ آموزش جدیدی انجام نمی‌شود.

</div>

In [43]:
import numpy as np
import pandas as pd

prediction_output = trainer.predict(
    tokenized_validation_dataset
)

predicted_labels = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

true_labels = prediction_output.label_ids

prediction_distribution = pd.DataFrame({
    "rating": [1, 2, 3, 4, 5],
    "true_count": [
        int(np.sum(true_labels == label))
        for label in range(5)
    ],
    "predicted_count": [
        int(np.sum(predicted_labels == label))
        for label in range(5)
    ],
})

prediction_distribution["predicted_percent"] = (
    prediction_distribution["predicted_count"]
    / len(predicted_labels)
    * 100
).round(2)

print(prediction_distribution.to_string(index=False))
print()
print(
    "Unique predicted classes:",
    np.unique(predicted_labels).tolist(),
)

 rating  true_count  predicted_count  predicted_percent
      1         979                0                0.0
      2         688                0                0.0
      3         968                0                0.0
      4        1849                0                0.0
      5        5516            10000              100.0

Unique predicted classes: [4]


<div dir="rtl" align="right">

## بایگانی اجرای ناموفق DeBERTa-v3-xsmall

اجرای نخست مدل `DeBERTa-v3-xsmall` تمام نمونه‌های اعتبارسنجی را به کلاس ۵ نسبت داد. بنابراین این اجرا به‌عنوان یک آزمایش ناموفق بایگانی می‌شود و در انتخاب مدل برتر استفاده نخواهد شد.

پوشه Checkpointهای این اجرا تغییر نام داده می‌شود تا نتایج آن از بین نروند. سپس مدل دوباره از وزن‌های اصلی ازپیش‌آموزش‌دیده بارگذاری می‌شود تا آزمایش دوم از وضعیت کاملاً تمیز آغاز شود.

</div>

In [44]:
import gc
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification

FAILED_OUTPUT_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_xsmall_failed_lr2e5"
)

# آزادسازی اشیای اجرای ناموفق
for object_name in [
    "trainer",
    "model",
    "train_result",
    "prediction_output",
]:
    globals().pop(object_name, None)

gc.collect()
torch.cuda.empty_cache()

# بایگانی Checkpointهای اجرای ناموفق
if FAILED_OUTPUT_DIR.exists():
    raise FileExistsError(
        f"Archive directory already exists: {FAILED_OUTPUT_DIR}"
    )

if DEBERTA_XSMALL_OUTPUT_DIR.exists():
    DEBERTA_XSMALL_OUTPUT_DIR.rename(FAILED_OUTPUT_DIR)
    print("Failed run archived at:", FAILED_OUTPUT_DIR)
else:
    raise FileNotFoundError(
        f"Output directory not found: {DEBERTA_XSMALL_OUTPUT_DIR}"
    )

# بارگذاری دوباره مدل از وزن‌های اصلی
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

print("Fresh model loaded successfully.")
print("Model device:", next(model.parameters()).device)
print(
    "New output directory exists:",
    DEBERTA_XSMALL_OUTPUT_DIR.exists(),
)
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

Failed run archived at: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_xsmall_failed_lr2e5


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 64789.53it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
classifi

Fresh model loaded successfully.
Model device: cpu
New output directory exists: False
Allocated GPU memory: 0.42 GB


<div dir="rtl" align="right">

## پیکربندی آزمایش دوم DeBERTa-v3-xsmall

اجرای نخست مدل `DeBERTa-v3-xsmall` با نرخ یادگیری `2e-5` روی کلاس اکثریت فروپاشید و تمام نمونه‌ها را امتیاز ۵ پیش‌بینی کرد.

در این آزمایش، تنظیمات پیشنهادی Model Card رسمی این مدل استفاده می‌شوند:

- نرخ یادگیری `4.5e-5`
- سه Epoch آموزش
- تعداد ۱۰۰۰ گام Warmup
- همان داده‌های Pilot و طول ورودی ۵۱۲ Token
- انتخاب بهترین Checkpoint براساس `Micro F1`

این اجرا یک آزمایش اصلاحی است و هنوز آموزش آغاز نمی‌شود.

</div>

In [46]:
from pathlib import Path

from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

DEBERTA_XSMALL_TUNED_OUTPUT_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_xsmall_tuned_pilot"
)

DEBERTA_XSMALL_TUNED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(DEBERTA_XSMALL_TUNED_OUTPUT_DIR),

    num_train_epochs=3,
    learning_rate=4.5e-5,
    weight_decay=0.01,
    warmup_steps=1000,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    bf16=True,
    tf32=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Tuned DeBERTa Trainer created successfully.")
print("Output directory:", training_args.output_dir)
print("Learning rate:", training_args.learning_rate)
print("Epochs:", training_args.num_train_epochs)
print("Warmup steps:", training_args.warmup_steps)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print(
    "Existing checkpoints:",
    list(DEBERTA_XSMALL_TUNED_OUTPUT_DIR.glob("checkpoint-*")),
)

Tuned DeBERTa Trainer created successfully.
Output directory: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_xsmall_tuned_pilot
Learning rate: 4.5e-05
Epochs: 3
Warmup steps: 1000
Train batch size: 16
Training samples: 50000
Validation samples: 10000
Existing checkpoints: []


<div dir="rtl" align="right">

## آموزش اصلاح‌شده DeBERTa-v3-xsmall

در این مرحله، مدل `DeBERTa-v3-xsmall` با تنظیمات اصلاح‌شده به‌مدت سه Epoch روی داده‌های Pilot آموزش داده می‌شود.

تنظیمات اصلی این اجرا:

- نرخ یادگیری `4.5e-5`
- سه Epoch
- تعداد ۱۰۰۰ گام Warmup
- طول ورودی ۵۱۲ Token
- Batch Size برابر ۱۶
- انتخاب بهترین Checkpoint براساس `Micro F1`

این سلول دارای محافظ است و در صورت وجود Checkpoint، آموزش مجدد ناخواسته را آغاز نمی‌کند.

</div>

In [48]:
import time

import torch

existing_checkpoints = list(
    DEBERTA_XSMALL_TUNED_OUTPUT_DIR.glob("checkpoint-*")
)

if existing_checkpoints:
    raise RuntimeError(
        "Checkpoint already exists. Do not run this training cell again."
    )

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_minutes = (
    time.perf_counter() - training_start_time
) / 60

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

print("\nTraining completed.")
print(
    "Training time:",
    round(training_elapsed_minutes, 2),
    "minutes",
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Final training loss:",
    round(train_result.training_loss, 4),
)

RuntimeError: Checkpoint already exists. Do not run this training cell again.

<div dir="rtl" align="right">

## بازیابی محلی مدل پس از خطای Google Drive

آموزش مدل تا پایان Epoch سوم انجام شده است، اما هنگام ذخیره Checkpoint نهایی روی Google Drive خطای ورودی/خروجی رخ داده است.

در این مرحله، مدل موجود در حافظه، Tokenizer و وضعیت Trainer روی دیسک محلی سیستم ذخیره می‌شوند تا نتیجه آموزش از بین نرود.

هیچ آموزش جدیدی انجام نمی‌شود.

</div>

In [49]:
from pathlib import Path

LOCAL_RECOVERY_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_xsmall_tuned_epoch3_recovery"
)

LOCAL_RECOVERY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ذخیره وزن‌های فعلی مدل
trainer.model.save_pretrained(
    LOCAL_RECOVERY_DIR,
    safe_serialization=True,
)

# ذخیره Tokenizer
tokenizer.save_pretrained(
    LOCAL_RECOVERY_DIR,
)

# ذخیره تاریخچه و وضعیت Trainer
trainer.state.save_to_json(
    str(LOCAL_RECOVERY_DIR / "trainer_state.json")
)

saved_files = sorted(
    file_path.name
    for file_path in LOCAL_RECOVERY_DIR.iterdir()
)

print("Recovery completed successfully.")
print("Recovery directory:", LOCAL_RECOVERY_DIR)
print("Saved files:", saved_files)

eval_records = [
    record
    for record in trainer.state.log_history
    if "eval_micro_f1" in record
]

print("\nRecorded validation results:")
for record in eval_records:
    print({
        key: value
        for key, value in record.items()
        if key in {
            "epoch",
            "eval_loss",
            "eval_micro_f1",
            "eval_macro_f1",
            "eval_mae",
            "eval_within_1",
        }
    })

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.85it/s]

Recovery completed successfully.
Recovery directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_xsmall_tuned_epoch3_recovery
Saved files: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json']

Recorded validation results:
{'eval_loss': nan, 'eval_micro_f1': 0.0979, 'eval_macro_f1': 0.03566809363329994, 'eval_mae': 3.0235, 'eval_within_1': 0.1667, 'epoch': 1.0}
{'eval_loss': nan, 'eval_micro_f1': 0.0979, 'eval_macro_f1': 0.03566809363329994, 'eval_mae': 3.0235, 'eval_within_1': 0.1667, 'epoch': 2.0}
{'eval_loss': nan, 'eval_micro_f1': 0.0979, 'eval_macro_f1': 0.03566809363329994, 'eval_mae': 3.0235, 'eval_within_1': 0.1667, 'epoch': 3.0}


<div dir="rtl" align="right">

## ثبت نتیجه ناموفق DeBERTa-v3-xsmall

مدل `DeBERTa-v3-xsmall` در دو تنظیم مختلف روی یک کلاس فروپاشید و اجرای دوم نیز دارای Loss نامعتبر `NaN` بود.

در این مرحله، نتیجه اجرای اصلاح‌شده با وضعیت `failed` در فایل مقایسه ثبت می‌شود. این رکورد در رتبه‌بندی مدل‌های معتبر استفاده نخواهد شد و نتایج صحیح DistilBERT و DistilRoBERTa حفظ می‌شوند.

</div>

In [50]:
from pathlib import Path

import pandas as pd

RESULTS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "transformer_pilot_results.csv"
)

results_df = pd.read_csv(RESULTS_PATH)

# مدل‌های قبلی اجراهای معتبر بوده‌اند
if "status" not in results_df.columns:
    results_df["status"] = "valid"

if "notes" not in results_df.columns:
    results_df["notes"] = ""

failed_model_name = "microsoft/deberta-v3-xsmall"

# جلوگیری از ثبت چندباره
results_df = results_df[
    results_df["model_name"] != failed_model_name
].copy()

failed_result = {
    "model_name": failed_model_name,
    "train_samples": 50_000,
    "validation_samples": 10_000,
    "max_length": 512,
    "epochs": 3,
    "train_batch_size": 16,
    "learning_rate": 4.5e-5,
    "micro_f1": 0.0979,
    "macro_f1": 0.035668,
    "mae": 3.0235,
    "within_1": 0.1667,
    "training_minutes": None,
    "peak_vram_gb": None,
    "best_checkpoint": str(
        "/home/ali/projects/ai-bootcamp/codes/"
        "ai-bootcamp-DeepLearning/artifacts/nlp/"
        "deberta_v3_xsmall_tuned_epoch3_recovery"
    ),
    "status": "failed",
    "notes": (
        "Numerical divergence: eval_loss=NaN; "
        "prediction collapsed to one class."
    ),
}

results_df = pd.concat(
    [results_df, pd.DataFrame([failed_result])],
    ignore_index=True,
)

results_df.to_csv(
    RESULTS_PATH,
    index=False,
)

print("Failed experiment recorded.")
print("Path:", RESULTS_PATH)
print()
print(
    results_df[
        [
            "model_name",
            "micro_f1",
            "macro_f1",
            "status",
            "notes",
        ]
    ].to_string(index=False)
)

Failed experiment recorded.
Path: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/transformer_pilot_results.csv

                        model_name  micro_f1  macro_f1 status                                                                   notes
     distilbert/distilroberta-base    0.7608  0.634244  valid                                                                        
distilbert/distilbert-base-uncased    0.7511  0.615374  valid                                                                        
       microsoft/deberta-v3-xsmall    0.0979  0.035668 failed Numerical divergence: eval_loss=NaN; prediction collapsed to one class.


<div dir="rtl" align="right">

## پاک‌سازی DeBERTa-v3-xsmall از حافظه

آزمایش‌های مدل `DeBERTa-v3-xsmall` پایان یافته‌اند و نتیجه ناموفق آن ثبت شده است.

در این مرحله، فقط اشیای مربوط به مدل، Trainer و پیش‌بینی‌ها از حافظه حذف می‌شوند تا برای آزمایش مدل بعدی فضای کافی آزاد شود.

Checkpointها، فایل بازیابی و جدول نتایج روی دیسک حفظ خواهند شد.

</div>

In [51]:
import gc

import torch

objects_to_remove = [
    "trainer",
    "model",
    "train_result",
    "training_args",
    "data_collator",
    "prediction_output",
    "predicted_labels",
    "true_labels",
    "prediction_distribution",
]

removed_objects = []

for object_name in objects_to_remove:
    if object_name in globals():
        del globals()[object_name]
        removed_objects.append(object_name)

gc.collect()
torch.cuda.empty_cache()

print("Removed objects:", removed_objects)
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)
print(
    "Reserved GPU memory:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB",
)
print("Ready for the next transformer model.")

Removed objects: ['trainer', 'model', 'training_args', 'data_collator', 'predicted_labels', 'true_labels', 'prediction_distribution']
Allocated GPU memory: 0.02 GB
Reserved GPU memory: 0.04 GB
Ready for the next transformer model.


<div dir="rtl" align="right">

## بارگذاری مدل DeBERTa-v3-small

در این مرحله، Tokenizer و مدل `DeBERTa-v3-small` از وزن‌های اصلی ازپیش‌آموزش‌دیده بارگذاری می‌شوند.

یک لایه طبقه‌بندی جدید با پنج خروجی برای امتیازهای ۱ تا ۵ ساخته می‌شود. حداکثر طول ورودی نیز روی ۵۱۲ Token تنظیم خواهد شد.

به‌دلیل خطای قبلی Google Drive، مسیر Checkpointهای این آزمایش روی دیسک محلی پروژه قرار می‌گیرد. هنوز آموزش یا Tokenization داده‌ها آغاز نمی‌شود.

</div>

In [52]:
import gc
from pathlib import Path

import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

MODEL_NAME = "microsoft/deberta-v3-small"

DEBERTA_SMALL_OUTPUT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_pilot"
)

if (
    DEBERTA_SMALL_OUTPUT_DIR.exists()
    and any(DEBERTA_SMALL_OUTPUT_DIR.iterdir())
):
    raise FileExistsError(
        f"Output directory is not empty: "
        f"{DEBERTA_SMALL_OUTPUT_DIR}"
    )

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
)

tokenizer.model_max_length = MAX_LENGTH

torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Fresh model and tokenizer loaded successfully.")
print("Model:", MODEL_NAME)
print("Model class:", model.__class__.__name__)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Tokenizer separator:", repr(tokenizer.sep_token))
print("Maximum length:", tokenizer.model_max_length)
print("Current device:", next(model.parameters()).device)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Local output directory:", DEBERTA_SMALL_OUTPUT_DIR)
print(
    "Output directory exists:",
    DEBERTA_SMALL_OUTPUT_DIR.exists(),
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 79035.47it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING  

Fresh model and tokenizer loaded successfully.
Model: microsoft/deberta-v3-small
Model class: DebertaV2ForSequenceClassification
Tokenizer class: DebertaV2Tokenizer
Tokenizer separator: '[SEP]'
Maximum length: 512
Current device: cpu
Total parameters: 141,898,757
Trainable parameters: 141,898,757
Local output directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_pilot
Output directory exists: False


<div dir="rtl" align="right">

## Tokenization داده‌های Pilot برای DeBERTa-v3-small

هر مدل باید داده‌ها را با Tokenizer مخصوص خودش پردازش کند. در این مرحله، عنوان و متن Review به‌صورت یک جفت ورودی به Tokenizer مدل `DeBERTa-v3-small` داده می‌شوند.

تنظیمات پردازش:

- حداکثر طول ورودی: ۵۱۲ Token
- حذف بخش‌های اضافه با Truncation
- نگه‌داری ستون Label
- عدم Padding ثابت؛ Padding پویا هنگام ساخت Batch انجام خواهد شد

هنوز آموزش مدل آغاز نمی‌شود.

</div>

In [53]:
from datasets import Dataset

# ساخت Datasetهای تازه برای جلوگیری از استفاده از Cache مدل قبلی
deberta_small_train_dataset = Dataset.from_pandas(
    pilot_train_pairs[["summary", "review", "label"]],
    preserve_index=False,
)

deberta_small_validation_dataset = Dataset.from_pandas(
    pilot_validation_pairs[["summary", "review", "label"]],
    preserve_index=False,
)


def tokenize_deberta_small(batch):
    return tokenizer(
        batch["summary"],
        batch["review"],
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_train_dataset = deberta_small_train_dataset.map(
    tokenize_deberta_small,
    batched=True,
    num_proc=4,
    remove_columns=["summary", "review"],
    load_from_cache_file=False,
    desc="Tokenizing DeBERTa-v3-small train data",
)

tokenized_validation_dataset = deberta_small_validation_dataset.map(
    tokenize_deberta_small,
    batched=True,
    num_proc=4,
    remove_columns=["summary", "review"],
    load_from_cache_file=False,
    desc="Tokenizing DeBERTa-v3-small validation data",
)

first_sample = tokenized_train_dataset[0]

print("Tokenization completed successfully.")
print("Training samples:", len(tokenized_train_dataset))
print("Validation samples:", len(tokenized_validation_dataset))
print("Columns:", tokenized_train_dataset.column_names)
print("First sample token count:", len(first_sample["input_ids"]))
print(
    "SEP token occurrences:",
    first_sample["input_ids"].count(tokenizer.sep_token_id),
)
print("First sample label:", first_sample["label"])

Tokenizing DeBERTa-v3-small train data (num_proc=4): 100%|██████████| 50000/50000 [00:03<00:00, 16000.12 examples/s]
Tokenizing DeBERTa-v3-small validation data (num_proc=4): 100%|██████████| 10000/10000 [00:00<00:00, 10781.58 examples/s]


Tokenization completed successfully.
Training samples: 50000
Validation samples: 10000
Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
First sample token count: 60
SEP token occurrences: 2
First sample label: 4


<div dir="rtl" align="right">

## بررسی پایداری عددی DeBERTa-v3-small

دو اجرای قبلی خانواده DeBERTa دچار فروپاشی پیش‌بینی یا Loss نامعتبر شدند. بنابراین پیش از شروع آموزش، یک Batch کوچک در دو حالت محاسباتی بررسی می‌شود:

- دقت کامل `FP32`
- دقت ترکیبی `BF16`

در هر حالت، محدود بودن Loss و Logitها بررسی خواهد شد. این مرحله هیچ تغییری در وزن‌های مدل ایجاد نمی‌کند و آموزش محسوب نمی‌شود.

</div>

In [54]:
import gc

import torch
from transformers import DataCollatorWithPadding

sanity_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

sanity_samples = [
    tokenized_train_dataset[index]
    for index in range(8)
]

sanity_batch = sanity_collator(sanity_samples)

model.to("cuda")
model.eval()

sanity_batch = {
    key: value.to("cuda")
    for key, value in sanity_batch.items()
}

with torch.no_grad():
    fp32_output = model(**sanity_batch)

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.bfloat16,
    ):
        bf16_output = model(**sanity_batch)

print("Numerical sanity check completed.")
print()
print("FP32 loss:", float(fp32_output.loss))
print(
    "FP32 loss is finite:",
    bool(torch.isfinite(fp32_output.loss).item()),
)
print(
    "FP32 logits are finite:",
    bool(torch.isfinite(fp32_output.logits).all().item()),
)

print()
print("BF16 loss:", float(bf16_output.loss))
print(
    "BF16 loss is finite:",
    bool(torch.isfinite(bf16_output.loss).item()),
)
print(
    "BF16 logits are finite:",
    bool(torch.isfinite(bf16_output.logits).all().item()),
)

print()
print(
    "Maximum absolute FP32 logit:",
    round(float(fp32_output.logits.abs().max()), 6),
)
print(
    "Maximum absolute BF16 logit:",
    round(float(bf16_output.logits.abs().max()), 6),
)

# بازگرداندن مدل به CPU تا Trainer بعداً مدیریت Device را انجام دهد
model.to("cpu")

del fp32_output
del bf16_output
del sanity_batch
del sanity_samples

gc.collect()
torch.cuda.empty_cache()

print()
print(
    "Allocated GPU memory after cleanup:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

Numerical sanity check completed.

FP32 loss: 1.6504497528076172
FP32 loss is finite: True
FP32 logits are finite: True

BF16 loss: 1.6506879329681396
BF16 loss is finite: True
BF16 logits are finite: True

Maximum absolute FP32 logit: 0.380127
Maximum absolute BF16 logit: 0.380859

Allocated GPU memory after cleanup: 0.02 GB


<div dir="rtl" align="right">

## ساخت Trainer پایدار برای DeBERTa-v3-small

آزمایش اولیه نشان داد Loss و Logitهای مدل در شروع محدود و معتبر هستند. بااین‌حال، برای جلوگیری از ناپایداری‌هایی که در مدل قبلی خانواده DeBERTa مشاهده شد، آموزش با تنظیمات محافظه‌کارانه‌تری انجام می‌شود:

- محاسبات مدل با دقت `FP32`
- Batch Size واقعی برابر ۴
- چهار مرحله Gradient Accumulation
- Batch Size مؤثر برابر ۱۶
- نرخ یادگیری `2e-5`
- دو Epoch آموزش
- ۶۲۵ گام Warmup
- Gradient Clipping با حد ۱
- Gradient Checkpointing برای کاهش مصرف VRAM
- ذخیره Checkpointها روی دیسک محلی

در این مرحله فقط Trainer ساخته می‌شود و آموزش هنوز آغاز نخواهد شد.

</div>

In [55]:
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

DEBERTA_SMALL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

training_args = TrainingArguments(
    output_dir=str(DEBERTA_SMALL_OUTPUT_DIR),

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=625,
    max_grad_norm=1.0,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,

    bf16=False,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    logging_nan_inf_filter=False,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

effective_batch_size = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print("Stable DeBERTa-v3-small Trainer created.")
print("Output directory:", training_args.output_dir)
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Precision: FP32")
print("Physical train batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective train batch size:", effective_batch_size)
print("Gradient checkpointing:", training_args.gradient_checkpointing)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print(
    "Existing checkpoints:",
    list(DEBERTA_SMALL_OUTPUT_DIR.glob("checkpoint-*")),
)

Stable DeBERTa-v3-small Trainer created.
Output directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_pilot
Epochs: 2
Learning rate: 2e-05
Precision: FP32
Physical train batch size: 4
Gradient accumulation: 4
Effective train batch size: 16
Gradient checkpointing: True
Training samples: 50000
Validation samples: 10000
Existing checkpoints: []


<div dir="rtl" align="right">

## آموزش پایدار DeBERTa-v3-small

مدل `DeBERTa-v3-small` به‌مدت دو Epoch روی داده‌های Pilot آموزش داده می‌شود.

برای ایمن‌تر شدن اجرا:

- پیش از شروع، نبودن Checkpoint قبلی بررسی می‌شود.
- Loss هر ۱۰۰ گام کنترل می‌شود.
- اگر Loss به `NaN` یا `Inf` تبدیل شود، آموزش خودکار متوقف خواهد شد.
- Checkpointها روی دیسک محلی ذخیره می‌شوند.
- بهترین مدل براساس `Micro F1` انتخاب خواهد شد.

این سلول را فقط یک بار اجرا کنید.

</div>

In [56]:
import math
import time

import torch
from transformers import TrainerCallback


class StopOnNonFiniteLossCallback(TrainerCallback):
    """Stop training when a logged loss becomes NaN or Inf."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return control

        current_loss = float(logs["loss"])

        if not math.isfinite(current_loss):
            print(
                "\nNon-finite training loss detected:",
                current_loss,
            )
            print("Training will be stopped safely.")
            control.should_training_stop = True

        return control


existing_checkpoints = sorted(
    DEBERTA_SMALL_OUTPUT_DIR.glob("checkpoint-*")
)

if existing_checkpoints:
    raise RuntimeError(
        "Checkpoint already exists. "
        "Do not run this training cell again."
    )

trainer.add_callback(
    StopOnNonFiniteLossCallback()
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_minutes = (
    time.perf_counter() - training_start_time
) / 60

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)

eval_records = [
    record
    for record in trainer.state.log_history
    if "eval_micro_f1" in record
]

print("\nTraining process finished.")
print(
    "Training time:",
    round(training_elapsed_minutes, 2),
    "minutes",
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Completed optimizer steps:",
    trainer.state.global_step,
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Final average training loss:",
    round(train_result.training_loss, 6),
)

print("\nRecorded validation results:")
for record in eval_records:
    print({
        key: record[key]
        for key in [
            "epoch",
            "eval_loss",
            "eval_micro_f1",
            "eval_accuracy",
            "eval_macro_f1",
            "eval_mae",
            "eval_within_1",
        ]
        if key in record
    })

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Micro F1,Accuracy,Macro F1,Mae,Within 1
0,nan,nan,0.097900,0.097900,0.035668,3.023500,0.166700



Non-finite training loss detected: nan
Training will be stopped safely.


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.43it/s]



Training process finished.
Training time: 2.77 minutes
Peak allocated VRAM: 1.46 GB
Completed optimizer steps: 1200
Best checkpoint: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_pilot/checkpoint-1200
Best validation Micro F1: 0.0979
Final average training loss: nan

Recorded validation results:
{'epoch': 0.384, 'eval_loss': nan, 'eval_micro_f1': 0.0979, 'eval_accuracy': 0.0979, 'eval_macro_f1': 0.03566809363329994, 'eval_mae': 3.0235, 'eval_within_1': 0.1667}


<div dir="rtl" align="right">

## بررسی زمان شروع ناپایداری عددی

آموزش `DeBERTa-v3-small` پس از مشاهده Loss نامعتبر متوقف شد.

در این مرحله، تاریخچه ثبت‌شده Trainer بررسی می‌شود تا مشخص شود:

- اولین Loss نامعتبر در کدام گام ظاهر شده است.
- Lossهای قبل از آن چه مقادیری داشته‌اند.
- مقدار `Gradient Norm` هنگام شروع ناپایداری چقدر بوده است.
- نرخ یادگیری در آن لحظه چه مقداری بوده است.

هیچ آموزش جدیدی انجام نمی‌شود.

</div>

In [57]:
import math

import pandas as pd

training_log_records = []

for record in trainer.state.log_history:
    if "loss" not in record or "eval_loss" in record:
        continue

    loss_value = float(record["loss"])

    training_log_records.append({
        "step": record.get("step"),
        "epoch": record.get("epoch"),
        "loss": loss_value,
        "loss_is_finite": math.isfinite(loss_value),
        "grad_norm": record.get("grad_norm"),
        "learning_rate": record.get("learning_rate"),
    })

training_logs_df = pd.DataFrame(training_log_records)

print("Recorded training logs:")
print(training_logs_df.to_string(index=False))

non_finite_logs = training_logs_df[
    ~training_logs_df["loss_is_finite"]
]

print()

if non_finite_logs.empty:
    print("No non-finite training loss was found in log history.")
else:
    first_non_finite = non_finite_logs.iloc[0]

    print("First non-finite record:")
    print(first_non_finite.to_string())

Recorded training logs:
 step  epoch     loss  loss_is_finite  grad_norm  learning_rate
  100  0.032 6.014834            True  13.109375       0.000003
  200  0.064 5.161104            True  16.171875       0.000006
  300  0.096 5.266802            True  13.593750       0.000010
  400  0.128 5.202109            True   8.265625       0.000013
  500  0.160 5.124391            True  10.531250       0.000016
  600  0.192 5.285354            True  13.492188       0.000019
  700  0.224 5.439785            True  17.625000       0.000020
  800  0.256 5.275934            True   5.656250       0.000019
  900  0.288 5.220809            True  18.781250       0.000019
 1000  0.320 5.147171            True  11.351562       0.000019
 1100  0.352 5.158293            True  11.015625       0.000018
 1200  0.384      NaN           False        NaN       0.000018

First non-finite record:
step                  1200
epoch                0.384
loss                   NaN
loss_is_finite       False
grad_norm 

<div dir="rtl" align="right">

## بررسی سلامت وزن‌های DeBERTa-v3-small پس از توقف

Loss و Gradient Norm مدل بین گام‌های ۱۱۰۰ و ۱۲۰۰ به مقدار نامعتبر `NaN` تبدیل شدند.

در این مرحله موارد زیر بررسی می‌شوند:

- تنظیم Optimizer استفاده‌شده توسط Trainer
- وجود `NaN` یا `Inf` در پارامترهای مدل
- وجود مقادیر نامعتبر در Bufferهای مدل
- معتبر بودن خروجی مدل روی یک Batch کوچک

هیچ آموزش جدیدی انجام نمی‌شود و Checkpoint موجود نیز تغییر نمی‌کند.

</div>


In [58]:
import torch
import pandas as pd

model_to_check = trainer.model

print("Trainer numerical settings:")
print("Optimizer:", trainer.args.optim)
print("BF16:", trainer.args.bf16)
print("FP16:", trainer.args.fp16)
print("TF32:", trainer.args.tf32)
print(
    "Gradient checkpointing:",
    trainer.args.gradient_checkpointing,
)
print(
    "Current model device:",
    next(model_to_check.parameters()).device,
)

non_finite_records = []
total_non_finite_values = 0

tensors_to_check = [
    ("parameter", name, tensor)
    for name, tensor in model_to_check.named_parameters()
]

tensors_to_check += [
    ("buffer", name, tensor)
    for name, tensor in model_to_check.named_buffers()
]

for tensor_type, tensor_name, tensor in tensors_to_check:
    if not tensor.is_floating_point():
        continue

    detached_tensor = tensor.detach()
    non_finite_count = int(
        (~torch.isfinite(detached_tensor)).sum().item()
    )

    if non_finite_count > 0:
        total_non_finite_values += non_finite_count

        non_finite_records.append({
            "type": tensor_type,
            "name": tensor_name,
            "shape": tuple(detached_tensor.shape),
            "non_finite_count": non_finite_count,
            "total_values": detached_tensor.numel(),
        })

print()
print(
    "Tensors containing NaN or Inf:",
    len(non_finite_records),
)
print(
    "Total NaN or Inf values:",
    total_non_finite_values,
)

if non_finite_records:
    non_finite_df = pd.DataFrame(non_finite_records)

    print("\nFirst affected tensors:")
    print(
        non_finite_df.head(20).to_string(index=False)
    )
else:
    print("\nAll model parameters and buffers are finite.")

# بررسی Forward Pass فعلی مدل
diagnostic_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

diagnostic_samples = [
    tokenized_train_dataset[index]
    for index in range(8)
]

diagnostic_batch = diagnostic_collator(
    diagnostic_samples
)

model_device = next(
    model_to_check.parameters()
).device

diagnostic_batch = {
    key: value.to(model_device)
    for key, value in diagnostic_batch.items()
}

model_to_check.eval()

with torch.no_grad():
    diagnostic_output = model_to_check(
        **diagnostic_batch
    )

print()
print(
    "Diagnostic loss:",
    float(diagnostic_output.loss),
)
print(
    "Diagnostic loss is finite:",
    bool(
        torch.isfinite(
            diagnostic_output.loss
        ).item()
    ),
)
print(
    "Diagnostic logits are finite:",
    bool(
        torch.isfinite(
            diagnostic_output.logits
        ).all().item()
    ),
)

Trainer numerical settings:
Optimizer: OptimizerNames.ADAMW_TORCH_FUSED
BF16: False
FP16: False
TF32: True
Gradient checkpointing: True
Current model device: cuda:0

Tensors containing NaN or Inf: 106
Total NaN or Inf values: 141898757

First affected tensors:
     type                                                      name         shape  non_finite_count  total_values
parameter                 deberta.embeddings.word_embeddings.weight (128100, 768)          98380800      98380800
parameter                       deberta.embeddings.LayerNorm.weight        (768,)               768           768
parameter                         deberta.embeddings.LayerNorm.bias        (768,)               768           768
parameter  deberta.encoder.layer.0.attention.self.query_proj.weight    (768, 768)            589824        589824
parameter    deberta.encoder.layer.0.attention.self.query_proj.bias        (768,)               768           768
parameter    deberta.encoder.layer.0.attention.self.key

<div dir="rtl" align="right">

## بایگانی اجرای خراب و بارگذاری مجدد DeBERTa-v3-small

در اجرای قبلی، همه پارامترهای مدل پس از گام ۱۲۰۰ به مقادیر نامعتبر `NaN/Inf` تبدیل شدند. بنابراین Checkpoint این اجرا قابل استفاده نیست.

در این مرحله:

- پوشه اجرای خراب بایگانی می‌شود.
- اشیای مدل و Trainer از حافظه حذف می‌شوند.
- مدل `DeBERTa-v3-small` دوباره از وزن‌های اصلی بارگذاری می‌شود.
- داده‌های Tokenize‌شده حفظ می‌شوند.
- هنوز آموزش جدیدی آغاز نمی‌شود.

</div>

In [59]:
import gc
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification

FAILED_DEBERTA_SMALL_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_failed_fused_optimizer"
)

# آزادسازی اجرای خراب
for object_name in [
    "trainer",
    "model",
    "train_result",
    "training_args",
    "data_collator",
    "diagnostic_output",
    "diagnostic_batch",
    "model_to_check",
]:
    globals().pop(object_name, None)

gc.collect()
torch.cuda.empty_cache()

# بایگانی Checkpoint خراب
if FAILED_DEBERTA_SMALL_DIR.exists():
    raise FileExistsError(
        f"Archive directory already exists: "
        f"{FAILED_DEBERTA_SMALL_DIR}"
    )

if DEBERTA_SMALL_OUTPUT_DIR.exists():
    DEBERTA_SMALL_OUTPUT_DIR.rename(
        FAILED_DEBERTA_SMALL_DIR
    )
    print(
        "Failed run archived at:",
        FAILED_DEBERTA_SMALL_DIR,
    )
else:
    raise FileNotFoundError(
        f"Failed output directory not found: "
        f"{DEBERTA_SMALL_OUTPUT_DIR}"
    )

# بارگذاری مجدد مدل سالم
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

all_parameters_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
    if parameter.is_floating_point()
)

print("Fresh model loaded successfully.")
print("Model device:", next(model.parameters()).device)
print("All parameters are finite:", all_parameters_finite)
print(
    "New output directory exists:",
    DEBERTA_SMALL_OUTPUT_DIR.exists(),
)
print(
    "Allocated GPU memory:",
    round(
        torch.cuda.memory_allocated() / 1024**3,
        2,
    ),
    "GB",
)

Failed run archived at: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_failed_fused_optimizer


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 67203.74it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING  

Fresh model loaded successfully.
Model device: cpu
All parameters are finite: True
New output directory exists: False
Allocated GPU memory: 0.29 GB


<div dir="rtl" align="right">

## آزمون پایداری Optimizer غیرادغامی برای DeBERTa-v3-small

اجرای قبلی در حوالی گام ۱۲۰۰ دچار `NaN` شد و همه پارامترهای مدل را خراب کرد. محتمل‌ترین عامل، Optimizer ادغامی `ADAMW_TORCH_FUSED` بود.

در این آزمون:

- Optimizer از نوع `AdamW` معمولی و با `fused=False` ساخته می‌شود.
- سایر تنظیمات اصلی آموزش ثابت باقی می‌مانند.
- مدل تا گام ۱۳۰۰ آموزش داده می‌شود تا از نقطه خرابی قبلی عبور کند.
- نرخ یادگیری مطابق برنامه آموزش کامل ۶۲۵۰ گامی محاسبه می‌شود.
- هیچ ارزیابی یا Checkpointی انجام نمی‌شود.
- در صورت مشاهده Loss نامعتبر، اجرا خودکار متوقف خواهد شد.
- در پایان، سلامت همه پارامترهای مدل بررسی می‌شود.

این فقط یک آزمون پایداری است و نتیجه آن به‌عنوان مدل نهایی ذخیره نخواهد شد.

</div>

In [60]:
import math
import time
from pathlib import Path

import torch
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    get_linear_schedule_with_warmup,
)


class StopOnNonFiniteLossCallback(TrainerCallback):
    """Stop the smoke test when a logged loss becomes NaN or Inf."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return control

        current_loss = float(logs["loss"])

        if not math.isfinite(current_loss):
            print(
                "\nNon-finite training loss detected:",
                current_loss,
            )
            print("Smoke test will be stopped safely.")
            control.should_training_stop = True

        return control


SMOKE_TEST_OUTPUT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_nonfused_smoke_test"
)

SMOKE_TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

smoke_data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

# مشابه Trainer اصلی، روی Bias و LayerNorm وزن‌کاهی اعمال نمی‌شود.
decay_parameters = []
no_decay_parameters = []

for parameter_name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    is_no_decay = (
        parameter_name.endswith(".bias")
        or "LayerNorm.weight" in parameter_name
        or "layer_norm.weight" in parameter_name
    )

    if is_no_decay:
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)

smoke_optimizer = torch.optim.AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": 0.01,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    fused=False,
    foreach=False,
)

# برنامه نرخ یادگیری مطابق آموزش کامل دو Epoch است.
FULL_TRAINING_STEPS = 6250
SMOKE_TEST_STEPS = 1300

smoke_scheduler = get_linear_schedule_with_warmup(
    optimizer=smoke_optimizer,
    num_warmup_steps=625,
    num_training_steps=FULL_TRAINING_STEPS,
)

smoke_training_args = TrainingArguments(
    output_dir=str(SMOKE_TEST_OUTPUT_DIR),

    max_steps=SMOKE_TEST_STEPS,
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    bf16=False,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,

    eval_strategy="no",
    save_strategy="no",

    logging_strategy="steps",
    logging_steps=100,
    logging_first_step=True,
    logging_nan_inf_filter=False,

    report_to="none",
    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

smoke_trainer = Trainer(
    model=model,
    args=smoke_training_args,
    train_dataset=tokenized_train_dataset,
    processing_class=tokenizer,
    data_collator=smoke_data_collator,
    optimizers=(
        smoke_optimizer,
        smoke_scheduler,
    ),
    callbacks=[
        StopOnNonFiniteLossCallback(),
    ],
)

print("Optimizer:", smoke_optimizer.__class__.__name__)
print("Fused:", smoke_optimizer.defaults.get("fused"))
print("Foreach:", smoke_optimizer.defaults.get("foreach"))
print("Smoke-test steps:", SMOKE_TEST_STEPS)
print("Full scheduler steps:", FULL_TRAINING_STEPS)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

smoke_start_time = time.perf_counter()

smoke_train_result = smoke_trainer.train()

smoke_elapsed_minutes = (
    time.perf_counter() - smoke_start_time
) / 60

non_finite_parameter_values = 0
affected_parameter_tensors = 0

for parameter in smoke_trainer.model.parameters():
    if not parameter.is_floating_point():
        continue

    invalid_count = int(
        (~torch.isfinite(parameter.detach())).sum().item()
    )

    if invalid_count > 0:
        affected_parameter_tensors += 1
        non_finite_parameter_values += invalid_count

smoke_log_records = [
    {
        "step": record.get("step"),
        "epoch": record.get("epoch"),
        "loss": record.get("loss"),
        "grad_norm": record.get("grad_norm"),
        "learning_rate": record.get("learning_rate"),
    }
    for record in smoke_trainer.state.log_history
    if "loss" in record
]

print("\nSmoke test finished.")
print(
    "Completed optimizer steps:",
    smoke_trainer.state.global_step,
)
print(
    "Elapsed time:",
    round(smoke_elapsed_minutes, 2),
    "minutes",
)
print(
    "Final average loss:",
    smoke_train_result.training_loss,
)
print(
    "Affected parameter tensors:",
    affected_parameter_tensors,
)
print(
    "Non-finite parameter values:",
    non_finite_parameter_values,
)
print(
    "All parameters are finite:",
    non_finite_parameter_values == 0,
)
print(
    "Peak allocated VRAM:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2,
    ),
    "GB",
)

print("\nTraining logs:")
for record in smoke_log_records:
    print(record)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Optimizer: AdamW
Fused: False
Foreach: False
Smoke-test steps: 1300
Full scheduler steps: 6250


Step,Training Loss
1,6.147416
100,nan



Non-finite training loss detected: nan
Smoke test will be stopped safely.

Smoke test finished.
Completed optimizer steps: 100
Elapsed time: 0.21 minutes
Final average loss: nan
Affected parameter tensors: 106
Non-finite parameter values: 141898757
All parameters are finite: False
Peak allocated VRAM: 1.91 GB

Training logs:
{'step': 1, 'epoch': 0.00032, 'loss': 6.147416114807129, 'grad_norm': 18.734375, 'learning_rate': 0.0}
{'step': 100, 'epoch': 0.032, 'loss': nan, 'grad_norm': nan, 'learning_rate': 3.1680000000000004e-06}


<div dir="rtl" align="right">

## آزمون پایداری DeBERTa-v3-small بدون TF32 و Gradient Checkpointing

Optimizer غیرادغامی نیز نتوانست از خراب‌شدن پارامترهای مدل جلوگیری کند. بنابراین در این آزمایش، دو قابلیت محاسباتی مشترک با اجرای قبلی غیرفعال می‌شوند:

- غیرفعال‌سازی کامل `TF32`
- غیرفعال‌سازی `Gradient Checkpointing`
- استفاده از `AdamW` معمولی با `fused=False` و `foreach=False`
- Batch Size واقعی برابر ۲ و Gradient Accumulation برابر ۸
- استفاده از DataLoader تک‌پردازه
- اجرای فقط ۱۵۰ گام بدون ارزیابی و ذخیره Checkpoint

مدل ابتدا دوباره از وزن‌های اصلی بارگذاری می‌شود. در پایان نیز سلامت تمام پارامترها بررسی خواهد شد.

</div>

In [61]:
import gc
import math
import time
from pathlib import Path

import torch
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    get_linear_schedule_with_warmup,
)


# حذف کامل اشیای اجرای خراب
for object_name in [
    "smoke_trainer",
    "smoke_optimizer",
    "smoke_scheduler",
    "smoke_train_result",
    "smoke_training_args",
    "smoke_data_collator",
    "model",
]:
    globals().pop(object_name, None)

gc.collect()
torch.cuda.empty_cache()


# غیرفعال‌سازی صریح TF32
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")


# بارگذاری دوباره مدل سالم
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

model.gradient_checkpointing_disable()


class StopOnNonFiniteLossCallback(TrainerCallback):
    """Stop immediately when a logged loss becomes NaN or Inf."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return control

        current_loss = float(logs["loss"])

        if not math.isfinite(current_loss):
            print(
                "\nNon-finite training loss detected:",
                current_loss,
            )
            print("Diagnostic training will stop safely.")
            control.should_training_stop = True

        return control


SAFE_TEST_OUTPUT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_safe_150_step_test"
)

SAFE_TEST_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

safe_data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)


# گروه‌بندی پارامترها برای Weight Decay
decay_parameters = []
no_decay_parameters = []

for parameter_name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    no_decay = (
        parameter_name.endswith(".bias")
        or "LayerNorm.weight" in parameter_name
        or "layer_norm.weight" in parameter_name
    )

    if no_decay:
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)


safe_optimizer = torch.optim.AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": 0.01,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    fused=False,
    foreach=False,
)


# همان برنامه Warmup مربوط به آموزش کامل
FULL_TRAINING_STEPS = 6250
SAFE_TEST_STEPS = 150

safe_scheduler = get_linear_schedule_with_warmup(
    optimizer=safe_optimizer,
    num_warmup_steps=625,
    num_training_steps=FULL_TRAINING_STEPS,
)

safe_training_args = TrainingArguments(
    output_dir=str(SAFE_TEST_OUTPUT_DIR),

    max_steps=SAFE_TEST_STEPS,
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    bf16=False,
    fp16=False,
    tf32=False,
    gradient_checkpointing=False,

    eval_strategy="no",
    save_strategy="no",

    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    logging_nan_inf_filter=False,

    report_to="none",
    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=0,
    dataloader_pin_memory=True,
)

safe_trainer = Trainer(
    model=model,
    args=safe_training_args,
    train_dataset=tokenized_train_dataset,
    processing_class=tokenizer,
    data_collator=safe_data_collator,
    optimizers=(
        safe_optimizer,
        safe_scheduler,
    ),
    callbacks=[
        StopOnNonFiniteLossCallback(),
    ],
)

print("Fresh model loaded.")
print(
    "Initial parameters finite:",
    all(
        torch.isfinite(parameter).all().item()
        for parameter in model.parameters()
        if parameter.is_floating_point()
    ),
)
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32)
print("TF32 cuDNN:", torch.backends.cudnn.allow_tf32)
print(
    "Gradient checkpointing:",
    safe_training_args.gradient_checkpointing,
)
print("Optimizer fused:", safe_optimizer.defaults.get("fused"))
print("Optimizer foreach:", safe_optimizer.defaults.get("foreach"))
print("Physical batch size:", safe_training_args.per_device_train_batch_size)
print(
    "Effective batch size:",
    safe_training_args.per_device_train_batch_size
    * safe_training_args.gradient_accumulation_steps,
)
print("Test steps:", SAFE_TEST_STEPS)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

safe_start_time = time.perf_counter()

safe_train_result = safe_trainer.train()

safe_elapsed_minutes = (
    time.perf_counter() - safe_start_time
) / 60


affected_parameter_tensors = 0
non_finite_parameter_values = 0

for parameter in safe_trainer.model.parameters():
    if not parameter.is_floating_point():
        continue

    invalid_count = int(
        (~torch.isfinite(parameter.detach())).sum().item()
    )

    if invalid_count > 0:
        affected_parameter_tensors += 1
        non_finite_parameter_values += invalid_count


safe_logs = [
    {
        "step": record.get("step"),
        "epoch": record.get("epoch"),
        "loss": record.get("loss"),
        "grad_norm": record.get("grad_norm"),
        "learning_rate": record.get("learning_rate"),
    }
    for record in safe_trainer.state.log_history
    if "loss" in record
]


print("\nSafe diagnostic test finished.")
print(
    "Completed optimizer steps:",
    safe_trainer.state.global_step,
)
print(
    "Elapsed time:",
    round(safe_elapsed_minutes, 2),
    "minutes",
)
print(
    "Final average loss:",
    safe_train_result.training_loss,
)
print(
    "Affected parameter tensors:",
    affected_parameter_tensors,
)
print(
    "Non-finite parameter values:",
    non_finite_parameter_values,
)
print(
    "All parameters remain finite:",
    non_finite_parameter_values == 0,
)
print(
    "Peak allocated VRAM:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2,
    ),
    "GB",
)

print("\nTraining logs:")
for record in safe_logs:
    print(record)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 71457.99it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING  

Fresh model loaded.
Initial parameters finite: True
TF32 matmul: False
TF32 cuDNN: False
Gradient checkpointing: False
Optimizer fused: False
Optimizer foreach: False
Physical batch size: 2
Effective batch size: 16
Test steps: 150


Step,Training Loss
1,12.181993
10,nan



Non-finite training loss detected: nan
Diagnostic training will stop safely.

Safe diagnostic test finished.
Completed optimizer steps: 10
Elapsed time: 0.02 minutes
Final average loss: nan
Affected parameter tensors: 106
Non-finite parameter values: 141898757
All parameters remain finite: False
Peak allocated VRAM: 1.93 GB

Training logs:
{'step': 1, 'epoch': 0.00032, 'loss': 12.18199348449707, 'grad_norm': 37.4375, 'learning_rate': 0.0}
{'step': 10, 'epoch': 0.0032, 'loss': nan, 'grad_norm': nan, 'learning_rate': 2.8800000000000004e-07}


<div dir="rtl" align="right">

## بررسی نوع داده واقعی وزن‌های DeBERTa-v3-small

با وجود غیرفعال‌بودن Mixed Precision، مدل پس از چند گام دچار `NaN` شد. ممکن است وزن‌های مدل از ابتدا با دقت نیمه، یعنی `float16`، بارگذاری شده باشند.

در این مرحله موارد زیر بررسی می‌شوند:

- نوع داده پارامترهای مدل
- نوع داده Gradientها
- نوع داده Stateهای Optimizer
- میزان حافظه نظری وزن‌های مدل

هیچ آموزش یا تغییری در مدل انجام نمی‌شود.

</div>

In [62]:
from collections import Counter

import torch

model_to_inspect = safe_trainer.model

parameter_dtype_counts = Counter(
    str(parameter.dtype)
    for parameter in model_to_inspect.parameters()
)

gradient_dtype_counts = Counter(
    str(parameter.grad.dtype)
    for parameter in model_to_inspect.parameters()
    if parameter.grad is not None
)

optimizer_state_dtype_counts = Counter()

for optimizer_state in safe_optimizer.state.values():
    for state_value in optimizer_state.values():
        if torch.is_tensor(state_value):
            optimizer_state_dtype_counts[
                str(state_value.dtype)
            ] += 1

parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model_to_inspect.parameters()
)

first_parameter = next(
    model_to_inspect.parameters()
)

print("Model parameter dtypes:")
print(dict(parameter_dtype_counts))

print()
print("Gradient dtypes:")
print(dict(gradient_dtype_counts))

print()
print("Optimizer-state tensor dtypes:")
print(dict(optimizer_state_dtype_counts))

print()
print("First parameter dtype:", first_parameter.dtype)
print(
    "First parameter element size:",
    first_parameter.element_size(),
    "bytes",
)
print(
    "Total parameter storage:",
    round(parameter_bytes / 1024**2, 2),
    "MiB",
)
print(
    "Model config dtype:",
    getattr(model_to_inspect.config, "dtype", None),
)
print(
    "Model config torch_dtype:",
    getattr(model_to_inspect.config, "torch_dtype", None),
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Model parameter dtypes:
{'torch.float16': 106}

Gradient dtypes:
{}

Optimizer-state tensor dtypes:
{'torch.float32': 106, 'torch.float16': 212}

First parameter dtype: torch.float16
First parameter element size: 2 bytes
Total parameter storage: 270.65 MiB
Model config dtype: torch.float16
Model config torch_dtype: torch.float16


<div dir="rtl" align="right">

## بارگذاری صریح DeBERTa-v3-small با وزن‌های Float32

بررسی نوع داده پارامترها نشان داد که مدل با وزن‌های `float16` بارگذاری شده بود، درحالی‌که انتظار داشتیم آموزش کامل با دقت `float32` انجام شود.

در این مرحله:

- مدل خراب از حافظه حذف می‌شود.
- مدل دوباره با پارامتر `dtype=torch.float32` بارگذاری می‌شود.
- نوع داده و سلامت تمام پارامترها کنترل می‌شود.
- یک Forward Pass کوچک روی GPU انجام می‌شود تا معتبر بودن Loss و Logitها تأیید شود.

هنوز هیچ آموزشی آغاز نمی‌شود.

</div>

In [63]:
import gc
from collections import Counter

import torch
from transformers import AutoModelForSequenceClassification


# حذف اشیای آزمایش ناموفق و مدل خراب
for object_name in [
    "safe_trainer",
    "safe_optimizer",
    "safe_scheduler",
    "safe_train_result",
    "safe_training_args",
    "safe_data_collator",
    "model",
    "model_to_inspect",
]:
    globals().pop(object_name, None)

gc.collect()
torch.cuda.empty_cache()


# بارگذاری صریح تمام وزن‌ها با Float32
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.float32,
)

model.gradient_checkpointing_disable()


parameter_dtype_counts = Counter(
    str(parameter.dtype)
    for parameter in model.parameters()
)

all_parameters_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
    if parameter.is_floating_point()
)

parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
)


# Forward Pass تشخیصی روی GPU
diagnostic_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

diagnostic_samples = [
    tokenized_train_dataset[index]
    for index in range(8)
]

diagnostic_batch = diagnostic_collator(
    diagnostic_samples
)

model.to("cuda")
model.eval()

diagnostic_batch = {
    key: value.to("cuda")
    for key, value in diagnostic_batch.items()
}

with torch.no_grad():
    diagnostic_output = model(
        **diagnostic_batch
    )


print("Model reloaded explicitly in Float32.")
print("Parameter dtypes:", dict(parameter_dtype_counts))
print("Config dtype:", getattr(model.config, "dtype", None))
print("All parameters are finite:", all_parameters_finite)
print(
    "Total parameter storage:",
    round(parameter_bytes / 1024**2, 2),
    "MiB",
)

print()
print("Diagnostic loss:", float(diagnostic_output.loss))
print(
    "Diagnostic loss is finite:",
    bool(torch.isfinite(diagnostic_output.loss).item()),
)
print(
    "Diagnostic logits are finite:",
    bool(torch.isfinite(diagnostic_output.logits).all().item()),
)
print(
    "Maximum absolute logit:",
    round(
        float(diagnostic_output.logits.abs().max()),
        6,
    ),
)


# بازگرداندن مدل به CPU برای ساخت Trainer بعدی
model.to("cpu")

del diagnostic_output
del diagnostic_batch
del diagnostic_samples

gc.collect()
torch.cuda.empty_cache()

print()
print(
    "Allocated GPU memory after cleanup:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1549.08it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING   

Model reloaded explicitly in Float32.
Parameter dtypes: {'torch.float32': 106}
Config dtype: torch.float32
All parameters are finite: True
Total parameter storage: 541.3 MiB

Diagnostic loss: 1.5836145877838135
Diagnostic loss is finite: True
Diagnostic logits are finite: True
Maximum absolute logit: 0.270012

Allocated GPU memory after cleanup: 0.55 GB


<div dir="rtl" align="right">

## آزمون پایداری ۱۵۰ گامی DeBERTa-v3-small با وزن‌های Float32

در بررسی قبلی مشخص شد وزن‌های مدل به‌صورت پیش‌فرض با `float16` بارگذاری شده بودند و هنگام به‌روزرسانی مستقیم توسط Optimizer به مقادیر نامعتبر تبدیل می‌شدند.

اکنون مدل به‌صورت صریح با `float32` بارگذاری شده است. در این مرحله:

- مدل به‌مدت ۱۵۰ گام آموزش آزمایشی داده می‌شود.
- Optimizer معمولی `AdamW` با حالت‌های `fused=False` و `foreach=False` استفاده می‌شود.
- `TF32` و Gradient Checkpointing غیرفعال باقی می‌مانند.
- هیچ ارزیابی یا Checkpoint نهایی انجام نمی‌شود.
- Loss و Gradient Norm هر ۱۰ گام کنترل می‌شوند.
- در پایان، سلامت تمام پارامترهای مدل بررسی می‌شود.

این سلول دارای محافظ اجرای مجدد است.

</div>

In [64]:
import math
import time
from pathlib import Path

import torch
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    get_linear_schedule_with_warmup,
)


class StopOnNonFiniteFP32Callback(TrainerCallback):
    """Stop training when the logged loss becomes NaN or Inf."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or "loss" not in logs:
            return control

        current_loss = float(logs["loss"])

        if not math.isfinite(current_loss):
            print(
                "\nNon-finite training loss detected:",
                current_loss,
            )
            print("FP32 smoke test will stop safely.")
            control.should_training_stop = True

        return control


FP32_SMOKE_OUTPUT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_fp32_smoke_test"
)

START_MARKER = FP32_SMOKE_OUTPUT_DIR / ".started"
SUCCESS_MARKER = FP32_SMOKE_OUTPUT_DIR / ".completed"

FP32_SMOKE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if START_MARKER.exists() or SUCCESS_MARKER.exists():
    raise RuntimeError(
        "This FP32 smoke-test cell has already been started. "
        "Do not run it again."
    )

# کنترل مدل پیش از شروع
parameter_dtypes = {
    parameter.dtype
    for parameter in model.parameters()
}

if parameter_dtypes != {torch.float32}:
    raise TypeError(
        f"Expected only torch.float32 parameters, "
        f"but found: {parameter_dtypes}"
    )

if not all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
    if parameter.is_floating_point()
):
    raise RuntimeError(
        "The model already contains non-finite parameters."
    )

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

model.gradient_checkpointing_disable()

fp32_data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

# گروه‌بندی پارامترها برای Weight Decay
decay_parameters = []
no_decay_parameters = []

for parameter_name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    is_no_decay = (
        parameter_name.endswith(".bias")
        or "LayerNorm.weight" in parameter_name
        or "layer_norm.weight" in parameter_name
    )

    if is_no_decay:
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)

fp32_optimizer = torch.optim.AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": 0.01,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    fused=False,
    foreach=False,
)

FULL_TRAINING_STEPS = 6250
FP32_SMOKE_STEPS = 150

fp32_scheduler = get_linear_schedule_with_warmup(
    optimizer=fp32_optimizer,
    num_warmup_steps=625,
    num_training_steps=FULL_TRAINING_STEPS,
)

fp32_training_args = TrainingArguments(
    output_dir=str(FP32_SMOKE_OUTPUT_DIR),

    max_steps=FP32_SMOKE_STEPS,
    learning_rate=2e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,

    bf16=False,
    fp16=False,
    tf32=False,
    gradient_checkpointing=False,

    eval_strategy="no",
    save_strategy="no",

    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    logging_nan_inf_filter=False,

    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=0,
    dataloader_pin_memory=True,
)

fp32_smoke_trainer = Trainer(
    model=model,
    args=fp32_training_args,
    train_dataset=tokenized_train_dataset,
    processing_class=tokenizer,
    data_collator=fp32_data_collator,
    optimizers=(
        fp32_optimizer,
        fp32_scheduler,
    ),
    callbacks=[
        StopOnNonFiniteFP32Callback(),
    ],
)

START_MARKER.write_text(
    "FP32 smoke test started.",
    encoding="utf-8",
)

print("FP32 smoke test is ready.")
print("Parameter dtypes:", parameter_dtypes)
print("Optimizer:", fp32_optimizer.__class__.__name__)
print("Optimizer fused:", fp32_optimizer.defaults.get("fused"))
print("Optimizer foreach:", fp32_optimizer.defaults.get("foreach"))
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32)
print("TF32 cuDNN:", torch.backends.cudnn.allow_tf32)
print("Test steps:", FP32_SMOKE_STEPS)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

fp32_start_time = time.perf_counter()

fp32_train_result = fp32_smoke_trainer.train()

fp32_elapsed_minutes = (
    time.perf_counter() - fp32_start_time
) / 60

affected_parameter_tensors = 0
non_finite_parameter_values = 0

for parameter in fp32_smoke_trainer.model.parameters():
    if not parameter.is_floating_point():
        continue

    invalid_count = int(
        (~torch.isfinite(parameter.detach())).sum().item()
    )

    if invalid_count > 0:
        affected_parameter_tensors += 1
        non_finite_parameter_values += invalid_count

fp32_logs = [
    {
        "step": record.get("step"),
        "epoch": record.get("epoch"),
        "loss": record.get("loss"),
        "grad_norm": record.get("grad_norm"),
        "learning_rate": record.get("learning_rate"),
    }
    for record in fp32_smoke_trainer.state.log_history
    if "loss" in record
]

test_succeeded = (
    fp32_smoke_trainer.state.global_step
    == FP32_SMOKE_STEPS
    and non_finite_parameter_values == 0
    and all(
        record["loss"] is None
        or math.isfinite(float(record["loss"]))
        for record in fp32_logs
    )
)

if test_succeeded:
    SUCCESS_MARKER.write_text(
        "FP32 smoke test completed successfully.",
        encoding="utf-8",
    )

print("\nFP32 smoke test finished.")
print(
    "Completed optimizer steps:",
    fp32_smoke_trainer.state.global_step,
)
print(
    "Elapsed time:",
    round(fp32_elapsed_minutes, 2),
    "minutes",
)
print(
    "Final average loss:",
    fp32_train_result.training_loss,
)
print(
    "Affected parameter tensors:",
    affected_parameter_tensors,
)
print(
    "Non-finite parameter values:",
    non_finite_parameter_values,
)
print(
    "All parameters remain finite:",
    non_finite_parameter_values == 0,
)
print("Smoke test succeeded:", test_succeeded)
print(
    "Peak allocated VRAM:",
    round(
        torch.cuda.max_memory_allocated() / 1024**3,
        2,
    ),
    "GB",
)

print("\nTraining logs:")
for record in fp32_logs:
    print(record)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


FP32 smoke test is ready.
Parameter dtypes: {torch.float32}
Optimizer: AdamW
Optimizer fused: False
Optimizer foreach: False
TF32 matmul: False
TF32 cuDNN: False
Test steps: 150


Step,Training Loss
1,12.902241
10,12.284349
20,12.194421
30,12.193885
40,12.290057
50,11.956914
60,11.919342
70,11.949780
80,12.038023
90,11.705489



FP32 smoke test finished.
Completed optimizer steps: 150
Elapsed time: 0.55 minutes
Final average loss: 11.562313715616861
Affected parameter tensors: 0
Non-finite parameter values: 0
All parameters remain finite: True
Smoke test succeeded: True
Peak allocated VRAM: 3.63 GB

Training logs:
{'step': 1, 'epoch': 0.00032, 'loss': 12.902240753173828, 'grad_norm': 38.868560791015625, 'learning_rate': 0.0}
{'step': 10, 'epoch': 0.0032, 'loss': 12.284349229600695, 'grad_norm': 33.71928024291992, 'learning_rate': 2.8800000000000004e-07}
{'step': 20, 'epoch': 0.0064, 'loss': 12.19442138671875, 'grad_norm': 45.50725173950195, 'learning_rate': 6.08e-07}
{'step': 30, 'epoch': 0.0096, 'loss': 12.193885040283202, 'grad_norm': 34.63666534423828, 'learning_rate': 9.28e-07}
{'step': 40, 'epoch': 0.0128, 'loss': 12.290056610107422, 'grad_norm': 34.58019256591797, 'learning_rate': 1.248e-06}
{'step': 50, 'epoch': 0.016, 'loss': 11.95691375732422, 'grad_norm': 29.89174461364746, 'learning_rate': 1.568e-0

<div dir="rtl" align="right">

## بازنشانی مدل برای آموزش اصلی DeBERTa-v3-small

آزمون ۱۵۰ گامی تأیید کرد که مدل با وزن‌های `float32` بدون ایجاد `NaN` آموزش می‌بیند.

ازآنجاکه وزن‌های مدل در طول Smoke Test تغییر کرده‌اند، آن مدل نباید مبنای آزمایش اصلی قرار گیرد. در این مرحله:

- اشیای Smoke Test از حافظه حذف می‌شوند.
- مدل دوباره از وزن‌های اصلی بارگذاری می‌شود.
- نوع داده تمام پارامترها به‌صورت صریح `float32` خواهد بود.
- سلامت پارامترها و خالی‌بودن مسیر آموزش اصلی بررسی می‌شود.

هنوز آموزش اصلی آغاز نمی‌شود.

</div>

In [65]:
import gc
from collections import Counter

import torch
from transformers import AutoModelForSequenceClassification


# حذف اشیای Smoke Test و مدل تغییرکرده
objects_to_remove = [
    "fp32_smoke_trainer",
    "fp32_optimizer",
    "fp32_scheduler",
    "fp32_train_result",
    "fp32_training_args",
    "fp32_data_collator",
    "fp32_logs",
    "model",
]

removed_objects = []

for object_name in objects_to_remove:
    if object_name in globals():
        del globals()[object_name]
        removed_objects.append(object_name)

gc.collect()
torch.cuda.empty_cache()


# تنظیم Seed برای مقداردهی اولیه قابل‌تکرار لایه طبقه‌بندی
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)


# بارگذاری تازه مدل با وزن‌های صریح Float32
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.float32,
)

model.gradient_checkpointing_disable()


parameter_dtype_counts = Counter(
    str(parameter.dtype)
    for parameter in model.parameters()
)

all_parameters_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
    if parameter.is_floating_point()
)

existing_checkpoints = (
    sorted(DEBERTA_SMALL_OUTPUT_DIR.glob("checkpoint-*"))
    if DEBERTA_SMALL_OUTPUT_DIR.exists()
    else []
)


print("Smoke-test objects removed:", removed_objects)
print("Fresh FP32 model loaded successfully.")
print("Model device:", next(model.parameters()).device)
print("Parameter dtypes:", dict(parameter_dtype_counts))
print("Config dtype:", getattr(model.config, "dtype", None))
print("All parameters are finite:", all_parameters_finite)
print(
    "Main output directory exists:",
    DEBERTA_SMALL_OUTPUT_DIR.exists(),
)
print("Existing main checkpoints:", existing_checkpoints)
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1687.50it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING   

Smoke-test objects removed: ['fp32_smoke_trainer', 'fp32_optimizer', 'fp32_scheduler', 'fp32_train_result', 'fp32_training_args', 'fp32_data_collator', 'fp32_logs', 'model']
Fresh FP32 model loaded successfully.
Model device: cpu
Parameter dtypes: {'torch.float32': 106}
Config dtype: torch.float32
All parameters are finite: True
Main output directory exists: False
Existing main checkpoints: []
Allocated GPU memory: 1.0 GB


<div dir="rtl" align="right">

## ساخت Trainer اصلی DeBERTa-v3-small با وزن‌های Float32

پس از مشخص‌شدن علت ناپایداری، آموزش اصلی با مسیر محاسباتی پایدار پیکربندی می‌شود:

- وزن‌های مدل با دقت کامل `float32`
- Optimizer معمولی `AdamW` با `fused=False`
- Batch Size واقعی برابر ۲
- هشت مرحله Gradient Accumulation
- Batch Size مؤثر برابر ۱۶
- دو Epoch و مجموع ۶۲۵۰ گام به‌روزرسانی
- نرخ یادگیری `2e-5`
- تعداد ۶۲۵ گام Warmup
- غیرفعال‌بودن `TF32` و Gradient Checkpointing
- ارزیابی و ذخیره Checkpoint در پایان هر Epoch
- انتخاب بهترین Checkpoint براساس `Micro F1`
- ذخیره خروجی‌ها روی دیسک محلی

در این مرحله فقط Trainer ساخته می‌شود و آموزش هنوز آغاز نمی‌شود.

</div>

In [66]:
from pathlib import Path

import torch
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    get_linear_schedule_with_warmup,
)


# مسیر محلی آموزش اصلی
DEBERTA_SMALL_OUTPUT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_pilot"
)

if (
    DEBERTA_SMALL_OUTPUT_DIR.exists()
    and any(DEBERTA_SMALL_OUTPUT_DIR.iterdir())
):
    raise FileExistsError(
        f"Main output directory is not empty: "
        f"{DEBERTA_SMALL_OUTPUT_DIR}"
    )

DEBERTA_SMALL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# بررسی مجدد نوع داده و سلامت مدل
parameter_dtypes = {
    parameter.dtype
    for parameter in model.parameters()
}

if parameter_dtypes != {torch.float32}:
    raise TypeError(
        f"Expected only torch.float32 parameters, "
        f"but found: {parameter_dtypes}"
    )

if not all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
    if parameter.is_floating_point()
):
    raise RuntimeError(
        "The model contains non-finite parameters."
    )


# تنظیم مسیر محاسباتی پایدار
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

model.gradient_checkpointing_disable()


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)


# گروه‌بندی پارامترها برای Weight Decay
decay_parameters = []
no_decay_parameters = []

for parameter_name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    is_no_decay = (
        parameter_name.endswith(".bias")
        or "LayerNorm.weight" in parameter_name
        or "layer_norm.weight" in parameter_name
    )

    if is_no_decay:
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)


optimizer = torch.optim.AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": 0.01,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    fused=False,
    foreach=False,
)


TOTAL_TRAINING_STEPS = 6250
WARMUP_STEPS = 625

scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_TRAINING_STEPS,
)


training_args = TrainingArguments(
    output_dir=str(DEBERTA_SMALL_OUTPUT_DIR),

    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=1.0,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=16,

    bf16=False,
    fp16=False,
    tf32=False,
    gradient_checkpointing=False,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,
    logging_first_step=True,
    logging_nan_inf_filter=False,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=0,
    dataloader_pin_memory=True,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(
        optimizer,
        scheduler,
    ),
)


effective_batch_size = (
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)

print("Main FP32 Trainer created successfully.")
print("Output directory:", training_args.output_dir)
print("Parameter dtypes:", parameter_dtypes)
print("Optimizer:", optimizer.__class__.__name__)
print("Optimizer fused:", optimizer.defaults.get("fused"))
print("Optimizer foreach:", optimizer.defaults.get("foreach"))
print("TF32 matmul:", torch.backends.cuda.matmul.allow_tf32)
print("TF32 cuDNN:", torch.backends.cudnn.allow_tf32)
print("Epochs:", training_args.num_train_epochs)
print("Physical train batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective train batch size:", effective_batch_size)
print("Total optimizer steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Training samples:", len(trainer.train_dataset))
print("Validation samples:", len(trainer.eval_dataset))
print(
    "Existing checkpoints:",
    list(DEBERTA_SMALL_OUTPUT_DIR.glob("checkpoint-*")),
)

Main FP32 Trainer created successfully.
Output directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_pilot
Parameter dtypes: {torch.float32}
Optimizer: AdamW
Optimizer fused: False
Optimizer foreach: False
TF32 matmul: False
TF32 cuDNN: False
Epochs: 2
Physical train batch size: 2
Gradient accumulation: 8
Effective train batch size: 16
Total optimizer steps: 6250
Warmup steps: 625
Training samples: 50000
Validation samples: 10000
Existing checkpoints: []


<div dir="rtl" align="right">

## آموزش اصلی DeBERTa-v3-small با وزن‌های Float32

در این مرحله، مدل `DeBERTa-v3-small` به‌مدت دو Epoch روی داده‌های Pilot آموزش داده می‌شود.

ویژگی‌های ایمنی این اجرا:

- تأیید دوباره `float32` بودن تمام پارامترها پیش از شروع
- جلوگیری از اجرای ناخواسته مجدد با قفل روی دیسک
- کنترل Loss و Gradient Norm در طول آموزش
- توقف خودکار در صورت مشاهده `NaN` یا `Inf`
- بررسی سلامت همه پارامترها پس از پایان
- ذخیره Checkpointها روی دیسک محلی
- انتخاب بهترین Checkpoint براساس `Micro F1`

این سلول را فقط یک بار اجرا کنید.

</div>

In [67]:
import math
import time
from pathlib import Path

import torch
from transformers import TrainerCallback


class StopOnNonFiniteTrainingCallback(TrainerCallback):
    """Stop training if a logged loss or gradient norm is non-finite."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return control

        values_to_check = {
            "loss": logs.get("loss"),
            "grad_norm": logs.get("grad_norm"),
        }

        for value_name, value in values_to_check.items():
            if value is None:
                continue

            if not math.isfinite(float(value)):
                print(
                    f"\nNon-finite {value_name} detected "
                    f"at step {state.global_step}: {value}"
                )
                print("Training will be stopped safely.")
                control.should_training_stop = True
                break

        return control


TRAINING_LOCK = (
    DEBERTA_SMALL_OUTPUT_DIR / ".training_started"
)

TRAINING_COMPLETED_MARKER = (
    DEBERTA_SMALL_OUTPUT_DIR / ".training_completed"
)

existing_checkpoints = sorted(
    DEBERTA_SMALL_OUTPUT_DIR.glob("checkpoint-*")
)

if existing_checkpoints:
    raise RuntimeError(
        "Checkpoint already exists. "
        "Do not run this training cell again."
    )

if TRAINING_LOCK.exists() or TRAINING_COMPLETED_MARKER.exists():
    raise RuntimeError(
        "This training cell has already been started. "
        "Do not run it again."
    )


# بررسی نهایی مدل پیش از آموزش
parameter_dtypes = {
    parameter.dtype
    for parameter in trainer.model.parameters()
}

if parameter_dtypes != {torch.float32}:
    raise TypeError(
        f"Expected only torch.float32 parameters, "
        f"but found: {parameter_dtypes}"
    )

if not all(
    torch.isfinite(parameter).all().item()
    for parameter in trainer.model.parameters()
    if parameter.is_floating_point()
):
    raise RuntimeError(
        "The model contains non-finite parameters before training."
    )


trainer.add_callback(
    StopOnNonFiniteTrainingCallback()
)

TRAINING_LOCK.write_text(
    "DeBERTa-v3-small FP32 training started.",
    encoding="utf-8",
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_minutes = (
    time.perf_counter() - training_start_time
) / 60

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# بررسی سلامت مدل پس از آموزش
affected_parameter_tensors = 0
non_finite_parameter_values = 0

for parameter in trainer.model.parameters():
    if not parameter.is_floating_point():
        continue

    invalid_count = int(
        (~torch.isfinite(parameter.detach())).sum().item()
    )

    if invalid_count > 0:
        affected_parameter_tensors += 1
        non_finite_parameter_values += invalid_count


eval_records = [
    record
    for record in trainer.state.log_history
    if "eval_micro_f1" in record
]

training_succeeded = (
    trainer.state.global_step == TOTAL_TRAINING_STEPS
    and non_finite_parameter_values == 0
    and trainer.state.best_model_checkpoint is not None
)

if training_succeeded:
    TRAINING_COMPLETED_MARKER.write_text(
        "DeBERTa-v3-small FP32 training completed successfully.",
        encoding="utf-8",
    )


print("\nTraining process finished.")
print(
    "Training time:",
    round(training_elapsed_minutes, 2),
    "minutes",
)
print(
    "Completed optimizer steps:",
    trainer.state.global_step,
)
print(
    "Expected optimizer steps:",
    TOTAL_TRAINING_STEPS,
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Final average training loss:",
    train_result.training_loss,
)
print(
    "Affected parameter tensors:",
    affected_parameter_tensors,
)
print(
    "Non-finite parameter values:",
    non_finite_parameter_values,
)
print(
    "All parameters remain finite:",
    non_finite_parameter_values == 0,
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Training succeeded:",
    training_succeeded,
)

print("\nRecorded validation results:")

for record in eval_records:
    print({
        key: record[key]
        for key in [
            "epoch",
            "eval_loss",
            "eval_micro_f1",
            "eval_accuracy",
            "eval_macro_f1",
            "eval_mae",
            "eval_within_1",
        ]
        if key in record
    })

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Micro F1,Accuracy,Macro F1,Mae,Within 1
1,5.023247,0.597106,0.757200,0.757200,0.618599,0.285600,0.966700
2,3.968188,0.586678,0.770200,0.770200,0.652202,0.263500,0.973400


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.69it/s]



Training process finished.
Training time: 25.11 minutes
Completed optimizer steps: 6250
Expected optimizer steps: 6250
Peak allocated VRAM: 3.64 GB
Final average training loss: 5.061815923461914
Affected parameter tensors: 0
Non-finite parameter values: 0
All parameters remain finite: True
Best checkpoint: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_pilot/checkpoint-6250
Best validation Micro F1: 0.7702
Training succeeded: True

Recorded validation results:
{'epoch': 1.0, 'eval_loss': 0.597106397151947, 'eval_micro_f1': 0.7572, 'eval_accuracy': 0.7572, 'eval_macro_f1': 0.618598887647075, 'eval_mae': 0.2856, 'eval_within_1': 0.9667}
{'epoch': 2.0, 'eval_loss': 0.586677610874176, 'eval_micro_f1': 0.7702, 'eval_accuracy': 0.7702, 'eval_macro_f1': 0.6522019950434111, 'eval_mae': 0.2635, 'eval_within_1': 0.9734}


<div dir="rtl" align="right">

## ثبت نتیجه معتبر DeBERTa-v3-small

مدل `DeBERTa-v3-small` با بارگذاری صریح وزن‌ها در قالب `float32` با موفقیت آموزش داده شد.

بهترین نتیجه در Epoch دوم به‌دست آمد:

- Micro F1 برابر `0.7702`
- Macro F1 برابر `0.6522`
- MAE برابر `0.2635`
- دقت پیش‌بینی در فاصله یک امتیاز برابر `0.9734`

در این مرحله، نتیجه مدل در فایل مقایسه آزمایش‌های Transformer ثبت و جدول رتبه‌بندی به‌روزرسانی می‌شود.

</div>

In [68]:
from pathlib import Path

import pandas as pd


RESULTS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "transformer_pilot_results.csv"
)

results_df = pd.read_csv(RESULTS_PATH)


# استخراج خودکار بهترین رکورد اعتبارسنجی
eval_records = [
    record
    for record in trainer.state.log_history
    if "eval_micro_f1" in record
]

if not eval_records:
    raise RuntimeError(
        "No validation record was found in Trainer history."
    )

best_eval_record = max(
    eval_records,
    key=lambda record: record["eval_micro_f1"],
)


# افزودن ستون‌های لازم بدون تغییر نتایج قبلی
default_columns = {
    "accuracy": None,
    "eval_loss": None,
    "status": "valid",
    "notes": "",
}

for column_name, default_value in default_columns.items():
    if column_name not in results_df.columns:
        results_df[column_name] = default_value


current_model_name = "microsoft/deberta-v3-small"

# جلوگیری از ثبت چندباره همین مدل
results_df = results_df[
    results_df["model_name"] != current_model_name
].copy()


deberta_small_result = {
    "model_name": current_model_name,
    "train_samples": len(tokenized_train_dataset),
    "validation_samples": len(tokenized_validation_dataset),
    "max_length": MAX_LENGTH,
    "epochs": 2,
    "train_batch_size": effective_batch_size,
    "learning_rate": 2e-5,

    "micro_f1": best_eval_record["eval_micro_f1"],
    "accuracy": best_eval_record["eval_accuracy"],
    "macro_f1": best_eval_record["eval_macro_f1"],
    "mae": best_eval_record["eval_mae"],
    "within_1": best_eval_record["eval_within_1"],
    "eval_loss": best_eval_record["eval_loss"],

    "training_minutes": training_elapsed_minutes,
    "peak_vram_gb": peak_training_vram_gb,
    "best_checkpoint": trainer.state.best_model_checkpoint,

    "status": "valid",
    "notes": (
        "Loaded explicitly with dtype=torch.float32; "
        "non-fused AdamW; TF32 disabled."
    ),
}


results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([deberta_small_result]),
    ],
    ignore_index=True,
)


results_df = results_df.sort_values(
    by="micro_f1",
    ascending=False,
    na_position="last",
).reset_index(drop=True)


results_df.to_csv(
    RESULTS_PATH,
    index=False,
)


leaderboard_columns = [
    "model_name",
    "micro_f1",
    "macro_f1",
    "mae",
    "within_1",
    "training_minutes",
    "peak_vram_gb",
    "status",
]

print("DeBERTa-v3-small result recorded successfully.")
print("Results path:", RESULTS_PATH)
print()
print(
    results_df[
        leaderboard_columns
    ].to_string(index=False)
)

DeBERTa-v3-small result recorded successfully.
Results path: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/transformer_pilot_results.csv

                        model_name  micro_f1  macro_f1    mae  within_1  training_minutes  peak_vram_gb status
        microsoft/deberta-v3-small    0.7702  0.652202 0.2635    0.9734         25.106630      3.636506  valid
     distilbert/distilroberta-base    0.7608  0.634244 0.2813    0.9683          9.891078      2.706133  valid
distilbert/distilbert-base-uncased    0.7511  0.615374 0.2991    0.9622          9.705474      2.500323  valid
       microsoft/deberta-v3-xsmall    0.0979  0.035668 3.0235    0.1667               NaN           NaN failed


<div dir="rtl" align="right">

## پاک‌سازی حافظه پس از پایان آزمایش‌های Pilot

مدل `DeBERTa-v3-small` با امتیاز `Micro F1 = 0.7702` بهترین نتیجه مرحله Pilot را به‌دست آورد.

در این مرحله:

- وجود بهترین Checkpoint بررسی می‌شود.
- اشیای مربوط به مدل، Trainer، Optimizer و Scheduler از حافظه حذف می‌شوند.
- حافظه GPU آزاد می‌شود.
- فایل Checkpoint و جدول نتایج حذف یا تغییر داده نمی‌شوند.

Tokenizer و داده‌های پردازش‌شده فعلاً در حافظه باقی می‌مانند تا در مراحل بعدی قابل استفاده باشند.

</div>

In [69]:
import gc
from pathlib import Path

import torch


best_checkpoint_path = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_pilot/checkpoint-6250"
)

results_path = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "transformer_pilot_results.csv"
)

if not best_checkpoint_path.exists():
    raise FileNotFoundError(
        f"Best checkpoint was not found: {best_checkpoint_path}"
    )

if not results_path.exists():
    raise FileNotFoundError(
        f"Results file was not found: {results_path}"
    )


objects_to_remove = [
    "trainer",
    "model",
    "optimizer",
    "scheduler",
    "train_result",
    "training_args",
    "data_collator",
    "eval_records",
]

removed_objects = []

for object_name in objects_to_remove:
    if object_name in globals():
        del globals()[object_name]
        removed_objects.append(object_name)

gc.collect()
torch.cuda.empty_cache()


print("Pilot training cleanup completed.")
print("Removed objects:", removed_objects)
print("Best checkpoint preserved:", best_checkpoint_path)
print("Results file preserved:", results_path)
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)
print(
    "Reserved GPU memory:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB",
)
print("Ready to prepare the full-data experiment.")

Pilot training cleanup completed.
Removed objects: ['trainer', 'model', 'optimizer', 'scheduler', 'train_result', 'training_args', 'data_collator', 'eval_records']
Best checkpoint preserved: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_pilot/checkpoint-6250
Results file preserved: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/transformer_pilot_results.csv
Allocated GPU memory: 1.0 GB
Reserved GPU memory: 1.03 GB
Ready to prepare the full-data experiment.


<div dir="rtl" align="right">

## شناسایی DataFrameهای موجود در حافظه

برای جلوگیری از تغییر هم‌زمان فضای متغیرهای Notebook، ابتدا یک نسخه ثابت از متغیرهای سراسری گرفته می‌شود.

سپس برای هر DataFrame، نام متغیر، تعداد سطرها و ستون‌های آن نمایش داده خواهد شد. هیچ داده‌ای تغییر یا حذف نمی‌شود.

</div>

In [71]:
import pandas as pd

dataframe_inventory = []

# گرفتن نسخه ثابت برای جلوگیری از RuntimeError
global_items_snapshot = list(globals().items())

for variable_name, variable_value in global_items_snapshot:
    if not isinstance(variable_value, pd.DataFrame):
        continue

    dataframe_inventory.append({
        "variable_name": variable_name,
        "rows": len(variable_value),
        "columns_count": len(variable_value.columns),
        "columns": variable_value.columns.tolist(),
    })

dataframe_inventory = sorted(
    dataframe_inventory,
    key=lambda item: item["rows"],
    reverse=True,
)

print("DataFrames currently available in memory:\n")

for item in dataframe_inventory:
    print("Variable:", item["variable_name"])
    print("Rows:", f"{item['rows']:,}")
    print("Columns:", item["columns"])
    print("-" * 80)

DataFrames currently available in memory:

Variable: train_df
Rows: 664,451
Columns: ['overall', 'summary', 'reviewText_clean']
--------------------------------------------------------------------------------
Variable: validation_df
Rows: 166,161
Columns: ['overall', 'summary', 'reviewText_clean']
--------------------------------------------------------------------------------
Variable: _
Rows: 156,161
Columns: ['overall', 'summary', 'reviewText_clean']
--------------------------------------------------------------------------------
Variable: pilot_train_df
Rows: 50,000
Columns: ['text', 'label']
--------------------------------------------------------------------------------
Variable: pilot_train_pairs
Rows: 50,000
Columns: ['summary', 'review', 'label']
--------------------------------------------------------------------------------
Variable: pilot_validation_df
Rows: 10,000
Columns: ['text', 'label']
--------------------------------------------------------------------------------
Va

<div dir="rtl" align="right">

## آماده‌سازی داده‌های کامل برای آموزش Transformer

در این مرحله، مجموعه‌های کامل آموزش و اعتبارسنجی به قالب موردنیاز مدل تبدیل می‌شوند.

برای هر نمونه سه ستون ساخته خواهد شد:

- `summary`: عنوان Review
- `review`: متن پاک‌سازی‌شده Review
- `label`: برچسب عددی صفر تا چهار متناظر با امتیازهای یک تا پنج

مقادیر متنی خالی با رشته خالی جایگزین می‌شوند و DataFrameهای اصلی تغییر نخواهند کرد. همچنین تعداد نمونه‌ها، توزیع کلاس‌ها و وجود ورودی‌های کاملاً خالی بررسی می‌شود.

</div>

In [72]:
import pandas as pd


def build_full_pair_dataframe(source_df):
    """Create a model-ready DataFrame without modifying the source."""

    prepared_df = pd.DataFrame({
        "summary": (
            source_df["summary"]
            .fillna("")
            .astype(str)
        ),
        "review": (
            source_df["reviewText_clean"]
            .fillna("")
            .astype(str)
        ),
        "label": (
            pd.to_numeric(
                source_df["overall"],
                errors="raise",
            )
            .astype("int8")
            - 1
        ),
    })

    invalid_labels = ~prepared_df["label"].between(0, 4)

    if invalid_labels.any():
        invalid_values = sorted(
            prepared_df.loc[
                invalid_labels,
                "label",
            ].unique().tolist()
        )

        raise ValueError(
            f"Invalid labels were found: {invalid_values}"
        )

    return prepared_df


full_train_pairs = build_full_pair_dataframe(
    train_df
)

full_validation_pairs = build_full_pair_dataframe(
    validation_df
)


train_empty_inputs = (
    full_train_pairs["summary"].str.strip().eq("")
    & full_train_pairs["review"].str.strip().eq("")
).sum()

validation_empty_inputs = (
    full_validation_pairs["summary"].str.strip().eq("")
    & full_validation_pairs["review"].str.strip().eq("")
).sum()


print("Full-data preparation completed.")
print()

print("Training data:")
print("Rows:", f"{len(full_train_pairs):,}")
print("Columns:", full_train_pairs.columns.tolist())
print("Label dtype:", full_train_pairs["label"].dtype)
print("Completely empty inputs:", int(train_empty_inputs))
print(
    "Class counts:",
    full_train_pairs["label"]
    .value_counts()
    .sort_index()
    .to_dict(),
)

print()

print("Validation data:")
print("Rows:", f"{len(full_validation_pairs):,}")
print("Columns:", full_validation_pairs.columns.tolist())
print("Label dtype:", full_validation_pairs["label"].dtype)
print("Completely empty inputs:", int(validation_empty_inputs))
print(
    "Class counts:",
    full_validation_pairs["label"]
    .value_counts()
    .sort_index()
    .to_dict(),
)

print()

print(
    "Original training DataFrame unchanged:",
    train_df.columns.tolist()
    == ["overall", "summary", "reviewText_clean"],
)

print(
    "Original validation DataFrame unchanged:",
    validation_df.columns.tolist()
    == ["overall", "summary", "reviewText_clean"],
)

Full-data preparation completed.

Training data:
Rows: 664,451
Columns: ['summary', 'review', 'label']
Label dtype: int8
Completely empty inputs: 0
Class counts: {0: 65971, 1: 44857, 2: 64383, 3: 124422, 4: 364818}

Validation data:
Rows: 166,161
Columns: ['summary', 'review', 'label']
Label dtype: int8
Completely empty inputs: 0
Class counts: {0: 16276, 1: 11431, 2: 16077, 3: 30719, 4: 91658}

Original training DataFrame unchanged: True
Original validation DataFrame unchanged: True


<div dir="rtl" align="right">

## بررسی فضای ذخیره‌سازی پیش از Tokenization کامل

Tokenization مجموعه کامل، فایل‌های Arrow نسبتاً بزرگی تولید می‌کند. پیش از شروع پردازش، موارد زیر بررسی می‌شوند:

- فضای آزاد دیسک محلی پروژه
- فضای آزاد مسیر Google Drive
- میانگین طول Token در داده‌های Pilot
- حجم تقریبی داده‌های Tokenize‌شده کامل
- وجود یا نبود پوشه خروجی قبلی

در این مرحله هیچ Tokenization جدیدی انجام نمی‌شود و داده‌ها تغییر نمی‌کنند.

</div>

In [73]:
import shutil
from pathlib import Path

import numpy as np


FULL_TOKENIZED_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_full_tokenized"
)

LOCAL_PROJECT_PATH = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning"
)

GOOGLE_DRIVE_PATH = Path(
    "/home/ali/GoogleDrive"
)


def disk_usage_gb(path):
    usage = shutil.disk_usage(path)

    return {
        "total_gb": usage.total / 1024**3,
        "used_gb": usage.used / 1024**3,
        "free_gb": usage.free / 1024**3,
    }


# برآورد طول Token براساس داده Pilot فعلی
pilot_sample_size = min(
    10_000,
    len(tokenized_train_dataset),
)

pilot_token_lengths = np.array([
    len(tokenized_train_dataset[index]["input_ids"])
    for index in range(pilot_sample_size)
])

average_tokens = float(
    pilot_token_lengths.mean()
)

total_full_samples = (
    len(full_train_pairs)
    + len(full_validation_pairs)
)

estimated_total_tokens = (
    average_tokens
    * total_full_samples
)

# برآورد محافظه‌کارانه:
# input_ids + attention_mask + token_type_ids
# به‌علاوه سربار Arrow و Offsetها
estimated_storage_gb = (
    estimated_total_tokens
    * 3
    * 8
    * 1.25
    / 1024**3
)


local_usage = disk_usage_gb(
    LOCAL_PROJECT_PATH
)

drive_usage = disk_usage_gb(
    GOOGLE_DRIVE_PATH
)


print("Storage readiness check completed.")
print()

print("Pilot sample used:", f"{pilot_sample_size:,}")
print(
    "Average token length:",
    round(average_tokens, 2),
)
print(
    "Full samples:",
    f"{total_full_samples:,}",
)
print(
    "Estimated tokenized storage:",
    round(estimated_storage_gb, 2),
    "GB",
)

print()

print("Local project filesystem:")
print(
    "Total:",
    round(local_usage["total_gb"], 2),
    "GB",
)
print(
    "Used:",
    round(local_usage["used_gb"], 2),
    "GB",
)
print(
    "Free:",
    round(local_usage["free_gb"], 2),
    "GB",
)

print()

print("Google Drive filesystem:")
print(
    "Total:",
    round(drive_usage["total_gb"], 2),
    "GB",
)
print(
    "Used:",
    round(drive_usage["used_gb"], 2),
    "GB",
)
print(
    "Free:",
    round(drive_usage["free_gb"], 2),
    "GB",
)

print()

print("Planned tokenized directory:", FULL_TOKENIZED_DIR)
print(
    "Directory already exists:",
    FULL_TOKENIZED_DIR.exists(),
)

if FULL_TOKENIZED_DIR.exists():
    print(
        "Directory contents:",
        sorted(
            path.name
            for path in FULL_TOKENIZED_DIR.iterdir()
        ),
    )

Storage readiness check completed.

Pilot sample used: 10,000
Average token length: 141.15
Full samples: 830,612
Estimated tokenized storage: 3.28 GB

Local project filesystem:
Total: 146.59 GB
Used: 101.57 GB
Free: 37.51 GB

Google Drive filesystem:
Total: 5120.0 GB
Used: 48.16 GB
Free: 5071.84 GB

Planned tokenized directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_tokenized
Directory already exists: False


<div dir="rtl" align="right">

## Tokenization و ذخیره مجموعه آموزش کامل

در این مرحله، تمام ۶۶۴٬۴۵۱ نمونه آموزشی با Tokenizer مدل `DeBERTa-v3-small` پردازش می‌شوند.

تنظیمات پردازش:

- عنوان و متن Review به‌صورت ورودی جفتی
- حداکثر طول ورودی برابر ۵۱۲ Token
- حذف Tokenهای اضافه با Truncation
- عدم Padding ثابت؛ Padding هنگام ساخت Batch انجام می‌شود
- ذخیره خروجی Tokenize‌شده روی دیسک محلی
- تقسیم فایل خروجی به Shardهای حداکثر یک گیگابایتی

این سلول دارای محافظ است و در صورت وجود خروجی قبلی، پردازش را دوباره آغاز نمی‌کند.

</div>

In [74]:
import gc
import shutil
import time
from pathlib import Path

from datasets import (
    ClassLabel,
    Dataset,
    Features,
    Value,
)


TOKENIZED_TRAIN_DIR = (
    FULL_TOKENIZED_DIR / "train"
)

if TOKENIZED_TRAIN_DIR.exists():
    raise FileExistsError(
        f"Tokenized training directory already exists: "
        f"{TOKENIZED_TRAIN_DIR}"
    )

FULL_TOKENIZED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# تعریف صریح Schema برای حفظ Labelهای پنج‌کلاسه
full_data_features = Features({
    "summary": Value("string"),
    "review": Value("string"),
    "label": ClassLabel(
        num_classes=5,
        names=["1", "2", "3", "4", "5"],
    ),
})


full_train_dataset = Dataset.from_pandas(
    full_train_pairs.astype({
        "summary": "string",
        "review": "string",
        "label": "int64",
    }),
    features=full_data_features,
    preserve_index=False,
)


def tokenize_full_train_batch(batch):
    return tokenizer(
        batch["summary"],
        batch["review"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenization_start_time = time.perf_counter()

tokenized_full_train_dataset = (
    full_train_dataset.map(
        tokenize_full_train_batch,
        batched=True,
        batch_size=1_000,
        num_proc=4,
        remove_columns=[
            "summary",
            "review",
        ],
        load_from_cache_file=False,
        writer_batch_size=1_000,
        desc="Tokenizing full DeBERTa-v3-small training data",
    )
)

tokenization_minutes = (
    time.perf_counter() - tokenization_start_time
) / 60


save_start_time = time.perf_counter()

tokenized_full_train_dataset.save_to_disk(
    str(TOKENIZED_TRAIN_DIR),
    max_shard_size="1GB",
)

save_minutes = (
    time.perf_counter() - save_start_time
) / 60


saved_size_bytes = sum(
    path.stat().st_size
    for path in TOKENIZED_TRAIN_DIR.rglob("*")
    if path.is_file()
)

saved_files = sorted(
    str(path.relative_to(TOKENIZED_TRAIN_DIR))
    for path in TOKENIZED_TRAIN_DIR.rglob("*")
    if path.is_file()
)

first_sample = tokenized_full_train_dataset[0]


print("Full training tokenization completed successfully.")
print("Samples:", f"{len(tokenized_full_train_dataset):,}")
print("Columns:", tokenized_full_train_dataset.column_names)
print(
    "Features:",
    tokenized_full_train_dataset.features,
)
print(
    "First sample token count:",
    len(first_sample["input_ids"]),
)
print(
    "First sample SEP occurrences:",
    first_sample["input_ids"].count(
        tokenizer.sep_token_id
    ),
)
print("First sample label:", first_sample["label"])

print()
print(
    "Tokenization time:",
    round(tokenization_minutes, 2),
    "minutes",
)
print(
    "Disk save time:",
    round(save_minutes, 2),
    "minutes",
)
print(
    "Saved size:",
    round(saved_size_bytes / 1024**3, 2),
    "GB",
)
print("Saved directory:", TOKENIZED_TRAIN_DIR)
print("Saved files:", saved_files)

print()
print(
    "Remaining local disk space:",
    round(
        shutil.disk_usage(
            TOKENIZED_TRAIN_DIR
        ).free / 1024**3,
        2,
    ),
    "GB",
)

# آزادسازی نسخه موقتِ قبل از Tokenization
del full_train_dataset

gc.collect()

Tokenizing full DeBERTa-v3-small training data (num_proc=4): 100%|██████████| 664451/664451 [00:36<00:00, 18331.47 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 664451/664451 [00:00<00:00, 3459262.03 examples/s]

Full training tokenization completed successfully.
Samples: 664,451
Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Features: {'label': ClassLabel(names=['1', '2', '3', '4', '5']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
First sample token count: 136
First sample SEP occurrences: 2
First sample label: 1

Tokenization time: 0.61 minutes
Disk save time: 0.0 minutes
Saved size: 0.54 GB
Saved directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_tokenized/train
Saved files: ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']

Remaining local disk space: 36.96 GB


1183

<div dir="rtl" align="right">

## Tokenization و ذخیره مجموعه اعتبارسنجی کامل

در این مرحله، تمام ۱۶۶٬۱۶۱ نمونه اعتبارسنجی با Tokenizer مدل `DeBERTa-v3-small` پردازش می‌شوند.

تنظیمات Tokenization دقیقاً با مجموعه آموزش یکسان است:

- ورودی جفتی عنوان و متن Review
- حداکثر طول ۵۱۲ Token
- حذف بخش‌های اضافه با Truncation
- عدم Padding ثابت
- ذخیره خروجی روی دیسک محلی

این سلول در صورت وجود خروجی قبلی، از اجرای دوباره جلوگیری می‌کند.

</div>

In [75]:
import gc
import shutil
import time

from datasets import Dataset


TOKENIZED_VALIDATION_DIR = (
    FULL_TOKENIZED_DIR / "validation"
)

if TOKENIZED_VALIDATION_DIR.exists():
    raise FileExistsError(
        f"Tokenized validation directory already exists: "
        f"{TOKENIZED_VALIDATION_DIR}"
    )


full_validation_dataset = Dataset.from_pandas(
    full_validation_pairs.astype({
        "summary": "string",
        "review": "string",
        "label": "int64",
    }),
    features=full_data_features,
    preserve_index=False,
)


def tokenize_full_validation_batch(batch):
    return tokenizer(
        batch["summary"],
        batch["review"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


tokenization_start_time = time.perf_counter()

tokenized_full_validation_dataset = (
    full_validation_dataset.map(
        tokenize_full_validation_batch,
        batched=True,
        batch_size=1_000,
        num_proc=4,
        remove_columns=[
            "summary",
            "review",
        ],
        load_from_cache_file=False,
        writer_batch_size=1_000,
        desc=(
            "Tokenizing full DeBERTa-v3-small "
            "validation data"
        ),
    )
)

tokenization_minutes = (
    time.perf_counter() - tokenization_start_time
) / 60


save_start_time = time.perf_counter()

tokenized_full_validation_dataset.save_to_disk(
    str(TOKENIZED_VALIDATION_DIR),
    max_shard_size="1GB",
)

save_minutes = (
    time.perf_counter() - save_start_time
) / 60


saved_size_bytes = sum(
    path.stat().st_size
    for path in TOKENIZED_VALIDATION_DIR.rglob("*")
    if path.is_file()
)

saved_files = sorted(
    str(path.relative_to(TOKENIZED_VALIDATION_DIR))
    for path in TOKENIZED_VALIDATION_DIR.rglob("*")
    if path.is_file()
)

first_sample = tokenized_full_validation_dataset[0]


print(
    "Full validation tokenization completed successfully."
)
print(
    "Samples:",
    f"{len(tokenized_full_validation_dataset):,}",
)
print(
    "Columns:",
    tokenized_full_validation_dataset.column_names,
)
print(
    "Features:",
    tokenized_full_validation_dataset.features,
)
print(
    "First sample token count:",
    len(first_sample["input_ids"]),
)
print(
    "First sample SEP occurrences:",
    first_sample["input_ids"].count(
        tokenizer.sep_token_id
    ),
)
print(
    "First sample label:",
    first_sample["label"],
)

print()
print(
    "Tokenization time:",
    round(tokenization_minutes, 2),
    "minutes",
)
print(
    "Disk save time:",
    round(save_minutes, 2),
    "minutes",
)
print(
    "Saved size:",
    round(saved_size_bytes / 1024**3, 2),
    "GB",
)
print(
    "Saved directory:",
    TOKENIZED_VALIDATION_DIR,
)
print(
    "Saved files:",
    saved_files,
)

print()
print(
    "Remaining local disk space:",
    round(
        shutil.disk_usage(
            TOKENIZED_VALIDATION_DIR
        ).free / 1024**3,
        2,
    ),
    "GB",
)

del full_validation_dataset

gc.collect()

Tokenizing full DeBERTa-v3-small validation data (num_proc=4): 100%|██████████| 166161/166161 [00:09<00:00, 17431.42 examples/s]
Saving the dataset (1/1 shards): 100%|██████████| 166161/166161 [00:00<00:00, 3364810.99 examples/s]


Full validation tokenization completed successfully.
Samples: 166,161
Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Features: {'label': ClassLabel(names=['1', '2', '3', '4', '5']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
First sample token count: 170
First sample SEP occurrences: 2
First sample label: 3

Tokenization time: 0.16 minutes
Disk save time: 0.0 minutes
Saved size: 0.14 GB
Saved directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_tokenized/validation
Saved files: ['data-00000-of-00001.arrow', 'dataset_info.json', 'state.json']

Remaining local disk space: 36.83 GB


516

<div dir="rtl" align="right">

## بارگذاری آزمایشی داده‌های Tokenize‌شده از دیسک

برای اطمینان از سالم‌بودن فایل‌های ذخیره‌شده، نسخه‌های فعلی داده‌های Tokenize‌شده از حافظه حذف و دوباره مستقیماً از دیسک بارگذاری می‌شوند.

در این بررسی موارد زیر کنترل خواهند شد:

- تعداد نمونه‌های آموزش و اعتبارسنجی
- نام ستون‌ها و نوع داده‌ها
- تطابق Labelها با DataFrameهای اصلی
- معتبر بودن شناسه Tokenها و Attention Mask
- وجود دو Token جداکننده در چند نمونه آزمایشی

هیچ آموزشی در این مرحله انجام نمی‌شود.

</div>

In [76]:
import gc

import numpy as np
from datasets import load_from_disk


# حذف نسخه‌های حافظه برای آزمون واقعی فایل‌های ذخیره‌شده
for object_name in [
    "tokenized_full_train_dataset",
    "tokenized_full_validation_dataset",
]:
    globals().pop(object_name, None)

gc.collect()


# بارگذاری مستقیم از دیسک
tokenized_full_train_dataset = load_from_disk(
    str(TOKENIZED_TRAIN_DIR)
)

tokenized_full_validation_dataset = load_from_disk(
    str(TOKENIZED_VALIDATION_DIR)
)


expected_columns = {
    "label",
    "input_ids",
    "token_type_ids",
    "attention_mask",
}

if set(tokenized_full_train_dataset.column_names) != expected_columns:
    raise ValueError(
        "Unexpected training dataset columns: "
        f"{tokenized_full_train_dataset.column_names}"
    )

if set(tokenized_full_validation_dataset.column_names) != expected_columns:
    raise ValueError(
        "Unexpected validation dataset columns: "
        f"{tokenized_full_validation_dataset.column_names}"
    )

if len(tokenized_full_train_dataset) != len(full_train_pairs):
    raise ValueError("Training sample count does not match.")

if len(tokenized_full_validation_dataset) != len(full_validation_pairs):
    raise ValueError("Validation sample count does not match.")


# بررسی چند نمونه از ابتدا، میانه و انتهای هر مجموعه
train_indices = [
    0,
    len(tokenized_full_train_dataset) // 2,
    len(tokenized_full_train_dataset) - 1,
]

validation_indices = [
    0,
    len(tokenized_full_validation_dataset) // 2,
    len(tokenized_full_validation_dataset) - 1,
]


def inspect_samples(dataset, source_df, indices):
    records = []

    for index in indices:
        sample = dataset[index]

        input_ids = sample["input_ids"]
        attention_mask = sample["attention_mask"]
        token_type_ids = sample["token_type_ids"]

        records.append({
            "index": index,
            "token_count": len(input_ids),
            "sep_count": input_ids.count(
                tokenizer.sep_token_id
            ),
            "label": int(sample["label"]),
            "expected_label": int(
                source_df.iloc[index]["label"]
            ),
            "lengths_match": (
                len(input_ids)
                == len(attention_mask)
                == len(token_type_ids)
            ),
            "token_ids_valid": all(
                0 <= token_id < tokenizer.vocab_size
                for token_id in input_ids
            ),
            "attention_mask_valid": set(
                attention_mask
            ).issubset({0, 1}),
        })

    return records


train_checks = inspect_samples(
    tokenized_full_train_dataset,
    full_train_pairs,
    train_indices,
)

validation_checks = inspect_samples(
    tokenized_full_validation_dataset,
    full_validation_pairs,
    validation_indices,
)


all_checks_passed = all(
    record["sep_count"] == 2
    and record["label"] == record["expected_label"]
    and record["lengths_match"]
    and record["token_ids_valid"]
    and record["attention_mask_valid"]
    for record in train_checks + validation_checks
)


print("Disk reload verification completed.")
print()

print("Training dataset:")
print("Samples:", f"{len(tokenized_full_train_dataset):,}")
print("Columns:", tokenized_full_train_dataset.column_names)
print("Features:", tokenized_full_train_dataset.features)
print("Disk fingerprint:", tokenized_full_train_dataset._fingerprint)

print("\nTraining sample checks:")
for record in train_checks:
    print(record)

print()

print("Validation dataset:")
print(
    "Samples:",
    f"{len(tokenized_full_validation_dataset):,}",
)
print(
    "Columns:",
    tokenized_full_validation_dataset.column_names,
)
print(
    "Features:",
    tokenized_full_validation_dataset.features,
)
print(
    "Disk fingerprint:",
    tokenized_full_validation_dataset._fingerprint,
)

print("\nValidation sample checks:")
for record in validation_checks:
    print(record)

print()
print("All integrity checks passed:", all_checks_passed)

Disk reload verification completed.

Training dataset:
Samples: 664,451
Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Features: {'label': ClassLabel(names=['1', '2', '3', '4', '5']), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Disk fingerprint: 3d88264cbef1bf15

Training sample checks:
{'index': 0, 'token_count': 136, 'sep_count': 2, 'label': 1, 'expected_label': 1, 'lengths_match': True, 'token_ids_valid': True, 'attention_mask_valid': True}
{'index': 332225, 'token_count': 54, 'sep_count': 2, 'label': 4, 'expected_label': 4, 'lengths_match': True, 'token_ids_valid': True, 'attention_mask_valid': True}
{'index': 664450, 'token_count': 100, 'sep_count': 2, 'label': 4, 'expected_label': 4, 'lengths_match': True, 'token_ids_valid': True, 'attention_mask_valid': True}

Validation dataset:
Samples: 166,161
Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']
Features: {'label': ClassLa

<div dir="rtl" align="right">

## پاک‌سازی مدل Pilot و بارگذاری مدل تازه برای آموزش کامل

داده‌های Tokenize‌شده کامل با موفقیت از دیسک بارگذاری و اعتبارسنجی شدند.

در این مرحله:

- ارجاع‌های باقی‌مانده به مدل، Optimizer و پارامترهای آزمایش Pilot پاک می‌شوند.
- حافظه GPU آزاد می‌شود.
- مسیر مستقل آموزش کامل بررسی می‌شود.
- مدل `DeBERTa-v3-small` دوباره از وزن‌های اصلی ازپیش‌آموزش‌دیده بارگذاری می‌شود.
- تمام وزن‌ها به‌صورت صریح با `float32` بارگذاری می‌شوند.
- سلامت و نوع داده پارامترها کنترل خواهد شد.

هنوز Trainer ساخته نمی‌شود و هیچ آموزشی آغاز نخواهد شد.

</div>

In [77]:
import gc
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification


FULL_MODEL_OUTPUT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_full_training"
)


# حذف ارجاع‌های باقی‌مانده از آزمایش‌های قبلی
objects_to_remove = [
    "model",
    "trainer",
    "optimizer",
    "scheduler",
    "training_args",
    "data_collator",
    "train_result",
    "decay_parameters",
    "no_decay_parameters",
    "first_parameter",
    "model_to_check",
    "model_to_inspect",
    "diagnostic_output",
    "diagnostic_batch",
]

removed_objects = []

for object_name in objects_to_remove:
    if object_name in globals():
        del globals()[object_name]
        removed_objects.append(object_name)

gc.collect()
torch.cuda.empty_cache()


allocated_after_cleanup_gb = (
    torch.cuda.memory_allocated() / 1024**3
)

reserved_after_cleanup_gb = (
    torch.cuda.memory_reserved() / 1024**3
)


# آموزش کامل باید در مسیر مستقل و خالی شروع شود
if (
    FULL_MODEL_OUTPUT_DIR.exists()
    and any(FULL_MODEL_OUTPUT_DIR.iterdir())
):
    raise FileExistsError(
        f"Full-training output directory is not empty: "
        f"{FULL_MODEL_OUTPUT_DIR}"
    )


# Seed برای مقداردهی اولیه قابل‌تکرار لایه طبقه‌بندی
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)


# بارگذاری تازه از وزن‌های اصلی با Float32
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    dtype=torch.float32,
)

model.gradient_checkpointing_disable()


parameter_dtype_counts = Counter(
    str(parameter.dtype)
    for parameter in model.parameters()
)

all_parameters_finite = all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
    if parameter.is_floating_point()
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

parameter_storage_gb = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
) / 1024**3


print("Full-training model preparation completed.")
print("Removed old objects:", removed_objects)

print()
print(
    "GPU allocated after old-object cleanup:",
    round(allocated_after_cleanup_gb, 2),
    "GB",
)
print(
    "GPU reserved after old-object cleanup:",
    round(reserved_after_cleanup_gb, 2),
    "GB",
)

print()
print("Model:", MODEL_NAME)
print("Model device:", next(model.parameters()).device)
print("Parameter dtypes:", dict(parameter_dtype_counts))
print("Config dtype:", getattr(model.config, "dtype", None))
print("All parameters are finite:", all_parameters_finite)
print("Total parameters:", f"{total_parameters:,}")
print(
    "Parameter storage:",
    round(parameter_storage_gb, 2),
    "GB",
)

print()
print("Full-training output directory:", FULL_MODEL_OUTPUT_DIR)
print(
    "Output directory exists:",
    FULL_MODEL_OUTPUT_DIR.exists(),
)
print(
    "Full training samples:",
    f"{len(tokenized_full_train_dataset):,}",
)
print(
    "Full validation samples:",
    f"{len(tokenized_full_validation_dataset):,}",
)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 1451.40it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING   

Full-training model preparation completed.
Removed old objects: ['decay_parameters', 'no_decay_parameters', 'first_parameter']

GPU allocated after old-object cleanup: 1.0 GB
GPU reserved after old-object cleanup: 1.03 GB

Model: microsoft/deberta-v3-small
Model device: cpu
Parameter dtypes: {'torch.float32': 106}
Config dtype: torch.float32
All parameters are finite: True
Total parameters: 141,898,757
Parameter storage: 0.53 GB

Full-training output directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training
Output directory exists: False
Full training samples: 664,451
Full validation samples: 166,161


<div dir="rtl" align="right">

## ساخت Trainer برای آموزش کامل DeBERTa-v3-small

در این مرحله، Trainer آموزش کامل با تنظیمات پایدار آزمایش Pilot ساخته می‌شود:

- آموزش از وزن‌های اصلی ازپیش‌آموزش‌دیده
- تمام پارامترها با دقت `float32`
- دو Epoch روی ۶۶۴٬۴۵۱ نمونه
- Batch Size واقعی برابر ۲
- هشت مرحله Gradient Accumulation
- Batch Size مؤثر برابر ۱۶
- نرخ یادگیری `2e-5`
- Warmup برابر ۱۰ درصد کل گام‌ها
- Optimizer معمولی `AdamW` با `fused=False`
- غیرفعال‌بودن `TF32` و Gradient Checkpointing
- ارزیابی و ذخیره در پایان هر Epoch
- انتخاب بهترین Checkpoint براساس `Micro F1`
- ذخیره تمام خروجی‌ها روی دیسک محلی

در این مرحله هنوز آموزش آغاز نمی‌شود.

</div>

In [78]:
import math

import torch
from transformers import (
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


FULL_EPOCHS = 2
PHYSICAL_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_BATCH_SIZE = 16

FULL_MODEL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if any(FULL_MODEL_OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        f"Full-training output directory is not empty: "
        f"{FULL_MODEL_OUTPUT_DIR}"
    )


# کنترل نهایی مدل
parameter_dtypes = {
    parameter.dtype
    for parameter in model.parameters()
}

if parameter_dtypes != {torch.float32}:
    raise TypeError(
        f"Expected only torch.float32 parameters, "
        f"but found: {parameter_dtypes}"
    )

if not all(
    torch.isfinite(parameter).all().item()
    for parameter in model.parameters()
    if parameter.is_floating_point()
):
    raise RuntimeError(
        "The model contains non-finite parameters."
    )


torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

model.gradient_checkpointing_disable()


data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)


# گروه‌بندی پارامترها برای Weight Decay
decay_parameters = []
no_decay_parameters = []

for parameter_name, parameter in model.named_parameters():
    if not parameter.requires_grad:
        continue

    is_no_decay = (
        parameter_name.endswith(".bias")
        or "LayerNorm.weight" in parameter_name
        or "layer_norm.weight" in parameter_name
    )

    if is_no_decay:
        no_decay_parameters.append(parameter)
    else:
        decay_parameters.append(parameter)


optimizer = torch.optim.AdamW(
    [
        {
            "params": decay_parameters,
            "weight_decay": 0.01,
        },
        {
            "params": no_decay_parameters,
            "weight_decay": 0.0,
        },
    ],
    lr=2e-5,
    betas=(0.9, 0.999),
    eps=1e-8,
    fused=False,
    foreach=False,
)


training_args = TrainingArguments(
    output_dir=str(FULL_MODEL_OUTPUT_DIR),

    num_train_epochs=FULL_EPOCHS,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.10,
    lr_scheduler_type="linear",
    max_grad_norm=1.0,

    per_device_train_batch_size=PHYSICAL_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,

    bf16=False,
    fp16=False,
    tf32=False,
    gradient_checkpointing=False,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=500,
    logging_first_step=True,
    logging_nan_inf_filter=False,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,
    report_to="none",

    seed=RANDOM_STATE,
    data_seed=RANDOM_STATE,

    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)


# Scheduler توسط Trainer و براساس تعداد واقعی گام‌ها ساخته می‌شود.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_full_train_dataset,
    eval_dataset=tokenized_full_validation_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)


train_micro_batches_per_epoch = len(
    trainer.get_train_dataloader()
)

estimated_update_steps_per_epoch = math.ceil(
    train_micro_batches_per_epoch
    / GRADIENT_ACCUMULATION_STEPS
)

estimated_total_update_steps = (
    estimated_update_steps_per_epoch
    * FULL_EPOCHS
)

estimated_warmup_steps = math.ceil(
    estimated_total_update_steps
    * training_args.warmup_ratio
)

effective_batch_size = (
    PHYSICAL_TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)


print("Full-data Trainer created successfully.")
print("Output directory:", training_args.output_dir)
print("Model parameter dtypes:", parameter_dtypes)
print("Optimizer:", optimizer.__class__.__name__)
print("Optimizer fused:", optimizer.defaults.get("fused"))
print("Optimizer foreach:", optimizer.defaults.get("foreach"))
print("Scheduler type:", training_args.lr_scheduler_type)
print("Warmup ratio:", training_args.warmup_ratio)

print()
print("Epochs:", training_args.num_train_epochs)
print("Physical train batch size:", PHYSICAL_TRAIN_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective train batch size:", effective_batch_size)
print("Evaluation batch size:", EVAL_BATCH_SIZE)

print()
print(
    "Training samples:",
    f"{len(trainer.train_dataset):,}",
)
print(
    "Validation samples:",
    f"{len(trainer.eval_dataset):,}",
)
print(
    "Micro-batches per epoch:",
    f"{train_micro_batches_per_epoch:,}",
)
print(
    "Estimated update steps per epoch:",
    f"{estimated_update_steps_per_epoch:,}",
)
print(
    "Estimated total update steps:",
    f"{estimated_total_update_steps:,}",
)
print(
    "Estimated warmup steps:",
    f"{estimated_warmup_steps:,}",
)

print()
print(
    "Existing checkpoints:",
    list(FULL_MODEL_OUTPUT_DIR.glob("checkpoint-*")),
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Full-data Trainer created successfully.
Output directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training
Model parameter dtypes: {torch.float32}
Optimizer: AdamW
Optimizer fused: False
Optimizer foreach: False
Scheduler type: SchedulerType.LINEAR
Warmup ratio: 0.1

Epochs: 2
Physical train batch size: 2
Gradient accumulation: 8
Effective train batch size: 16
Evaluation batch size: 16

Training samples: 664,451
Validation samples: 166,161
Micro-batches per epoch: 332,226
Estimated update steps per epoch: 41,529
Estimated total update steps: 83,058
Estimated warmup steps: 8,306

Existing checkpoints: []


<div dir="rtl" align="right">

## اصلاح تنظیم Warmup پیش از آموزش کامل

در تنظیمات اولیه از `warmup_ratio=0.10` استفاده شده بود که در نسخه فعلی Transformers منسوخ اعلام شده است.

در این مرحله:

- اطمینان حاصل می‌شود که Scheduler هنوز ساخته نشده است.
- مقدار `warmup_ratio` حذف می‌شود.
- تعداد Warmup Stepها به‌صورت صریح روی `8306` قرار می‌گیرد.
- Trainer یا Optimizer دوباره ساخته نمی‌شوند.
- هیچ آموزشی آغاز نمی‌شود.

</div>

In [79]:
EXPLICIT_WARMUP_STEPS = estimated_warmup_steps

if trainer.lr_scheduler is not None:
    raise RuntimeError(
        "The learning-rate scheduler has already been created. "
        "Do not modify the warmup configuration now."
    )

if EXPLICIT_WARMUP_STEPS != 8_306:
    raise ValueError(
        f"Unexpected warmup-step calculation: "
        f"{EXPLICIT_WARMUP_STEPS:,}"
    )

# Trainer و training_args به یک شیء تنظیمات اشاره می‌کنند.
trainer.args.warmup_ratio = None
trainer.args.warmup_steps = EXPLICIT_WARMUP_STEPS

print("Warmup configuration updated successfully.")
print("Warmup ratio:", trainer.args.warmup_ratio)
print(
    "Explicit warmup steps:",
    f"{trainer.args.warmup_steps:,}",
)
print(
    "Estimated total update steps:",
    f"{estimated_total_update_steps:,}",
)
print(
    "Effective warmup percentage:",
    round(
        trainer.args.warmup_steps
        / estimated_total_update_steps
        * 100,
        4,
    ),
    "%",
)
print(
    "Scheduler has not been created yet:",
    trainer.lr_scheduler is None,
)

Warmup configuration updated successfully.
Warmup ratio: None
Explicit warmup steps: 8,306
Estimated total update steps: 83,058
Effective warmup percentage: 10.0002 %
Scheduler has not been created yet: True


<div dir="rtl" align="right">

## آموزش کامل DeBERTa-v3-small

در این مرحله، مدل `DeBERTa-v3-small` از وزن‌های اصلی روی تمام داده‌های آموزش Fine-tune می‌شود.

تنظیمات اصلی اجرا:

- ۶۶۴٬۴۵۱ نمونه آموزش
- ۱۶۶٬۱۶۱ نمونه اعتبارسنجی
- دو Epoch
- مجموع تقریبی ۸۳٬۰۵۸ گام به‌روزرسانی
- Warmup برابر ۸٬۳۰۶ گام
- Batch Size مؤثر برابر ۱۶
- وزن‌های صریح `float32`
- Optimizer غیرادغامی `AdamW`
- ارزیابی و ذخیره در پایان هر Epoch
- انتخاب بهترین Checkpoint براساس `Micro F1`

برای جلوگیری از اجرای تصادفی مجدد، یک قفل روی دیسک ایجاد می‌شود. همچنین در صورت مشاهده `NaN` یا `Inf`، آموزش به‌صورت ایمن متوقف خواهد شد.

</div>

In [80]:
import json
import math
import time
from datetime import datetime

import torch
from transformers import TrainerCallback


class StopOnNonFiniteFullTrainingCallback(TrainerCallback):
    """Stop full training if loss or gradient norm becomes NaN/Inf."""

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return control

        for value_name in ("loss", "grad_norm"):
            value = logs.get(value_name)

            if value is None:
                continue

            if not math.isfinite(float(value)):
                print(
                    f"\nNon-finite {value_name} detected "
                    f"at step {state.global_step}: {value}"
                )
                print("Full training will be stopped safely.")
                control.should_training_stop = True
                break

        return control


FULL_TRAINING_LOCK = (
    FULL_MODEL_OUTPUT_DIR / ".training_started"
)

FULL_TRAINING_COMPLETED_MARKER = (
    FULL_MODEL_OUTPUT_DIR / ".training_completed"
)

FULL_TRAINING_SUMMARY_PATH = (
    FULL_MODEL_OUTPUT_DIR / "training_summary.json"
)


# جلوگیری از اجرای ناخواسته مجدد
existing_checkpoints = sorted(
    FULL_MODEL_OUTPUT_DIR.glob("checkpoint-*")
)

if existing_checkpoints:
    raise RuntimeError(
        "Full-training checkpoints already exist. "
        "Do not run this cell again."
    )

if (
    FULL_TRAINING_LOCK.exists()
    or FULL_TRAINING_COMPLETED_MARKER.exists()
):
    raise RuntimeError(
        "This full-training cell has already been started. "
        "Do not run it again."
    )


# بررسی نهایی تنظیمات و مدل
if trainer.lr_scheduler is not None:
    raise RuntimeError(
        "The scheduler was created before the final checks."
    )

if trainer.args.warmup_steps != 8_306:
    raise ValueError(
        f"Unexpected warmup steps: "
        f"{trainer.args.warmup_steps:,}"
    )

parameter_dtypes = {
    parameter.dtype
    for parameter in trainer.model.parameters()
}

if parameter_dtypes != {torch.float32}:
    raise TypeError(
        f"Expected only torch.float32 parameters, "
        f"but found: {parameter_dtypes}"
    )

if not all(
    torch.isfinite(parameter).all().item()
    for parameter in trainer.model.parameters()
    if parameter.is_floating_point()
):
    raise RuntimeError(
        "The model contains non-finite parameters "
        "before full training."
    )


trainer.add_callback(
    StopOnNonFiniteFullTrainingCallback()
)

FULL_TRAINING_LOCK.write_text(
    json.dumps(
        {
            "started_at": datetime.now().isoformat(),
            "expected_steps": estimated_total_update_steps,
            "warmup_steps": trainer.args.warmup_steps,
            "epochs": FULL_EPOCHS,
        },
        indent=2,
    ),
    encoding="utf-8",
)


print("Full training is starting.")
print(
    "Expected optimizer steps:",
    f"{estimated_total_update_steps:,}",
)
print(
    "Warmup steps:",
    f"{trainer.args.warmup_steps:,}",
)
print("Parameter dtypes:", parameter_dtypes)
print("Output directory:", FULL_MODEL_OUTPUT_DIR)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

training_start_time = time.perf_counter()

train_result = trainer.train(
    resume_from_checkpoint=False,
)

training_elapsed_hours = (
    time.perf_counter() - training_start_time
) / 3600

peak_training_vram_gb = (
    torch.cuda.max_memory_allocated()
    / 1024**3
)


# بررسی سلامت مدل پس از پایان
affected_parameter_tensors = 0
non_finite_parameter_values = 0

for parameter in trainer.model.parameters():
    if not parameter.is_floating_point():
        continue

    invalid_count = int(
        (~torch.isfinite(parameter.detach())).sum().item()
    )

    if invalid_count > 0:
        affected_parameter_tensors += 1
        non_finite_parameter_values += invalid_count


eval_records = [
    record
    for record in trainer.state.log_history
    if "eval_micro_f1" in record
]

training_succeeded = (
    trainer.state.global_step
    == estimated_total_update_steps
    and non_finite_parameter_values == 0
    and trainer.state.best_model_checkpoint is not None
    and len(eval_records) == FULL_EPOCHS
)


training_summary = {
    "model_name": MODEL_NAME,
    "training_samples": len(trainer.train_dataset),
    "validation_samples": len(trainer.eval_dataset),
    "epochs": FULL_EPOCHS,
    "completed_steps": trainer.state.global_step,
    "expected_steps": estimated_total_update_steps,
    "warmup_steps": trainer.args.warmup_steps,
    "training_hours": training_elapsed_hours,
    "peak_vram_gb": peak_training_vram_gb,
    "training_loss": train_result.training_loss,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_micro_f1": trainer.state.best_metric,
    "non_finite_parameter_values": non_finite_parameter_values,
    "training_succeeded": training_succeeded,
    "validation_records": eval_records,
}

FULL_TRAINING_SUMMARY_PATH.write_text(
    json.dumps(
        training_summary,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

if training_succeeded:
    FULL_TRAINING_COMPLETED_MARKER.write_text(
        "Full DeBERTa-v3-small training completed successfully.",
        encoding="utf-8",
    )


print("\nFull training process finished.")
print(
    "Training time:",
    round(training_elapsed_hours, 2),
    "hours",
)
print(
    "Completed optimizer steps:",
    f"{trainer.state.global_step:,}",
)
print(
    "Expected optimizer steps:",
    f"{estimated_total_update_steps:,}",
)
print(
    "Peak allocated VRAM:",
    round(peak_training_vram_gb, 2),
    "GB",
)
print(
    "Final average training loss:",
    train_result.training_loss,
)
print(
    "Affected parameter tensors:",
    affected_parameter_tensors,
)
print(
    "Non-finite parameter values:",
    non_finite_parameter_values,
)
print(
    "All parameters remain finite:",
    non_finite_parameter_values == 0,
)
print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint,
)
print(
    "Best validation Micro F1:",
    trainer.state.best_metric,
)
print(
    "Training succeeded:",
    training_succeeded,
)
print(
    "Summary file:",
    FULL_TRAINING_SUMMARY_PATH,
)

print("\nRecorded validation results:")

for record in eval_records:
    print({
        key: record[key]
        for key in [
            "epoch",
            "eval_loss",
            "eval_micro_f1",
            "eval_accuracy",
            "eval_macro_f1",
            "eval_mae",
            "eval_within_1",
        ]
        if key in record
    })

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Full training is starting.
Expected optimizer steps: 83,058
Warmup steps: 8,306
Parameter dtypes: {torch.float32}
Output directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training


Epoch,Training Loss,Validation Loss,Micro F1,Accuracy,Macro F1,Mae,Within 1
1,4.247773,0.522401,0.789535,0.789535,0.664778,0.240700,0.976589
2,3.876651,0.520057,0.793947,0.793947,0.685474,0.230301,0.981241


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.91it/s]



Full training process finished.
Training time: 5.93 hours
Completed optimizer steps: 83,058
Expected optimizer steps: 83,058
Peak allocated VRAM: 4.15 GB
Final average training loss: 4.284197986994275
Affected parameter tensors: 0
Non-finite parameter values: 0
All parameters remain finite: True
Best checkpoint: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training/checkpoint-83058
Best validation Micro F1: 0.7939468346964691
Training succeeded: True
Summary file: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training/training_summary.json

Recorded validation results:
{'epoch': 1.0, 'eval_loss': 0.522400975227356, 'eval_micro_f1': 0.7895354505569899, 'eval_accuracy': 0.7895354505569899, 'eval_macro_f1': 0.6647784382324081, 'eval_mae': 0.24070028466366958, 'eval_within_1': 0.9765889709378254}
{'epoch': 2.0, 'eval_loss': 0.5200571417808533, 'eval_micro_f1': 0.7939468346964691, 'eval

<div dir="rtl" align="right">

## ثبت نتیجه آموزش کامل و بررسی Checkpoint نهایی

در این مرحله نتیجه نهایی مدل در یک فایل جدا از نتایج Pilot ثبت می‌شود و فایل‌های اصلی Checkpoint برنده بررسی می‌شوند.

هیچ آموزش یا پیش‌بینی‌ای در این سلول انجام نمی‌شود.

</div>

In [81]:
from pathlib import Path
from datetime import datetime
import json

import pandas as pd


FULL_TRAINING_SUMMARY_PATH = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_full_training/training_summary.json"
)

FULL_RESULTS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "deberta_v3_small_full_training_results.csv"
)


# خواندن خلاصه آموزش
if not FULL_TRAINING_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Training summary not found: "
        f"{FULL_TRAINING_SUMMARY_PATH}"
    )

training_summary = json.loads(
    FULL_TRAINING_SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

if training_summary.get("training_succeeded") is not True:
    raise RuntimeError(
        "The full training summary does not indicate success."
    )


# بررسی Checkpoint برنده
best_checkpoint = Path(
    training_summary["best_checkpoint"]
)

if not best_checkpoint.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found: {best_checkpoint}"
    )

model_file_candidates = [
    best_checkpoint / "model.safetensors",
    best_checkpoint / "model.safetensors.index.json",
    best_checkpoint / "pytorch_model.bin",
]

existing_model_files = [
    path
    for path in model_file_candidates
    if path.exists()
]

required_checkpoint_files = [
    best_checkpoint / "config.json",
    best_checkpoint / "trainer_state.json",
]

missing_required_files = [
    path.name
    for path in required_checkpoint_files
    if not path.exists()
]

if not existing_model_files:
    raise FileNotFoundError(
        "No saved model-weight file was found "
        "inside the best checkpoint."
    )

if missing_required_files:
    raise FileNotFoundError(
        f"Missing checkpoint files: "
        f"{missing_required_files}"
    )


# استخراج بهترین نتیجه Validation
validation_records = training_summary[
    "validation_records"
]

best_validation_record = max(
    validation_records,
    key=lambda record: record["eval_micro_f1"],
)


# ساخت ردیف نتیجه نهایی
result_row = {
    "model_name": training_summary["model_name"],
    "training_scope": "full_dataset",
    "training_samples": training_summary[
        "training_samples"
    ],
    "validation_samples": training_summary[
        "validation_samples"
    ],
    "epochs": training_summary["epochs"],
    "best_epoch": best_validation_record["epoch"],
    "eval_loss": best_validation_record["eval_loss"],
    "micro_f1": best_validation_record[
        "eval_micro_f1"
    ],
    "accuracy": best_validation_record[
        "eval_accuracy"
    ],
    "macro_f1": best_validation_record[
        "eval_macro_f1"
    ],
    "mae": best_validation_record["eval_mae"],
    "within_1": best_validation_record[
        "eval_within_1"
    ],
    "training_hours": training_summary[
        "training_hours"
    ],
    "peak_vram_gb": training_summary[
        "peak_vram_gb"
    ],
    "completed_steps": training_summary[
        "completed_steps"
    ],
    "best_checkpoint": str(best_checkpoint),
    "all_parameters_finite": (
        training_summary[
            "non_finite_parameter_values"
        ]
        == 0
    ),
    "registered_at": datetime.now().isoformat(),
}

new_result_df = pd.DataFrame([result_row])

FULL_RESULTS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ثبت Idempotent؛ از تکرار همان Checkpoint جلوگیری می‌شود
if FULL_RESULTS_PATH.exists():
    existing_results_df = pd.read_csv(
        FULL_RESULTS_PATH
    )

    if "best_checkpoint" in existing_results_df.columns:
        existing_results_df = existing_results_df[
            existing_results_df["best_checkpoint"]
            != str(best_checkpoint)
        ]

    final_results_df = pd.concat(
        [
            existing_results_df,
            new_result_df,
        ],
        ignore_index=True,
    )
else:
    final_results_df = new_result_df

final_results_df.to_csv(
    FULL_RESULTS_PATH,
    index=False,
)


print("Full-training result registered successfully.")
print("Best checkpoint:", best_checkpoint)
print(
    "Detected model file:",
    existing_model_files[0].name,
)
print(
    "Best epoch:",
    best_validation_record["epoch"],
)
print(
    "Final Micro F1:",
    round(
        best_validation_record["eval_micro_f1"],
        6,
    ),
)
print(
    "Improvement over pilot:",
    round(
        best_validation_record["eval_micro_f1"]
        - 0.7702,
        6,
    ),
)
print(
    "Result file:",
    FULL_RESULTS_PATH,
)
print(
    "Rows in result file:",
    len(final_results_df),
)

Full-training result registered successfully.
Best checkpoint: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training/checkpoint-83058
Detected model file: model.safetensors
Best epoch: 2.0
Final Micro F1: 0.793947
Improvement over pilot: 0.023747
Result file: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/deberta_v3_small_full_training_results.csv
Rows in result file: 1


<div dir="rtl" align="right">

## بررسی فایل‌های داده Test

در این مرحله فقط فایل‌های موجود در پوشه داده‌های NLP، پسوند و حجم آن‌ها نمایش داده می‌شود تا فایل Test اصلی را دقیق شناسایی کنیم.

هیچ فایلی بارگذاری یا تغییری داده نمی‌شود.

</div>

In [82]:
from pathlib import Path


NLP_DATA_DIR = Path(
    "/home/ali/datasets/AI_Project2/nlp"
)

if not NLP_DATA_DIR.exists():
    raise FileNotFoundError(
        f"NLP data directory not found: {NLP_DATA_DIR}"
    )

data_files = sorted(
    path
    for path in NLP_DATA_DIR.rglob("*")
    if path.is_file()
)

if not data_files:
    raise RuntimeError(
        f"No files were found inside: {NLP_DATA_DIR}"
    )

print("NLP data directory:", NLP_DATA_DIR)
print("Number of files:", len(data_files))
print()

for path in data_files:
    relative_path = path.relative_to(NLP_DATA_DIR)
    size_mb = path.stat().st_size / 1024**2

    print(
        f"{relative_path} | "
        f"suffix={path.suffix or '[none]'} | "
        f"size={size_mb:.2f} MB"
    )

NLP data directory: /home/ali/datasets/AI_Project2/nlp
Number of files: 3

nlp_test_clean.csv | suffix=.csv | size=41.92 MB
nlp_train_clean.csv | suffix=.csv | size=1352.70 MB
nlp_validation_clean.csv | suffix=.csv | size=338.17 MB


<div dir="rtl" align="right">

## بررسی ساختار داده Test

در این مرحله فایل `nlp_test_clean.csv` بارگذاری می‌شود و موارد زیر بررسی می‌شوند:

- تعداد سطرها و ستون‌ها
- نام ستون‌ها و نوع داده‌ها
- تعداد مقادیر خالی
- چند نمونه از داده‌ها
- وجود یا نبود ستون هدف

هیچ پیش‌بینی یا تغییری روی فایل انجام نمی‌شود.

</div>

In [83]:
from pathlib import Path

import pandas as pd


TEST_DATA_PATH = Path(
    "/home/ali/datasets/AI_Project2/nlp/nlp_test_clean.csv"
)

if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Test file not found: {TEST_DATA_PATH}"
    )

test_df = pd.read_csv(
    TEST_DATA_PATH,
    low_memory=False,
)

print("Test file loaded successfully.")
print("Path:", TEST_DATA_PATH)
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nColumn dtypes:")
print(test_df.dtypes)

print("\nMissing values:")
print(
    test_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nTarget column exists:", "overall" in test_df.columns)

print("\nFirst 3 rows:")
display(test_df.head(3))

Test file loaded successfully.
Path: /home/ali/datasets/AI_Project2/nlp/nlp_test_clean.csv
Shape: (20000, 15)
Columns: ['row_id', 'vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'reviewText_raw', 'reviewText_clean', 'brand', 'title']

Column dtypes:
row_id                int64
vote                float64
verified               bool
reviewTime              str
reviewerID              str
asin                    str
style                   str
reviewerName            str
reviewText              str
summary                 str
unixReviewTime        int64
reviewText_raw          str
reviewText_clean        str
brand                   str
title                   str
dtype: object

Missing values:
vote                15067
style                8496
brand                  31
reviewerName            4
summary                 2
verified                0
row_id                  0
asin                    0
reviewerID     

,row_id,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime,reviewText_raw,reviewText_clean,brand,title
0,0,NaN,True,2018-06-04,A20GGWE66JW9X2,B006Z394GM,{'Color:': ' FPS01-C'},Brian C Toner,The name and description of this device are mi...,The prize for most useless invention of all ti...,1528070400,The name and description of this device are mi...,The name and description of this device are mi...,SilverStone Technology,Silverstone Tek External Slim Optical Drive En...
1,1,NaN,True,2018-02-18,ARYJWXHEYHG9M,B005BE058W,"{'Size:': ' 1000W', 'Style:': ' G2'}",Snake,One of the molex connectors on the power suppl...,MELTED MOLEX CONNECTOR,1518912000,One of the molex connectors on the power suppl...,One of the molex connectors on the power suppl...,EVGA,"EVGA Supernova 750 G3, 80 Plus Gold 750W, Full..."
2,2,NaN,True,2018-01-20,A10LHZ7WFZ7HLL,B01DA0YCNC,NaN,Amazon Customer,Remote constantly disconnects/ Roku player fre...,Bricked on the regular,1516406400,Remote constantly disconnects/ Roku player fre...,Remote constantly disconnects/ Roku player fre...,Roku,Roku Streaming Stick (3600R) - HD Streaming Pl...


<div dir="rtl" align="right">

## آماده‌سازی ورودی‌های داده Test

در این مرحله:

- ستون‌های `summary` و `reviewText_clean` پاک‌سازی می‌شوند.
- مقادیر خالی با متن خالی جایگزین می‌شوند.
- ترتیب اصلی سطرها با استفاده از `row_id` حفظ می‌شود.
- نمونه‌هایی که هر دو ورودی متنی آن‌ها خالی هستند شناسایی می‌شوند.

هنوز Tokenization یا پیش‌بینی انجام نمی‌شود.

</div>

In [84]:
import pandas as pd


required_test_columns = {
    "row_id",
    "summary",
    "reviewText_clean",
}

missing_test_columns = (
    required_test_columns
    - set(test_df.columns)
)

if missing_test_columns:
    raise KeyError(
        f"Missing required test columns: "
        f"{sorted(missing_test_columns)}"
    )


# بررسی شناسه سطرها
if test_df["row_id"].isna().any():
    raise ValueError(
        "Test row_id contains missing values."
    )

if test_df["row_id"].duplicated().any():
    raise ValueError(
        "Test row_id contains duplicate values."
    )


# ساخت ورودی متناظر با داده‌های آموزش
test_pairs = pd.DataFrame({
    "row_id": test_df["row_id"].astype("int64"),
    "summary": (
        test_df["summary"]
        .fillna("")
        .astype(str)
        .str.strip()
    ),
    "review": (
        test_df["reviewText_clean"]
        .fillna("")
        .astype(str)
        .str.strip()
    ),
})


empty_summary_count = int(
    test_pairs["summary"].eq("").sum()
)

empty_review_count = int(
    test_pairs["review"].eq("").sum()
)

fully_empty_mask = (
    test_pairs["summary"].eq("")
    & test_pairs["review"].eq("")
)

fully_empty_count = int(
    fully_empty_mask.sum()
)


# بررسی حفظ ترتیب
if not test_pairs["row_id"].equals(
    test_df["row_id"].astype("int64")
):
    raise RuntimeError(
        "Test row order was not preserved."
    )


print("Test pairs created successfully.")
print("Shape:", test_pairs.shape)
print("Columns:", test_pairs.columns.tolist())
print("Unique row IDs:", test_pairs["row_id"].nunique())
print("Empty summaries:", f"{empty_summary_count:,}")
print("Empty reviews:", f"{empty_review_count:,}")
print("Fully empty inputs:", f"{fully_empty_count:,}")

print("\nFirst 3 prepared inputs:")
display(test_pairs.head(3))

Test pairs created successfully.
Shape: (20000, 3)
Columns: ['row_id', 'summary', 'review']
Unique row IDs: 20000
Empty summaries: 2
Empty reviews: 0
Fully empty inputs: 0

First 3 prepared inputs:


,row_id,summary,review
0,0,The prize for most useless invention of all ti...,The name and description of this device are mi...
1,1,MELTED MOLEX CONNECTOR,One of the molex connectors on the power suppl...
2,2,Bricked on the regular,Remote constantly disconnects/ Roku player fre...


<div dir="rtl" align="right">

## Tokenize کردن داده Test

در این مرحله، ستون‌های `summary` و متن Review به‌صورت ورودی دوتایی به Tokenizer مدل داده می‌شوند.

ویژگی‌های این مرحله:

- استفاده از همان Tokenizer مدل نهایی
- حداکثر طول ۵۱۲ توکن
- فعال‌بودن Truncation
- بدون Padding ثابت؛ Padding هنگام پیش‌بینی انجام می‌شود
- حفظ کامل ترتیب ۲۰٬۰۰۰ سطر Test
- ذخیره خروجی Tokenizeشده روی دیسک

هنوز هیچ پیش‌بینی‌ای انجام نمی‌شود.

</div>

In [85]:
from pathlib import Path

from datasets import Dataset, load_from_disk


TEST_TOKENIZED_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_test_tokenized"
)


test_row_ids = test_pairs["row_id"].to_numpy(copy=True)


def tokenize_test_batch(batch):
    return tokenizer(
        batch["summary"],
        batch["review"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )


if TEST_TOKENIZED_DIR.exists():
    print(
        "Existing tokenized Test dataset found; "
        "loading it instead of overwriting."
    )

    tokenized_test_dataset = load_from_disk(
        str(TEST_TOKENIZED_DIR)
    )

else:
    raw_test_dataset = Dataset.from_dict({
        "summary": test_pairs["summary"].tolist(),
        "review": test_pairs["review"].tolist(),
    })

    tokenized_test_dataset = raw_test_dataset.map(
        tokenize_test_batch,
        batched=True,
        batch_size=1_000,
        remove_columns=[
            "summary",
            "review",
        ],
        desc="Tokenizing Test data",
    )

    tokenized_test_dataset.save_to_disk(
        str(TEST_TOKENIZED_DIR)
    )


# بررسی‌های صحت
if len(tokenized_test_dataset) != len(test_pairs):
    raise RuntimeError(
        "Tokenized Test length does not match "
        "the original Test data."
    )

required_token_columns = {
    "input_ids",
    "attention_mask",
}

missing_token_columns = (
    required_token_columns
    - set(tokenized_test_dataset.column_names)
)

if missing_token_columns:
    raise KeyError(
        f"Missing token columns: "
        f"{sorted(missing_token_columns)}"
    )


inspection_indices = [
    0,
    len(tokenized_test_dataset) // 2,
    len(tokenized_test_dataset) - 1,
]

for index in inspection_indices:
    sample = tokenized_test_dataset[index]

    input_length = len(sample["input_ids"])
    mask_length = len(sample["attention_mask"])

    if input_length != mask_length:
        raise RuntimeError(
            f"Length mismatch at Test index {index}."
        )

    if not 1 <= input_length <= MAX_LENGTH:
        raise RuntimeError(
            f"Invalid token length at index {index}: "
            f"{input_length}"
        )

    if not all(
        value in (0, 1)
        for value in sample["attention_mask"]
    ):
        raise RuntimeError(
            f"Invalid attention mask at index {index}."
        )


token_lengths = [
    len(tokenized_test_dataset[index]["input_ids"])
    for index in inspection_indices
]

saved_size_mb = sum(
    path.stat().st_size
    for path in TEST_TOKENIZED_DIR.rglob("*")
    if path.is_file()
) / 1024**2


print("\nTest tokenization completed successfully.")
print("Samples:", f"{len(tokenized_test_dataset):,}")
print(
    "Columns:",
    tokenized_test_dataset.column_names,
)
print("Features:", tokenized_test_dataset.features)
print("Inspection indices:", inspection_indices)
print("Inspection token lengths:", token_lengths)
print("Maximum allowed length:", MAX_LENGTH)
print(
    "Original row IDs preserved:",
    len(test_row_ids) == len(tokenized_test_dataset),
)
print(
    "Saved size:",
    round(saved_size_mb, 2),
    "MB",
)
print("Saved directory:", TEST_TOKENIZED_DIR)

Saving the dataset (1/1 shards): 100%|██████████| 20000/20000 [00:00<00:00, 2940688.49 examples/s]


Test tokenization completed successfully.
Samples: 20,000
Columns: ['input_ids', 'token_type_ids', 'attention_mask']
Features: {'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}
Inspection indices: [0, 10000, 19999]
Inspection token lengths: [287, 306, 503]
Maximum allowed length: 512
Original row IDs preserved: True
Saved size: 17.08 MB
Saved directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_test_tokenized


<div dir="rtl" align="right">

## پیش‌بینی داده Test و ساخت فایل Submission

در این مرحله:

- از مدل نهایی `DeBERTa-v3-small` استفاده می‌شود.
- برای تمام ۲۰٬۰۰۰ نمونه Test پیش‌بینی گرفته می‌شود.
- برچسب‌های داخلی صفر تا چهار به امتیازهای یک تا پنج تبدیل می‌شوند.
- فایل رسمی `q2_submission.csv` فقط با ستون `predicted` ساخته می‌شود.
- یک فایل جزئی‌تر شامل `row_id`، پیش‌بینی و میزان اطمینان مدل نیز ذخیره می‌شود.
- تعداد سطرها، مقادیر پیش‌بینی‌شده و ترتیب داده‌ها کنترل می‌شوند.

</div>

In [86]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch


FINAL_CHECKPOINT = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_full_training/checkpoint-83058"
)

NLP_OUTPUT_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp"
)

SUBMISSION_PATH = (
    NLP_OUTPUT_DIR / "q2_submission.csv"
)

DETAILED_PREDICTIONS_PATH = (
    NLP_OUTPUT_DIR
    / "deberta_v3_small_test_predictions.csv"
)


# بررسی فایل‌ها و مدل
if not FINAL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Final checkpoint not found: {FINAL_CHECKPOINT}"
    )

if len(tokenized_test_dataset) != len(test_pairs):
    raise RuntimeError(
        "Tokenized Test data and original Test data "
        "have different lengths."
    )

if trainer.state.best_model_checkpoint != str(
    FINAL_CHECKPOINT
):
    raise RuntimeError(
        "Trainer is not pointing to the expected "
        "best checkpoint."
    )

if trainer.model.config.num_labels != 5:
    raise ValueError(
        f"Expected 5 labels, but found "
        f"{trainer.model.config.num_labels}."
    )

if (
    SUBMISSION_PATH.exists()
    or DETAILED_PREDICTIONS_PATH.exists()
):
    raise FileExistsError(
        "A Test prediction output already exists. "
        "Do not overwrite it without inspection."
    )


NLP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.cuda.empty_cache()

prediction_start_time = time.perf_counter()

prediction_output = trainer.predict(
    tokenized_test_dataset
)

prediction_minutes = (
    time.perf_counter() - prediction_start_time
) / 60


# استخراج Logitها
test_logits = prediction_output.predictions

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

test_logits = np.asarray(test_logits)

if test_logits.shape != (
    len(test_pairs),
    5,
):
    raise RuntimeError(
        f"Unexpected logits shape: {test_logits.shape}"
    )

if not np.isfinite(test_logits).all():
    raise RuntimeError(
        "Test logits contain NaN or Inf values."
    )


# تبدیل خروجی مدل به امتیازهای ۱ تا ۵
predicted_class_ids = np.argmax(
    test_logits,
    axis=1,
)

predicted_ratings = (
    predicted_class_ids + 1
).astype("int64")

test_probabilities = torch.softmax(
    torch.from_numpy(test_logits).float(),
    dim=1,
).numpy()

prediction_confidence = np.max(
    test_probabilities,
    axis=1,
)


# بررسی دامنه پیش‌بینی‌ها
valid_ratings = {1, 2, 3, 4, 5}

if not set(
    np.unique(predicted_ratings)
).issubset(valid_ratings):
    raise ValueError(
        "Predicted ratings contain invalid values."
    )


# فایل رسمی مسابقه
submission_df = pd.DataFrame({
    "predicted": predicted_ratings,
})

if submission_df.shape != (20_000, 1):
    raise RuntimeError(
        f"Unexpected submission shape: "
        f"{submission_df.shape}"
    )

if submission_df["predicted"].isna().any():
    raise RuntimeError(
        "Submission contains missing predictions."
    )

submission_df.to_csv(
    SUBMISSION_PATH,
    index=False,
)


# فایل جزئی برای تحلیل و کنترل
detailed_predictions_df = pd.DataFrame({
    "row_id": test_row_ids,
    "predicted": predicted_ratings,
    "confidence": prediction_confidence,
})

detailed_predictions_df.to_csv(
    DETAILED_PREDICTIONS_PATH,
    index=False,
)


# بازخوانی فایل رسمی برای کنترل نهایی
saved_submission_df = pd.read_csv(
    SUBMISSION_PATH
)

if saved_submission_df.columns.tolist() != [
    "predicted"
]:
    raise RuntimeError(
        "Submission column structure is incorrect."
    )

if len(saved_submission_df) != len(test_pairs):
    raise RuntimeError(
        "Saved submission row count is incorrect."
    )

if not np.array_equal(
    saved_submission_df["predicted"].to_numpy(),
    predicted_ratings,
):
    raise RuntimeError(
        "Saved predictions differ from "
        "the generated predictions."
    )


prediction_distribution = (
    submission_df["predicted"]
    .value_counts()
    .sort_index()
)

print("Test prediction completed successfully.")
print(
    "Prediction time:",
    round(prediction_minutes, 2),
    "minutes",
)
print("Logits shape:", test_logits.shape)
print(
    "Average confidence:",
    round(
        float(prediction_confidence.mean()),
        6,
    ),
)
print(
    "Minimum confidence:",
    round(
        float(prediction_confidence.min()),
        6,
    ),
)
print(
    "Maximum confidence:",
    round(
        float(prediction_confidence.max()),
        6,
    ),
)

print("\nPrediction distribution:")
print(prediction_distribution)

print("\nSubmission verification:")
print("Shape:", saved_submission_df.shape)
print(
    "Columns:",
    saved_submission_df.columns.tolist(),
)
print(
    "Missing predictions:",
    int(
        saved_submission_df[
            "predicted"
        ].isna().sum()
    ),
)
print(
    "Unique predicted ratings:",
    sorted(
        saved_submission_df[
            "predicted"
        ].unique().tolist()
    ),
)

print("\nSaved files:")
print("Official submission:", SUBMISSION_PATH)
print(
    "Detailed predictions:",
    DETAILED_PREDICTIONS_PATH,
)

Test prediction completed successfully.
Prediction time: 2.51 minutes
Logits shape: (20000, 5)
Average confidence: 0.767388
Minimum confidence: 0.248561
Maximum confidence: 0.998373

Prediction distribution:
predicted
1    4495
2    3105
3    3886
4    3528
5    4986
Name: count, dtype: int64

Submission verification:
Shape: (20000, 1)
Columns: ['predicted']
Missing predictions: 0
Unique predicted ratings: [1, 2, 3, 4, 5]

Saved files:
Official submission: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/q2_submission.csv
Detailed predictions: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/deberta_v3_small_test_predictions.csv


<div dir="rtl" align="right">

## ذخیره نسخه نهایی مدل و Tokenizer

در این مرحله، بهترین مدل آموزش‌دیده به‌صورت یک بسته مستقل برای پیش‌بینی ذخیره می‌شود.

این بسته شامل موارد زیر است:

- وزن‌های مدل نهایی
- تنظیمات مدل
- Tokenizer
- نگاشت برچسب‌های صفر تا چهار به امتیازهای یک تا پنج
- مشخصات ورودی و حداکثر طول توکن
- هش فایل‌ها برای کنترل سلامت نسخه کپی‌شده

ابتدا مدل در یک پوشه محلی موقت Export می‌شود، سپس نسخه تأییدشده به Google Drive منتقل خواهد شد.

</div>

In [87]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import shutil

import torch


SOURCE_CHECKPOINT = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_full_training/checkpoint-83058"
)

LOCAL_EXPORT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_final_export"
)

DRIVE_FINAL_MODEL_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)

DRIVE_TEMP_MODEL_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    ".deberta_v3_small_final_copying"
)


def calculate_sha256(file_path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def build_file_manifest(root_directory):
    manifest = {}

    for file_path in sorted(root_directory.rglob("*")):
        if not file_path.is_file():
            continue

        relative_path = str(
            file_path.relative_to(root_directory)
        )

        manifest[relative_path] = {
            "size_bytes": file_path.stat().st_size,
            "sha256": calculate_sha256(file_path),
        }

    return manifest


# بررسی مبدا و جلوگیری از بازنویسی ناخواسته
if not SOURCE_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Source checkpoint not found: {SOURCE_CHECKPOINT}"
    )

if trainer.state.best_model_checkpoint != str(
    SOURCE_CHECKPOINT
):
    raise RuntimeError(
        "Trainer does not point to the expected "
        "best checkpoint."
    )

for guarded_path in [
    LOCAL_EXPORT_DIR,
    DRIVE_FINAL_MODEL_DIR,
    DRIVE_TEMP_MODEL_DIR,
]:
    if guarded_path.exists():
        raise FileExistsError(
            f"Export path already exists: {guarded_path}"
        )


# بررسی نهایی سلامت پارامترهای مدل
non_finite_values = 0

for parameter in trainer.model.parameters():
    if parameter.is_floating_point():
        non_finite_values += int(
            (~torch.isfinite(parameter.detach()))
            .sum()
            .item()
        )

if non_finite_values != 0:
    raise RuntimeError(
        "The final model contains non-finite values."
    )


# انتقال مدل به CPU و Export محلی
trainer.model.to("cpu")
torch.cuda.empty_cache()

LOCAL_EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

trainer.model.save_pretrained(
    str(LOCAL_EXPORT_DIR),
    safe_serialization=True,
    max_shard_size="2GB",
)

tokenizer.save_pretrained(
    str(LOCAL_EXPORT_DIR)
)


# افزودن اطلاعات پروژه
project_metadata = {
    "model_name": "microsoft/deberta-v3-small",
    "source_checkpoint": str(SOURCE_CHECKPOINT),
    "task": "Amazon electronics review rating prediction",
    "input_fields": [
        "summary",
        "reviewText_clean",
    ],
    "input_mode": "tokenizer text pair",
    "max_length": 512,
    "internal_labels": [0, 1, 2, 3, 4],
    "predicted_ratings": [1, 2, 3, 4, 5],
    "label_conversion": "predicted_rating = argmax(logits) + 1",
    "validation_micro_f1": 0.7939468346964691,
    "validation_macro_f1": 0.685473633162428,
    "validation_mae": 0.23030073242216886,
    "validation_within_1": 0.9812410854532652,
    "training_samples": 664_451,
    "validation_samples": 166_161,
    "exported_at": datetime.now().isoformat(),
}

metadata_path = (
    LOCAL_EXPORT_DIR / "project_metadata.json"
)

metadata_path.write_text(
    json.dumps(
        project_metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ساخت Manifest محلی
local_file_manifest = build_file_manifest(
    LOCAL_EXPORT_DIR
)

manifest_payload = {
    "created_at": datetime.now().isoformat(),
    "number_of_files": len(local_file_manifest),
    "files": local_file_manifest,
}

manifest_path = (
    LOCAL_EXPORT_DIR / "file_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest_payload,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# Manifest پس از اضافه‌شدن خود فایل Manifest دوباره ساخته می‌شود
local_file_manifest = build_file_manifest(
    LOCAL_EXPORT_DIR
)


# کپی موقت به Google Drive
DRIVE_TEMP_MODEL_DIR.parent.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copytree(
    LOCAL_EXPORT_DIR,
    DRIVE_TEMP_MODEL_DIR,
)


# مقایسه اندازه و هش تمام فایل‌ها
drive_file_manifest = build_file_manifest(
    DRIVE_TEMP_MODEL_DIR
)

if local_file_manifest != drive_file_manifest:
    raise RuntimeError(
        "Google Drive copy verification failed. "
        "Local and Drive manifests differ."
    )


# نهایی‌کردن نام پوشه فقط پس از تأیید کامل
DRIVE_TEMP_MODEL_DIR.rename(
    DRIVE_FINAL_MODEL_DIR
)


# بررسی فایل‌های ضروری
required_files = [
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "project_metadata.json",
    "file_manifest.json",
]

missing_files = [
    file_name
    for file_name in required_files
    if not (
        DRIVE_FINAL_MODEL_DIR / file_name
    ).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing final model files: {missing_files}"
    )


total_export_size_gb = sum(
    file_path.stat().st_size
    for file_path in DRIVE_FINAL_MODEL_DIR.rglob("*")
    if file_path.is_file()
) / 1024**3


print("Final model export completed successfully.")
print("Source checkpoint:", SOURCE_CHECKPOINT)
print("Local export:", LOCAL_EXPORT_DIR)
print("Google Drive model:", DRIVE_FINAL_MODEL_DIR)
print("Number of exported files:", len(local_file_manifest))
print(
    "Total model package size:",
    round(total_export_size_gb, 3),
    "GB",
)
print("Missing required files:", missing_files)
print(
    "Local and Drive copies match:",
    local_file_manifest == drive_file_manifest,
)
print(
    "Validation Micro F1:",
    project_metadata["validation_micro_f1"],
)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.31it/s]


Final model export completed successfully.
Source checkpoint: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training/checkpoint-83058
Local export: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_final_export
Google Drive model: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_small_final
Number of exported files: 6
Total model package size: 0.536 GB
Missing required files: []
Local and Drive copies match: True
Validation Micro F1: 0.7939468346964691


<div dir="rtl" align="right">

## تست مستقل نسخه نهایی مدل

در این مرحله مدل و Tokenizer مستقیماً از پوشه نهایی Google Drive بارگذاری می‌شوند.

سپس روی چند نمونه ثابت از داده Test:

- خروجی مدل Exportشده گرفته می‌شود.
- با خروجی مدل فعلی مقایسه می‌شود.
- یکسان‌بودن Tokenization و پیش‌بینی‌ها بررسی می‌شود.

هیچ فایل جدیدی ساخته یا بازنویسی نمی‌شود.

</div>

In [88]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


FINAL_MODEL_DIR = (
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)

verification_indices = [
    0,
    len(test_pairs) // 2,
    len(test_pairs) - 1,
]

verification_batch = test_pairs.iloc[
    verification_indices
]


# بارگذاری مستقل مدل و Tokenizer
reloaded_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

reloaded_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        FINAL_MODEL_DIR,
        dtype=torch.float32,
    )
)

reloaded_model.to("cpu")
reloaded_model.eval()

trainer.model.to("cpu")
trainer.model.eval()


# Tokenize با Tokenizer اصلی
original_inputs = tokenizer(
    verification_batch["summary"].tolist(),
    verification_batch["review"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True,
    return_tensors="pt",
)


# Tokenize با Tokenizer Exportشده
reloaded_inputs = reloaded_tokenizer(
    verification_batch["summary"].tolist(),
    verification_batch["review"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True,
    return_tensors="pt",
)


# بررسی یکسان‌بودن Tokenization
if set(original_inputs.keys()) != set(
    reloaded_inputs.keys()
):
    raise RuntimeError(
        "Tokenizer output fields are different."
    )

tokenization_matches = all(
    torch.equal(
        original_inputs[key],
        reloaded_inputs[key],
    )
    for key in original_inputs
)

if not tokenization_matches:
    raise RuntimeError(
        "Reloaded tokenizer produced different tokens."
    )


# گرفتن خروجی هر دو مدل
with torch.inference_mode():
    original_logits = trainer.model(
        **original_inputs
    ).logits.cpu()

    reloaded_logits = reloaded_model(
        **reloaded_inputs
    ).logits.cpu()


maximum_logit_difference = float(
    torch.max(
        torch.abs(
            original_logits - reloaded_logits
        )
    ).item()
)

logits_match = torch.allclose(
    original_logits,
    reloaded_logits,
    rtol=1e-5,
    atol=1e-6,
)

original_predictions = (
    torch.argmax(original_logits, dim=1) + 1
).numpy()

reloaded_predictions = (
    torch.argmax(reloaded_logits, dim=1) + 1
).numpy()

predictions_match = np.array_equal(
    original_predictions,
    reloaded_predictions,
)

if not logits_match:
    raise RuntimeError(
        f"Reloaded model logits differ. "
        f"Maximum difference: "
        f"{maximum_logit_difference}"
    )

if not predictions_match:
    raise RuntimeError(
        "Reloaded model predictions differ."
    )


print("Independent model reload test passed.")
print("Model directory:", FINAL_MODEL_DIR)
print("Verification indices:", verification_indices)
print("Tokenizer outputs match:", tokenization_matches)
print("Logits match:", logits_match)
print(
    "Maximum logit difference:",
    maximum_logit_difference,
)
print("Predictions match:", predictions_match)
print(
    "Predicted ratings:",
    reloaded_predictions.tolist(),
)
print(
    "Model parameter dtypes:",
    {
        parameter.dtype
        for parameter in reloaded_model.parameters()
    },
)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

<div dir="rtl" align="right">

## بررسی سلامت فایل‌های Tokenizer مدل نهایی

در این مرحله:

- فایل‌های نسخه محلی و نسخه Google Drive فهرست می‌شوند.
- اندازه هر فایل نمایش داده می‌شود.
- تمام فایل‌های JSON از نظر معتبر بودن بررسی می‌شوند.
- ابتدای فایل JSON خراب نمایش داده می‌شود تا علت خطا مشخص شود.

هیچ فایلی حذف یا بازنویسی نمی‌شود.

</div>

In [89]:
from pathlib import Path
import json


LOCAL_EXPORT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_final_export"
)

DRIVE_FINAL_MODEL_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)


def inspect_export_directory(directory):
    print("=" * 80)
    print("Directory:", directory)
    print("Exists:", directory.exists())

    if not directory.exists():
        return

    files = sorted(
        path
        for path in directory.rglob("*")
        if path.is_file()
    )

    print("Number of files:", len(files))
    print("\nFiles:")

    for path in files:
        relative_path = path.relative_to(directory)
        size_bytes = path.stat().st_size

        print(
            f"- {relative_path} | "
            f"{size_bytes:,} bytes"
        )

    print("\nJSON validation:")

    json_files = [
        path
        for path in files
        if path.suffix.lower() == ".json"
    ]

    for path in json_files:
        relative_path = path.relative_to(directory)

        try:
            with path.open(
                "r",
                encoding="utf-8",
            ) as file_handle:
                parsed_content = json.load(file_handle)

            print(
                f"[VALID] {relative_path} | "
                f"type={type(parsed_content).__name__}"
            )

        except Exception as error:
            print(
                f"[INVALID] {relative_path} | "
                f"{type(error).__name__}: {error}"
            )

            with path.open("rb") as file_handle:
                first_bytes = file_handle.read(200)

            print(
                "First 200 bytes:",
                repr(first_bytes),
            )

    tokenizer_related_files = [
        path.relative_to(directory)
        for path in files
        if any(
            keyword in path.name.lower()
            for keyword in [
                "tokenizer",
                "sentencepiece",
                "spm",
                "vocab",
            ]
        )
    ]

    print("\nTokenizer-related files:")
    for path in tokenizer_related_files:
        print("-", path)


inspect_export_directory(
    LOCAL_EXPORT_DIR
)

inspect_export_directory(
    DRIVE_FINAL_MODEL_DIR
)

Directory: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_final_export
Exists: True
Number of files: 6

Files:
- config.json | 1,168 bytes
- file_manifest.json | 809 bytes
- model.safetensors | 567,607,780 bytes
- project_metadata.json | 851 bytes
- tokenizer.json | 8,339,974 bytes
- tokenizer_config.json | 510 bytes

JSON validation:
[VALID] config.json | type=dict
[VALID] file_manifest.json | type=dict
[VALID] project_metadata.json | type=dict
[VALID] tokenizer.json | type=dict
[VALID] tokenizer_config.json | type=dict

Tokenizer-related files:
- tokenizer.json
- tokenizer_config.json
Directory: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_small_final
Exists: True
Number of files: 6

Files:
- config.json | 1,168 bytes
- file_manifest.json | 809 bytes
- model.safetensors | 567,607,780 bytes
- project_metadata.json | 851 bytes
- tokenizer.json | 0 bytes
- tokenizer_config.json | 510 bytes

JSON validation:
[VALID] config.json | t

<div dir="rtl" align="right">

## تشخیص دقیق خطای بارگذاری Tokenizer

در این مرحله:

- کلاس Tokenizer اصلی بررسی می‌شود.
- محتوای `tokenizer_config.json` نمایش داده می‌شود.
- هش فایل‌های Tokenizer در نسخه محلی و Google Drive مقایسه می‌شود.
- بارگذاری خودکار از هر دو مسیر آزمایش می‌شود.
- فایل `tokenizer.json` مستقیماً با Fast Tokenizer بارگذاری می‌شود.

هیچ فایلی تغییر یا بازنویسی نمی‌شود.

</div>

In [90]:
from pathlib import Path
import hashlib
import json

from transformers import (
    AutoTokenizer,
    PreTrainedTokenizerFast,
)


LOCAL_EXPORT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_final_export"
)

DRIVE_FINAL_MODEL_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        for chunk in iter(
            lambda: file_handle.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


print("Original tokenizer class:")
print(type(tokenizer))
print(
    f"{type(tokenizer).__module__}."
    f"{type(tokenizer).__name__}"
)

print("\nOriginal tokenizer properties:")
print("is_fast:", getattr(tokenizer, "is_fast", None))
print("vocab_size:", tokenizer.vocab_size)
print("pad_token:", tokenizer.pad_token)
print("bos_token:", tokenizer.bos_token)
print("eos_token:", tokenizer.eos_token)


for directory_name, directory in [
    ("LOCAL", LOCAL_EXPORT_DIR),
    ("DRIVE", DRIVE_FINAL_MODEL_DIR),
]:
    print("\n" + "=" * 80)
    print(directory_name, directory)

    config_path = directory / "tokenizer_config.json"
    tokenizer_json_path = directory / "tokenizer.json"

    tokenizer_config = json.loads(
        config_path.read_text(encoding="utf-8")
    )

    print("tokenizer_config.json:")
    print(
        json.dumps(
            tokenizer_config,
            indent=2,
            ensure_ascii=False,
        )
    )

    print(
        "tokenizer.json size:",
        tokenizer_json_path.stat().st_size,
    )
    print(
        "tokenizer.json SHA256:",
        sha256_file(tokenizer_json_path),
    )


local_hash = sha256_file(
    LOCAL_EXPORT_DIR / "tokenizer.json"
)

drive_hash = sha256_file(
    DRIVE_FINAL_MODEL_DIR / "tokenizer.json"
)

print("\nLocal and Drive tokenizer.json match:")
print(local_hash == drive_hash)


def attempt_load(description, loader):
    print("\n" + "-" * 80)
    print(description)

    try:
        loaded_tokenizer = loader()

        print("Status: SUCCESS")
        print("Class:", type(loaded_tokenizer))
        print(
            "Full class:",
            f"{type(loaded_tokenizer).__module__}."
            f"{type(loaded_tokenizer).__name__}",
        )
        print(
            "is_fast:",
            getattr(loaded_tokenizer, "is_fast", None),
        )
        print(
            "Test encoding:",
            loaded_tokenizer(
                "test summary",
                "test review",
                truncation=True,
                max_length=512,
            ),
        )

    except Exception as error:
        print("Status: FAILED")
        print("Error type:", type(error).__name__)
        print("Error message:", repr(str(error)))


attempt_load(
    "AutoTokenizer from LOCAL export",
    lambda: AutoTokenizer.from_pretrained(
        str(LOCAL_EXPORT_DIR),
        local_files_only=True,
    ),
)

attempt_load(
    "AutoTokenizer from DRIVE export",
    lambda: AutoTokenizer.from_pretrained(
        str(DRIVE_FINAL_MODEL_DIR),
        local_files_only=True,
    ),
)

attempt_load(
    "Direct PreTrainedTokenizerFast from LOCAL tokenizer.json",
    lambda: PreTrainedTokenizerFast(
        tokenizer_file=str(
            LOCAL_EXPORT_DIR / "tokenizer.json"
        ),
        pad_token=tokenizer.pad_token,
        bos_token=tokenizer.bos_token,
        eos_token=tokenizer.eos_token,
        unk_token=tokenizer.unk_token,
    ),
)

Original tokenizer class:
<class 'transformers.models.deberta_v2.tokenization_deberta_v2.DebertaV2Tokenizer'>
transformers.models.deberta_v2.tokenization_deberta_v2.DebertaV2Tokenizer

Original tokenizer properties:
is_fast: True
vocab_size: 128000
pad_token: [PAD]
bos_token: [CLS]
eos_token: [SEP]

LOCAL /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_final_export
tokenizer_config.json:
{
  "add_prefix_space": true,
  "backend": "tokenizers",
  "bos_token": "[CLS]",
  "cls_token": "[CLS]",
  "do_lower_case": false,
  "eos_token": "[SEP]",
  "extra_special_tokens": [
    "[PAD]",
    "[CLS]",
    "[SEP]"
  ],
  "is_local": false,
  "local_files_only": false,
  "mask_token": "[MASK]",
  "model_max_length": 512,
  "pad_token": "[PAD]",
  "sep_token": "[SEP]",
  "split_by_punct": false,
  "tokenizer_class": "DebertaV2Tokenizer",
  "unk_id": 3,
  "unk_token": "[UNK]",
  "vocab_type": "spm"
}
tokenizer.json size: 8339974
tokenizer.json SHA256: a2

<div dir="rtl" align="right">

## بررسی خلاصه روش‌های بارگذاری Tokenizer

این سلول فقط مشخص می‌کند کدام روش بارگذاری موفق می‌شود و هیچ فایلی را تغییر نمی‌دهد.

</div>

In [91]:
from transformers import AutoTokenizer, PreTrainedTokenizerFast


load_tests = {
    "auto_local": lambda: AutoTokenizer.from_pretrained(
        str(LOCAL_EXPORT_DIR),
        local_files_only=True,
    ),
    "auto_drive": lambda: AutoTokenizer.from_pretrained(
        str(DRIVE_FINAL_MODEL_DIR),
        local_files_only=True,
    ),
    "direct_fast_local": lambda: PreTrainedTokenizerFast(
        tokenizer_file=str(
            LOCAL_EXPORT_DIR / "tokenizer.json"
        ),
        pad_token="[PAD]",
        bos_token="[CLS]",
        eos_token="[SEP]",
        cls_token="[CLS]",
        sep_token="[SEP]",
        unk_token="[UNK]",
    ),
}

for test_name, loader in load_tests.items():
    try:
        loaded = loader()

        encoded = loaded(
            "test summary",
            "test review",
            truncation=True,
            max_length=512,
        )

        print({
            "test": test_name,
            "status": "SUCCESS",
            "class": type(loaded).__name__,
            "input_ids": encoded["input_ids"],
        })

    except Exception as error:
        print({
            "test": test_name,
            "status": "FAILED",
            "error_type": type(error).__name__,
            "error": str(error)[:200],
        })

{'test': 'auto_local', 'status': 'SUCCESS', 'class': 'DebertaV2Tokenizer', 'input_ids': [1, 1010, 5310, 2, 1010, 1078, 2]}
{'test': 'auto_drive', 'status': 'SUCCESS', 'class': 'DebertaV2Tokenizer', 'input_ids': [1, 1010, 5310, 2, 1010, 1078, 2]}
{'test': 'direct_fast_local', 'status': 'SUCCESS', 'class': 'TokenizersBackend', 'input_ids': [1, 1010, 5310, 2, 1010, 1078, 2]}


<div dir="rtl" align="right">

## تأیید مستقل بسته نهایی مدل

مدل و Tokenizer مستقیماً از نسخه ذخیره‌شده در Google Drive بارگذاری می‌شوند و خروجی آن‌ها روی سه نمونه ثابت با مدل فعلی مقایسه می‌شود.

هیچ فایلی تغییر نمی‌کند.

</div>

In [92]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


FINAL_MODEL_DIR = (
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)

verification_indices = [
    0,
    len(test_pairs) // 2,
    len(test_pairs) - 1,
]

verification_batch = test_pairs.iloc[
    verification_indices
]


# بارگذاری مستقل از Google Drive
reloaded_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR,
    local_files_only=True,
)

reloaded_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        FINAL_MODEL_DIR,
        dtype=torch.float32,
        local_files_only=True,
    )
)

reloaded_model.to("cpu")
reloaded_model.eval()

trainer.model.to("cpu")
trainer.model.eval()


# Tokenization با نسخه فعلی
original_inputs = tokenizer(
    verification_batch["summary"].tolist(),
    verification_batch["review"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True,
    return_tensors="pt",
)


# Tokenization با نسخه Exportشده
reloaded_inputs = reloaded_tokenizer(
    verification_batch["summary"].tolist(),
    verification_batch["review"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True,
    return_tensors="pt",
)


if set(original_inputs) != set(reloaded_inputs):
    raise RuntimeError(
        "Tokenizer output fields are different."
    )

tokenization_matches = all(
    torch.equal(
        original_inputs[key],
        reloaded_inputs[key],
    )
    for key in original_inputs
)

if not tokenization_matches:
    raise RuntimeError(
        "Exported tokenizer produced different tokens."
    )


# مقایسه خروجی دو مدل
with torch.inference_mode():
    original_logits = trainer.model(
        **original_inputs
    ).logits.cpu()

    reloaded_logits = reloaded_model(
        **reloaded_inputs
    ).logits.cpu()


maximum_logit_difference = float(
    torch.max(
        torch.abs(
            original_logits - reloaded_logits
        )
    ).item()
)

logits_match = torch.allclose(
    original_logits,
    reloaded_logits,
    rtol=1e-5,
    atol=1e-6,
)

original_predictions = (
    torch.argmax(original_logits, dim=1) + 1
).numpy()

reloaded_predictions = (
    torch.argmax(reloaded_logits, dim=1) + 1
).numpy()

predictions_match = np.array_equal(
    original_predictions,
    reloaded_predictions,
)


if not logits_match:
    raise RuntimeError(
        "Reloaded model logits differ. "
        f"Maximum difference: {maximum_logit_difference}"
    )

if not predictions_match:
    raise RuntimeError(
        "Reloaded model predictions differ."
    )


print("Independent final-model test passed.")
print("Model directory:", FINAL_MODEL_DIR)
print("Verification indices:", verification_indices)
print(
    "Reloaded tokenizer class:",
    type(reloaded_tokenizer).__name__,
)
print("Tokenizer outputs match:", tokenization_matches)
print("Logits match:", logits_match)
print(
    "Maximum logit difference:",
    maximum_logit_difference,
)
print("Predictions match:", predictions_match)
print(
    "Predicted ratings:",
    reloaded_predictions.tolist(),
)
print(
    "Reloaded model parameter dtypes:",
    {
        parameter.dtype
        for parameter in reloaded_model.parameters()
    },
)

: 

<div dir="rtl" align="right">

## تست سبک و مستقل مدل نهایی پس از Restart کرنل

در این مرحله فقط مدل نهایی ذخیره‌شده از Google Drive بارگذاری می‌شود.

پیش‌بینی سه نمونه ثابت از داده Test با پیش‌بینی‌های قبلی ذخیره‌شده مقایسه خواهد شد. این روش حافظه بسیار کمتری مصرف می‌کند و به متغیرهای کرنل قبلی وابسته نیست.

</div>

In [1]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


FINAL_MODEL_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)

TEST_DATA_PATH = Path(
    "/home/ali/datasets/AI_Project2/nlp/"
    "nlp_test_clean.csv"
)

SAVED_PREDICTIONS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "deberta_v3_small_test_predictions.csv"
)

VERIFICATION_INDICES = [0, 10_000, 19_999]
MAX_LENGTH = 512


for required_path in [
    FINAL_MODEL_DIR,
    TEST_DATA_PATH,
    SAVED_PREDICTIONS_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path not found: {required_path}"
        )


# پاک‌سازی حافظه پس از Restart
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# فقط ستون‌های موردنیاز خوانده می‌شوند
test_text_df = pd.read_csv(
    TEST_DATA_PATH,
    usecols=[
        "row_id",
        "summary",
        "reviewText_clean",
    ],
    low_memory=False,
)

saved_predictions_df = pd.read_csv(
    SAVED_PREDICTIONS_PATH,
    usecols=[
        "row_id",
        "predicted",
    ],
)


verification_df = (
    test_text_df.iloc[VERIFICATION_INDICES]
    .copy()
)

verification_df["summary"] = (
    verification_df["summary"]
    .fillna("")
    .astype(str)
    .str.strip()
)

verification_df["reviewText_clean"] = (
    verification_df["reviewText_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
)


expected_predictions = (
    verification_df[["row_id"]]
    .merge(
        saved_predictions_df,
        on="row_id",
        how="left",
        validate="one_to_one",
    )
)

if expected_predictions["predicted"].isna().any():
    raise RuntimeError(
        "Saved predictions are missing for "
        "one or more verification rows."
    )


# بارگذاری تنها یک نسخه از مدل
final_tokenizer = AutoTokenizer.from_pretrained(
    str(FINAL_MODEL_DIR),
    local_files_only=True,
)

final_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        str(FINAL_MODEL_DIR),
        dtype=torch.float32,
        local_files_only=True,
        low_cpu_mem_usage=True,
    )
)

final_model.to("cpu")
final_model.eval()


encoded_inputs = final_tokenizer(
    verification_df["summary"].tolist(),
    verification_df["reviewText_clean"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True,
    return_tensors="pt",
)


with torch.inference_mode():
    logits = final_model(
        **encoded_inputs
    ).logits.cpu()

actual_predictions = (
    torch.argmax(logits, dim=1) + 1
).numpy()

expected_ratings = (
    expected_predictions["predicted"]
    .astype("int64")
    .to_numpy()
)

predictions_match = np.array_equal(
    actual_predictions,
    expected_ratings,
)

if not torch.isfinite(logits).all():
    raise RuntimeError(
        "Reloaded model produced NaN or Inf logits."
    )

if not predictions_match:
    raise RuntimeError(
        "Reloaded model predictions do not match "
        "the previously saved predictions."
    )


print("Lightweight independent model test passed.")
print("Model directory:", FINAL_MODEL_DIR)
print("Verification indices:", VERIFICATION_INDICES)
print(
    "Row IDs:",
    verification_df["row_id"].tolist(),
)
print(
    "Expected ratings:",
    expected_ratings.tolist(),
)
print(
    "Reloaded-model ratings:",
    actual_predictions.tolist(),
)
print("Predictions match:", predictions_match)
print("Logits are finite:", True)
print(
    "Tokenizer class:",
    type(final_tokenizer).__name__,
)
print(
    "Model parameter dtypes:",
    {
        parameter.dtype
        for parameter in final_model.parameters()
    },
)

/home/ali/miniconda3/envs/ai-bootcamp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SafetensorError: Error while deserializing header: header too small

<div dir="rtl" align="right">

## بررسی سلامت فایل‌های وزن مدل

در این مرحله فایل `model.safetensors` در سه مسیر بررسی می‌شود:

- Checkpoint اصلی آموزش
- Export محلی
- نسخه ذخیره‌شده در Google Drive

اندازه، هش، Header و امکان بازشدن با کتابخانه Safetensors کنترل می‌شود. هیچ فایلی تغییر نمی‌کند.

</div>

In [2]:
from pathlib import Path
import hashlib
import json
import struct

from safetensors import safe_open


MODEL_FILES = {
    "source_checkpoint": Path(
        "/home/ali/projects/ai-bootcamp/codes/"
        "ai-bootcamp-DeepLearning/artifacts/nlp/"
        "deberta_v3_small_full_training/checkpoint-83058/"
        "model.safetensors"
    ),
    "local_export": Path(
        "/home/ali/projects/ai-bootcamp/codes/"
        "ai-bootcamp-DeepLearning/artifacts/nlp/"
        "deberta_v3_small_final_export/"
        "model.safetensors"
    ),
    "drive_export": Path(
        "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
        "deberta_v3_small_final/"
        "model.safetensors"
    ),
}


def calculate_sha256(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


results = {}

for name, path in MODEL_FILES.items():
    print("=" * 90)
    print("Name:", name)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        results[name] = {
            "exists": False,
        }
        continue

    file_size = path.stat().st_size

    with path.open("rb") as file_handle:
        first_16_bytes = file_handle.read(16)

    header_length = None
    header_boundary_valid = False

    if len(first_16_bytes) >= 8:
        header_length = struct.unpack(
            "<Q",
            first_16_bytes[:8],
        )[0]

        header_boundary_valid = (
            header_length > 0
            and 8 + header_length <= file_size
        )

    print("File size:", f"{file_size:,}", "bytes")
    print("First 16 bytes:", repr(first_16_bytes))
    print("Declared header length:", header_length)
    print(
        "Header fits inside file:",
        header_boundary_valid,
    )

    sha256_value = calculate_sha256(path)
    print("SHA256:", sha256_value)

    safetensors_status = "FAILED"
    safetensors_error = None
    tensor_count = None
    first_tensor_name = None
    first_tensor_shape = None

    try:
        with safe_open(
            str(path),
            framework="pt",
            device="cpu",
        ) as file_handle:
            tensor_keys = list(file_handle.keys())
            tensor_count = len(tensor_keys)

            if tensor_keys:
                first_tensor_name = tensor_keys[0]
                first_tensor_shape = tuple(
                    file_handle.get_tensor(
                        first_tensor_name
                    ).shape
                )

        safetensors_status = "SUCCESS"

    except Exception as error:
        safetensors_error = (
            f"{type(error).__name__}: {error}"
        )

    print("Safetensors status:", safetensors_status)
    print("Tensor count:", tensor_count)
    print("First tensor:", first_tensor_name)
    print("First tensor shape:", first_tensor_shape)

    if safetensors_error:
        print("Safetensors error:", safetensors_error)

    results[name] = {
        "exists": True,
        "size_bytes": file_size,
        "sha256": sha256_value,
        "header_length": header_length,
        "header_boundary_valid": header_boundary_valid,
        "safetensors_status": safetensors_status,
        "safetensors_error": safetensors_error,
        "tensor_count": tensor_count,
    }


print("\n" + "=" * 90)
print("Pairwise comparison:")

available_names = [
    name
    for name, result in results.items()
    if result.get("exists")
]

for index, first_name in enumerate(available_names):
    for second_name in available_names[index + 1:]:
        first_result = results[first_name]
        second_result = results[second_name]

        print({
            "files": f"{first_name} vs {second_name}",
            "same_size": (
                first_result["size_bytes"]
                == second_result["size_bytes"]
            ),
            "same_sha256": (
                first_result["sha256"]
                == second_result["sha256"]
            ),
        })

Name: source_checkpoint
Path: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_full_training/checkpoint-83058/model.safetensors
Exists: True
File size: 567,607,780 bytes
First 16 bytes: b'\xc81\x00\x00\x00\x00\x00\x00{"__meta'
Declared header length: 12744
Header fits inside file: True
SHA256: 122f5b8efc0215fd7ddaa1e33c1b2d5bcf55d32df921e72920ae241de4cc2bb0
Safetensors status: SUCCESS
Tensor count: 106
First tensor: classifier.bias
First tensor shape: (5,)
Name: local_export
Path: /home/ali/projects/ai-bootcamp/codes/ai-bootcamp-DeepLearning/artifacts/nlp/deberta_v3_small_final_export/model.safetensors
Exists: True
File size: 567,607,780 bytes
First 16 bytes: b'\xc81\x00\x00\x00\x00\x00\x00{"__meta'
Declared header length: 12744
Header fits inside file: True
SHA256: 122f5b8efc0215fd7ddaa1e33c1b2d5bcf55d32df921e72920ae241de4cc2bb0
Safetensors status: SUCCESS
Tensor count: 106
First tensor: classifier.bias
First tensor shape: (5,)
Name: drive_e

<div dir="rtl" align="right">

## ترمیم نسخه مدل روی Google Drive

در این مرحله:

- نسخه خراب Google Drive موقتاً نگه‌داری می‌شود.
- فایل‌های مدل دوباره از Export محلی سالم کپی می‌شوند.
- پس از پایان نوشتن، اندازه و هش تمام فایل‌ها دوباره بررسی می‌شود.
- فایل وزن با کتابخانه Safetensors باز می‌شود.
- فقط بعد از تأیید کامل، نسخه خراب قبلی حذف خواهد شد.

نسخه محلی سالم و Checkpoint اصلی هیچ تغییری نمی‌کنند.

</div>

In [3]:
from pathlib import Path
from datetime import datetime
import hashlib
import os
import shutil
import time

from safetensors import safe_open


LOCAL_EXPORT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_final_export"
)

DRIVE_MODELS_PARENT = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp"
)

DRIVE_FINAL_MODEL_DIR = (
    DRIVE_MODELS_PARENT / "deberta_v3_small_final"
)

DRIVE_TEMP_MODEL_DIR = (
    DRIVE_MODELS_PARENT
    / ".deberta_v3_small_final_repairing"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

DRIVE_CORRUPT_BACKUP_DIR = (
    DRIVE_MODELS_PARENT
    / f".deberta_v3_small_final_corrupt_{timestamp}"
)


def calculate_sha256(
    file_path,
    chunk_size=16 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def build_manifest(root_directory):
    manifest = {}

    for file_path in sorted(
        root_directory.rglob("*")
    ):
        if not file_path.is_file():
            continue

        relative_path = str(
            file_path.relative_to(root_directory)
        )

        manifest[relative_path] = {
            "size_bytes": file_path.stat().st_size,
            "sha256": calculate_sha256(file_path),
        }

    return manifest


def copy_file_with_flush(source, destination):
    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with source.open("rb") as source_handle:
        with destination.open("wb") as destination_handle:
            shutil.copyfileobj(
                source_handle,
                destination_handle,
                length=16 * 1024 * 1024,
            )

            destination_handle.flush()
            os.fsync(destination_handle.fileno())

    shutil.copystat(
        source,
        destination,
        follow_symlinks=True,
    )


# بررسی نسخه سالم محلی
if not LOCAL_EXPORT_DIR.exists():
    raise FileNotFoundError(
        f"Local export not found: {LOCAL_EXPORT_DIR}"
    )

local_model_file = (
    LOCAL_EXPORT_DIR / "model.safetensors"
)

if not local_model_file.exists():
    raise FileNotFoundError(
        f"Local model weights not found: "
        f"{local_model_file}"
    )

with safe_open(
    str(local_model_file),
    framework="pt",
    device="cpu",
) as file_handle:
    local_tensor_count = len(
        list(file_handle.keys())
    )

if local_tensor_count != 106:
    raise RuntimeError(
        f"Unexpected local tensor count: "
        f"{local_tensor_count}"
    )


# جلوگیری از تداخل با Repair قبلی
if DRIVE_TEMP_MODEL_DIR.exists():
    raise FileExistsError(
        f"Temporary repair directory already exists: "
        f"{DRIVE_TEMP_MODEL_DIR}"
    )

if DRIVE_CORRUPT_BACKUP_DIR.exists():
    raise FileExistsError(
        f"Backup directory already exists: "
        f"{DRIVE_CORRUPT_BACKUP_DIR}"
    )


local_manifest = build_manifest(
    LOCAL_EXPORT_DIR
)

print("Local source verified.")
print(
    "Local files:",
    len(local_manifest),
)
print(
    "Local model SHA256:",
    local_manifest[
        "model.safetensors"
    ]["sha256"],
)


# کپی فایل‌به‌فایل همراه با Flush
DRIVE_TEMP_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

for source_file in sorted(
    LOCAL_EXPORT_DIR.rglob("*")
):
    if not source_file.is_file():
        continue

    relative_path = source_file.relative_to(
        LOCAL_EXPORT_DIR
    )

    destination_file = (
        DRIVE_TEMP_MODEL_DIR / relative_path
    )

    print("Copying:", relative_path)

    copy_file_with_flush(
        source_file,
        destination_file,
    )


# وادارکردن سیستم‌عامل به نوشتن داده‌های معلق
os.sync()
time.sleep(10)


# بررسی نسخه موقت پس از بسته‌شدن تمام فایل‌ها
temporary_manifest = build_manifest(
    DRIVE_TEMP_MODEL_DIR
)

if temporary_manifest != local_manifest:
    differing_files = sorted(
        set(local_manifest)
        | set(temporary_manifest)
    )

    mismatch_report = []

    for file_name in differing_files:
        if (
            local_manifest.get(file_name)
            != temporary_manifest.get(file_name)
        ):
            mismatch_report.append(file_name)

    raise RuntimeError(
        "Temporary Google Drive copy verification "
        f"failed. Different files: {mismatch_report}"
    )


temporary_model_file = (
    DRIVE_TEMP_MODEL_DIR / "model.safetensors"
)

with safe_open(
    str(temporary_model_file),
    framework="pt",
    device="cpu",
) as file_handle:
    temporary_tensor_count = len(
        list(file_handle.keys())
    )

if temporary_tensor_count != local_tensor_count:
    raise RuntimeError(
        "Temporary model tensor count differs "
        "from the local model."
    )


# جایگزینی کنترل‌شده نسخه خراب
if DRIVE_FINAL_MODEL_DIR.exists():
    DRIVE_FINAL_MODEL_DIR.rename(
        DRIVE_CORRUPT_BACKUP_DIR
    )

try:
    DRIVE_TEMP_MODEL_DIR.rename(
        DRIVE_FINAL_MODEL_DIR
    )

    os.sync()
    time.sleep(10)

    final_manifest = build_manifest(
        DRIVE_FINAL_MODEL_DIR
    )

    if final_manifest != local_manifest:
        raise RuntimeError(
            "Final Google Drive verification failed."
        )

    final_model_file = (
        DRIVE_FINAL_MODEL_DIR
        / "model.safetensors"
    )

    with safe_open(
        str(final_model_file),
        framework="pt",
        device="cpu",
    ) as file_handle:
        final_tensor_count = len(
            list(file_handle.keys())
        )

    if final_tensor_count != local_tensor_count:
        raise RuntimeError(
            "Final model tensor count is incorrect."
        )

except Exception:
    # بازگردانی نسخه قبلی در صورت شکست جایگزینی
    if DRIVE_FINAL_MODEL_DIR.exists():
        shutil.rmtree(
            DRIVE_FINAL_MODEL_DIR
        )

    if DRIVE_CORRUPT_BACKUP_DIR.exists():
        DRIVE_CORRUPT_BACKUP_DIR.rename(
            DRIVE_FINAL_MODEL_DIR
        )

    raise


# حذف نسخه خراب فقط پس از تأیید کامل
if DRIVE_CORRUPT_BACKUP_DIR.exists():
    shutil.rmtree(
        DRIVE_CORRUPT_BACKUP_DIR
    )


print("\nGoogle Drive model repair completed successfully.")
print(
    "Final directory:",
    DRIVE_FINAL_MODEL_DIR,
)
print(
    "Files verified:",
    len(final_manifest),
)
print(
    "Model size:",
    f"{final_model_file.stat().st_size:,}",
    "bytes",
)
print(
    "Model SHA256:",
    calculate_sha256(final_model_file),
)
print(
    "Tensor count:",
    final_tensor_count,
)
print(
    "Local and Drive manifests match:",
    final_manifest == local_manifest,
)

Local source verified.
Local files: 6
Local model SHA256: 122f5b8efc0215fd7ddaa1e33c1b2d5bcf55d32df921e72920ae241de4cc2bb0
Copying: config.json
Copying: file_manifest.json
Copying: model.safetensors
Copying: project_metadata.json
Copying: tokenizer.json
Copying: tokenizer_config.json


RuntimeError: Final Google Drive verification failed.

<div dir="rtl" align="right">

## بررسی وضعیت مدل پس از Rollback

در این مرحله:

- وجود پوشه نهایی، موقت و نسخه پشتیبان بررسی می‌شود.
- فایل‌های Export محلی و Google Drive از نظر اندازه و هش مقایسه می‌شوند.
- فایل وزن محلی دوباره با Safetensors بررسی می‌شود.
- هیچ فایلی حذف، جابه‌جا یا بازنویسی نمی‌شود.

</div>


In [4]:
from pathlib import Path
import hashlib

from safetensors import safe_open


LOCAL_EXPORT_DIR = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_final_export"
)

DRIVE_MODELS_PARENT = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp"
)

DRIVE_FINAL_MODEL_DIR = (
    DRIVE_MODELS_PARENT / "deberta_v3_small_final"
)

DRIVE_TEMP_MODEL_DIR = (
    DRIVE_MODELS_PARENT
    / ".deberta_v3_small_final_repairing"
)


def calculate_sha256(
    file_path,
    chunk_size=16 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def inspect_directory(directory):
    result = {}

    if not directory.exists():
        return result

    for file_path in sorted(directory.rglob("*")):
        if not file_path.is_file():
            continue

        relative_path = str(
            file_path.relative_to(directory)
        )

        try:
            result[relative_path] = {
                "size_bytes": file_path.stat().st_size,
                "sha256": calculate_sha256(file_path),
                "read_status": "SUCCESS",
            }

        except Exception as error:
            result[relative_path] = {
                "size_bytes": None,
                "sha256": None,
                "read_status": (
                    f"{type(error).__name__}: {error}"
                ),
            }

    return result


print("Directory states:")
print(
    "Local export exists:",
    LOCAL_EXPORT_DIR.exists(),
)
print(
    "Drive final exists:",
    DRIVE_FINAL_MODEL_DIR.exists(),
)
print(
    "Drive temporary repair exists:",
    DRIVE_TEMP_MODEL_DIR.exists(),
)

backup_directories = sorted(
    DRIVE_MODELS_PARENT.glob(
        ".deberta_v3_small_final_corrupt_*"
    )
)

print(
    "Remaining backup directories:",
    [path.name for path in backup_directories],
)


# تأیید سلامت وزن محلی
local_model_file = (
    LOCAL_EXPORT_DIR / "model.safetensors"
)

with safe_open(
    str(local_model_file),
    framework="pt",
    device="cpu",
) as file_handle:
    local_tensor_count = len(
        list(file_handle.keys())
    )

print("\nLocal model verification:")
print(
    "Size:",
    f"{local_model_file.stat().st_size:,}",
)
print(
    "Tensor count:",
    local_tensor_count,
)
print(
    "Safetensors valid:",
    local_tensor_count == 106,
)


# مقایسه تمام فایل‌ها
local_manifest = inspect_directory(
    LOCAL_EXPORT_DIR
)

drive_manifest = inspect_directory(
    DRIVE_FINAL_MODEL_DIR
)

all_file_names = sorted(
    set(local_manifest)
    | set(drive_manifest)
)

print("\nFile comparison:")

different_files = []

for file_name in all_file_names:
    local_info = local_manifest.get(file_name)
    drive_info = drive_manifest.get(file_name)

    same = local_info == drive_info

    if not same:
        different_files.append(file_name)

    print({
        "file": file_name,
        "same": same,
        "local_size": (
            local_info.get("size_bytes")
            if local_info else None
        ),
        "drive_size": (
            drive_info.get("size_bytes")
            if drive_info else None
        ),
        "local_read": (
            local_info.get("read_status")
            if local_info else "MISSING"
        ),
        "drive_read": (
            drive_info.get("read_status")
            if drive_info else "MISSING"
        ),
    })


print("\nSummary:")
print(
    "Local file count:",
    len(local_manifest),
)
print(
    "Drive file count:",
    len(drive_manifest),
)
print(
    "Different files:",
    different_files,
)
print(
    "All files match:",
    local_manifest == drive_manifest,
)

Directory states:
Local export exists: True
Drive final exists: True
Drive temporary repair exists: False
Remaining backup directories: []

Local model verification:
Size: 567,607,780
Tensor count: 106
Safetensors valid: True

File comparison:
{'file': 'config.json', 'same': True, 'local_size': 1168, 'drive_size': 1168, 'local_read': 'SUCCESS', 'drive_read': 'SUCCESS'}
{'file': 'file_manifest.json', 'same': True, 'local_size': 809, 'drive_size': 809, 'local_read': 'SUCCESS', 'drive_read': 'SUCCESS'}
{'file': 'model.safetensors', 'same': False, 'local_size': 567607780, 'drive_size': 0, 'local_read': 'SUCCESS', 'drive_read': 'SUCCESS'}
{'file': 'project_metadata.json', 'same': True, 'local_size': 851, 'drive_size': 851, 'local_read': 'SUCCESS', 'drive_read': 'SUCCESS'}
{'file': 'tokenizer.json', 'same': True, 'local_size': 8339974, 'drive_size': 8339974, 'local_read': 'SUCCESS', 'drive_read': 'SUCCESS'}
{'file': 'tokenizer_config.json', 'same': True, 'local_size': 510, 'drive_size': 510,

<div dir="rtl" align="right">

## تلاش مجدد برای کپی فایل وزن مدل

فایل سالم `model.safetensors` در یک مسیر موقت روی Google Drive کپی می‌شود. پس از کپی، اندازه، هش و ساختار Safetensors بررسی می‌شود.

پوشه نهایی مدل و فایل‌های قبلی تغییر نمی‌کنند.

</div>

In [5]:
from pathlib import Path
import hashlib
import os
import shutil
import time

from safetensors import safe_open


SOURCE_MODEL_FILE = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_final_export/"
    "model.safetensors"
)

RETRY_MODEL_FILE = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    ".model.safetensors.retry"
)


def calculate_sha256(
    path,
    chunk_size=16 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


if not SOURCE_MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Source file not found: {SOURCE_MODEL_FILE}"
    )

if RETRY_MODEL_FILE.exists():
    raise FileExistsError(
        "Retry file already exists. "
        f"Do not overwrite it: {RETRY_MODEL_FILE}"
    )


source_size = SOURCE_MODEL_FILE.stat().st_size
source_hash = calculate_sha256(SOURCE_MODEL_FILE)

print("Source size:", f"{source_size:,}")
print("Source SHA256:", source_hash)
print("Copying to temporary Google Drive file...")


with SOURCE_MODEL_FILE.open("rb") as source_handle:
    with RETRY_MODEL_FILE.open("wb") as destination_handle:
        shutil.copyfileobj(
            source_handle,
            destination_handle,
            length=8 * 1024 * 1024,
        )

        destination_handle.flush()
        os.fsync(destination_handle.fileno())


os.sync()

print("Copy finished. Waiting for Drive synchronization...")
time.sleep(30)


first_size = RETRY_MODEL_FILE.stat().st_size

print(
    "Temporary file size after 30 seconds:",
    f"{first_size:,}",
)

if first_size != source_size:
    raise RuntimeError(
        "Temporary Drive file has an incorrect size. "
        f"Expected {source_size:,}, got {first_size:,}."
    )


retry_hash = calculate_sha256(RETRY_MODEL_FILE)

print("Temporary file SHA256:", retry_hash)
print("SHA256 matches:", retry_hash == source_hash)

if retry_hash != source_hash:
    raise RuntimeError(
        "Temporary Drive file hash does not match."
    )


with safe_open(
    str(RETRY_MODEL_FILE),
    framework="pt",
    device="cpu",
) as file_handle:
    tensor_names = list(file_handle.keys())

if len(tensor_names) != 106:
    raise RuntimeError(
        "Unexpected tensor count in temporary file: "
        f"{len(tensor_names)}"
    )


# بررسی دوباره برای اطمینان از صفرنشدن فایل پس از کمی تأخیر
time.sleep(20)

second_size = RETRY_MODEL_FILE.stat().st_size

if second_size != source_size:
    raise RuntimeError(
        "Temporary Drive file size changed after waiting. "
        f"Current size: {second_size:,}"
    )


print("\nRetry copy completed successfully.")
print("Temporary file:", RETRY_MODEL_FILE)
print("Final size:", f"{second_size:,}", "bytes")
print("SHA256 matches:", True)
print("Safetensors tensor count:", len(tensor_names))

Source size: 567,607,780
Source SHA256: 122f5b8efc0215fd7ddaa1e33c1b2d5bcf55d32df921e72920ae241de4cc2bb0
Copying to temporary Google Drive file...
Copy finished. Waiting for Drive synchronization...
Temporary file size after 30 seconds: 567,607,780
Temporary file SHA256: 122f5b8efc0215fd7ddaa1e33c1b2d5bcf55d32df921e72920ae241de4cc2bb0
SHA256 matches: True

Retry copy completed successfully.
Temporary file: /home/ali/GoogleDrive/AI_Project2/models/nlp/.model.safetensors.retry
Final size: 567,607,780 bytes
SHA256 matches: True
Safetensors tensor count: 106


<div dir="rtl" align="right">

## جایگزینی فایل وزن خراب با نسخه تأییدشده

فایل موقت سالم که اندازه، هش و ساختار Safetensors آن بررسی شده، جایگزین فایل صفر‌بایتی پوشه نهایی مدل می‌شود.

پس از جایگزینی، فایل نهایی دوباره از نظر اندازه، هش و تعداد Tensorها کنترل خواهد شد.

</div>

In [6]:
from pathlib import Path
import hashlib
import os
import time

from safetensors import safe_open


SOURCE_LOCAL_FILE = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_final_export/"
    "model.safetensors"
)

VERIFIED_RETRY_FILE = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    ".model.safetensors.retry"
)

FINAL_MODEL_FILE = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final/"
    "model.safetensors"
)

ZERO_BACKUP_FILE = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final/"
    ".model.safetensors.zero_backup"
)


def calculate_sha256(
    path,
    chunk_size=16 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# بررسی فایل محلی مرجع
if not SOURCE_LOCAL_FILE.exists():
    raise FileNotFoundError(
        f"Local source not found: {SOURCE_LOCAL_FILE}"
    )

expected_size = SOURCE_LOCAL_FILE.stat().st_size
expected_hash = calculate_sha256(
    SOURCE_LOCAL_FILE
)


# بررسی دوباره فایل موقت سالم
if not VERIFIED_RETRY_FILE.exists():
    raise FileNotFoundError(
        f"Verified retry file not found: "
        f"{VERIFIED_RETRY_FILE}"
    )

retry_size = VERIFIED_RETRY_FILE.stat().st_size
retry_hash = calculate_sha256(
    VERIFIED_RETRY_FILE
)

if retry_size != expected_size:
    raise RuntimeError(
        "Retry file size no longer matches "
        "the local source."
    )

if retry_hash != expected_hash:
    raise RuntimeError(
        "Retry file hash no longer matches "
        "the local source."
    )

with safe_open(
    str(VERIFIED_RETRY_FILE),
    framework="pt",
    device="cpu",
) as file_handle:
    retry_tensor_count = len(
        list(file_handle.keys())
    )

if retry_tensor_count != 106:
    raise RuntimeError(
        f"Unexpected retry tensor count: "
        f"{retry_tensor_count}"
    )


# کنترل وضعیت فایل نهایی
if not FINAL_MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Final model file not found: "
        f"{FINAL_MODEL_FILE}"
    )

if ZERO_BACKUP_FILE.exists():
    raise FileExistsError(
        f"Backup file already exists: "
        f"{ZERO_BACKUP_FILE}"
    )

print(
    "Current final file size:",
    f"{FINAL_MODEL_FILE.stat().st_size:,}",
    "bytes",
)


# نگه‌داری موقت فایل خراب
FINAL_MODEL_FILE.rename(
    ZERO_BACKUP_FILE
)


# انتقال فایل سالم به مسیر نهایی
VERIFIED_RETRY_FILE.rename(
    FINAL_MODEL_FILE
)

os.sync()
time.sleep(30)


# تأیید فایل نهایی
final_size = FINAL_MODEL_FILE.stat().st_size
final_hash = calculate_sha256(
    FINAL_MODEL_FILE
)

with safe_open(
    str(FINAL_MODEL_FILE),
    framework="pt",
    device="cpu",
) as file_handle:
    final_tensor_count = len(
        list(file_handle.keys())
    )

verification_passed = (
    final_size == expected_size
    and final_hash == expected_hash
    and final_tensor_count == 106
)

if not verification_passed:
    raise RuntimeError(
        "Final model verification failed. "
        "The zero-byte backup has been preserved."
    )


# حذف نسخه صفر‌بایتی فقط بعد از تأیید کامل
ZERO_BACKUP_FILE.unlink()


print("\nFinal model replacement completed successfully.")
print("Final file:", FINAL_MODEL_FILE)
print("Final size:", f"{final_size:,}", "bytes")
print("SHA256:", final_hash)
print("SHA256 matches local source:", final_hash == expected_hash)
print("Safetensors tensor count:", final_tensor_count)
print("Zero-byte backup removed:", not ZERO_BACKUP_FILE.exists())

Current final file size: 0 bytes

Final model replacement completed successfully.
Final file: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_small_final/model.safetensors
Final size: 567,607,780 bytes
SHA256: 122f5b8efc0215fd7ddaa1e33c1b2d5bcf55d32df921e72920ae241de4cc2bb0
SHA256 matches local source: True
Safetensors tensor count: 106
Zero-byte backup removed: True


<div dir="rtl" align="right">

## تست مستقل بارگذاری مدل نهایی

مدل و Tokenizer مستقیماً از Google Drive بارگذاری می‌شوند و پیش‌بینی سه نمونه ثابت با خروجی ذخیره‌شده قبلی مقایسه خواهد شد.

</div>

In [7]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)


FINAL_MODEL_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)

TEST_DATA_PATH = Path(
    "/home/ali/datasets/AI_Project2/nlp/"
    "nlp_test_clean.csv"
)

SAVED_PREDICTIONS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "deberta_v3_small_test_predictions.csv"
)

VERIFICATION_INDICES = [0, 10_000, 19_999]
MAX_LENGTH = 512


gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


test_df = pd.read_csv(
    TEST_DATA_PATH,
    usecols=[
        "row_id",
        "summary",
        "reviewText_clean",
    ],
    low_memory=False,
)

saved_predictions_df = pd.read_csv(
    SAVED_PREDICTIONS_PATH,
    usecols=[
        "row_id",
        "predicted",
    ],
)


verification_df = (
    test_df.iloc[VERIFICATION_INDICES]
    .copy()
)

verification_df["summary"] = (
    verification_df["summary"]
    .fillna("")
    .astype(str)
    .str.strip()
)

verification_df["reviewText_clean"] = (
    verification_df["reviewText_clean"]
    .fillna("")
    .astype(str)
    .str.strip()
)


expected_df = (
    verification_df[["row_id"]]
    .merge(
        saved_predictions_df,
        on="row_id",
        how="left",
        validate="one_to_one",
    )
)

if expected_df["predicted"].isna().any():
    raise RuntimeError(
        "Expected predictions are missing."
    )


final_tokenizer = AutoTokenizer.from_pretrained(
    str(FINAL_MODEL_DIR),
    local_files_only=True,
)

final_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        str(FINAL_MODEL_DIR),
        dtype=torch.float32,
        local_files_only=True,
        low_cpu_mem_usage=True,
    )
)

final_model.to("cpu")
final_model.eval()


encoded_inputs = final_tokenizer(
    verification_df["summary"].tolist(),
    verification_df["reviewText_clean"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True,
    return_tensors="pt",
)


with torch.inference_mode():
    logits = final_model(
        **encoded_inputs
    ).logits.cpu()


if not torch.isfinite(logits).all():
    raise RuntimeError(
        "Model produced NaN or Inf logits."
    )


actual_predictions = (
    torch.argmax(logits, dim=1) + 1
).numpy()

expected_predictions = (
    expected_df["predicted"]
    .astype("int64")
    .to_numpy()
)

predictions_match = np.array_equal(
    actual_predictions,
    expected_predictions,
)

if not predictions_match:
    raise RuntimeError(
        "Reloaded model predictions differ "
        "from saved predictions."
    )


print("Independent final-model test passed.")
print("Model directory:", FINAL_MODEL_DIR)
print("Verification indices:", VERIFICATION_INDICES)
print(
    "Row IDs:",
    verification_df["row_id"].tolist(),
)
print(
    "Expected ratings:",
    expected_predictions.tolist(),
)
print(
    "Reloaded-model ratings:",
    actual_predictions.tolist(),
)
print("Predictions match:", predictions_match)
print("Logits are finite:", True)
print(
    "Tokenizer class:",
    type(final_tokenizer).__name__,
)
print(
    "Model parameter dtypes:",
    {
        parameter.dtype
        for parameter in final_model.parameters()
    },
)

Loading weights: 100%|██████████| 106/106 [00:00<00:00, 15647.63it/s]


Independent final-model test passed.
Model directory: /home/ali/GoogleDrive/AI_Project2/models/nlp/deberta_v3_small_final
Verification indices: [0, 10000, 19999]
Row IDs: [0, 10000, 19999]
Expected ratings: [1, 3, 5]
Reloaded-model ratings: [1, 3, 5]
Predictions match: True
Logits are finite: True
Tokenizer class: DebertaV2Tokenizer
Model parameter dtypes: {torch.float32}


<div dir="rtl" align="right">

## ثبت نهایی آثار بخش NLP

در این مرحله فایل‌های نهایی پروژه شامل مدل، Submission و پیش‌بینی‌های تفصیلی بررسی می‌شوند.

سپس یک فایل Manifest شامل موارد زیر ساخته خواهد شد:

- مسیر و هش فایل‌های مدل
- معیارهای Validation
- مشخصات فایل Submission
- مسیر خروجی‌های نهایی
- نتیجه تست مستقل بارگذاری مدل

هیچ‌کدام از خروجی‌های قبلی بازنویسی نمی‌شوند.

</div>

In [8]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import pandas as pd


MODEL_DIR = Path(
    "/home/ali/GoogleDrive/AI_Project2/models/nlp/"
    "deberta_v3_small_final"
)

SUBMISSION_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "q2_submission.csv"
)

DETAILED_PREDICTIONS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "deberta_v3_small_test_predictions.csv"
)

FULL_RESULTS_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "deberta_v3_small_full_training_results.csv"
)

TRAINING_SUMMARY_PATH = Path(
    "/home/ali/projects/ai-bootcamp/codes/"
    "ai-bootcamp-DeepLearning/artifacts/nlp/"
    "deberta_v3_small_full_training/"
    "training_summary.json"
)

FINAL_MANIFEST_PATH = Path(
    "/home/ali/GoogleDrive/AI_Project2/outputs/nlp/"
    "nlp_final_artifacts_manifest.json"
)


def calculate_sha256(
    file_path,
    chunk_size=16 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def describe_file(file_path):
    return {
        "path": str(file_path),
        "size_bytes": file_path.stat().st_size,
        "sha256": calculate_sha256(file_path),
    }


required_paths = [
    MODEL_DIR,
    SUBMISSION_PATH,
    DETAILED_PREDICTIONS_PATH,
    FULL_RESULTS_PATH,
    TRAINING_SUMMARY_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required artifact not found: {required_path}"
        )

if FINAL_MANIFEST_PATH.exists():
    raise FileExistsError(
        "Final manifest already exists and will not be "
        f"overwritten: {FINAL_MANIFEST_PATH}"
    )


# بررسی Submission
submission_df = pd.read_csv(
    SUBMISSION_PATH
)

if submission_df.shape != (20_000, 1):
    raise RuntimeError(
        f"Unexpected submission shape: "
        f"{submission_df.shape}"
    )

if submission_df.columns.tolist() != ["predicted"]:
    raise RuntimeError(
        f"Unexpected submission columns: "
        f"{submission_df.columns.tolist()}"
    )

if submission_df["predicted"].isna().any():
    raise RuntimeError(
        "Submission contains missing predictions."
    )

submission_values = set(
    submission_df["predicted"]
    .astype("int64")
    .unique()
    .tolist()
)

if not submission_values.issubset(
    {1, 2, 3, 4, 5}
):
    raise RuntimeError(
        f"Invalid submission values: "
        f"{sorted(submission_values)}"
    )


# خواندن نتایج آموزش
full_results_df = pd.read_csv(
    FULL_RESULTS_PATH
)

with TRAINING_SUMMARY_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    training_summary = json.load(file_handle)


# ثبت تمام فایل‌های بسته مدل
model_files = {}

for file_path in sorted(MODEL_DIR.rglob("*")):
    if not file_path.is_file():
        continue

    relative_path = str(
        file_path.relative_to(MODEL_DIR)
    )

    model_files[relative_path] = (
        describe_file(file_path)
    )


required_model_files = {
    "config.json",
    "model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "project_metadata.json",
    "file_manifest.json",
}

missing_model_files = sorted(
    required_model_files - set(model_files)
)

if missing_model_files:
    raise RuntimeError(
        f"Missing final model files: "
        f"{missing_model_files}"
    )


manifest = {
    "created_at_utc": (
        datetime.now(timezone.utc).isoformat()
    ),
    "project_section": "NLP sentiment rating prediction",
    "final_model": {
        "architecture": "microsoft/deberta-v3-small",
        "directory": str(MODEL_DIR),
        "dtype": "torch.float32",
        "number_of_labels": 5,
        "files": model_files,
    },
    "validation_results": {
        "micro_f1": 0.7939468346964691,
        "macro_f1": 0.6854736332,
        "mae": 0.2303007324,
        "within_one_accuracy": 0.9812410855,
        "best_epoch": 2,
    },
    "training": {
        "training_summary_path": str(
            TRAINING_SUMMARY_PATH
        ),
        "training_summary": training_summary,
        "registered_results": (
            full_results_df.to_dict(
                orient="records"
            )
        ),
    },
    "test_outputs": {
        "submission": describe_file(
            SUBMISSION_PATH
        ),
        "detailed_predictions": describe_file(
            DETAILED_PREDICTIONS_PATH
        ),
        "submission_shape": list(
            submission_df.shape
        ),
        "submission_columns": (
            submission_df.columns.tolist()
        ),
        "submission_rating_values": sorted(
            submission_values
        ),
        "prediction_distribution": {
            str(int(rating)): int(count)
            for rating, count in (
                submission_df["predicted"]
                .value_counts()
                .sort_index()
                .items()
            )
        },
    },
    "independent_reload_test": {
        "status": "passed",
        "verification_indices": [
            0,
            10_000,
            19_999,
        ],
        "expected_ratings": [1, 3, 5],
        "reloaded_model_ratings": [1, 3, 5],
        "predictions_match": True,
        "logits_finite": True,
        "tokenizer_class": (
            "DebertaV2Tokenizer"
        ),
        "model_parameter_dtype": (
            "torch.float32"
        ),
    },
}


FINAL_MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with FINAL_MANIFEST_PATH.open(
    "x",
    encoding="utf-8",
) as file_handle:
    json.dump(
        manifest,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )


# بررسی دوباره فایل نوشته‌شده
with FINAL_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:
    verified_manifest = json.load(
        file_handle
    )

if (
    verified_manifest[
        "validation_results"
    ]["micro_f1"]
    != 0.7939468346964691
):
    raise RuntimeError(
        "Final manifest verification failed."
    )


print("Final NLP artifact manifest created successfully.")
print("Manifest path:", FINAL_MANIFEST_PATH)
print(
    "Manifest SHA256:",
    calculate_sha256(FINAL_MANIFEST_PATH),
)
print(
    "Model package files:",
    len(model_files),
)
print(
    "Submission shape:",
    submission_df.shape,
)
print(
    "Validation Micro F1:",
    manifest[
        "validation_results"
    ]["micro_f1"],
)
print(
    "Independent reload test:",
    manifest[
        "independent_reload_test"
    ]["status"],
)

Final NLP artifact manifest created successfully.
Manifest path: /home/ali/GoogleDrive/AI_Project2/outputs/nlp/nlp_final_artifacts_manifest.json
Manifest SHA256: 2b481de3ec878658c50260380c5c3f4960cca9194c84f27dd8c0c5e70175eb9a
Model package files: 6
Submission shape: (20000, 1)
Validation Micro F1: 0.7939468346964691
Independent reload test: passed
